<a href="https://colab.research.google.com/github/ThisumiWijesinghe/Evidence-Verified-RAG-Workflow-/blob/main/Main.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Install Ollama

In [1]:
!apt-get update -qq
!apt-get install -y zstd

W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu noble InRelease' does not seem to provide it (sources.list entry misspelt?)
Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
The following NEW packages will be installed:
  zstd
0 upgraded, 1 newly installed, 0 to remove and 48 not upgraded.
Need to get 644 kB of archives.
After this operation, 1,845 kB of additional disk space will be used.
Get:1 http://archive.ubuntu.com/ubuntu noble-updates/main amd64 zstd amd64 1.5.5+dfsg2-2build1.1 [644 kB]
Fetched 644 kB in 2s (365 kB/s)
Selecting previously unselected package zstd.
(Reading database ... 127324 files and directories currently installed.)
Preparing to unpack .../zstd_1.5.5+dfsg2-2build1.1_amd64.deb ...
Unpacking zstd (1.5.5+dfsg2-2build1.1) ...
Setting up zstd (1.5.5+dfsg2-2build1.1) ...
Processing triggers for man-db (2.12.0-4build2) ...


In [2]:
!curl -fsSL https://ollama.com/install.sh | sh

>>> Installing ollama to /usr/local
>>> Downloading ollama-linux-amd64.tar.zst
######################################################################## 100.0%
>>> Creating ollama user...
>>> Adding ollama user to video group...
>>> Adding current user to ollama group...
>>> Creating ollama systemd service...
>>> The Ollama API is now available at 127.0.0.1:11434.
>>> Install complete. Run "ollama" from the command line.


# Start Ollama Server in background

In [3]:
import subprocess
import time

process = subprocess.Popen(
    ["ollama", "serve"],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
)

# Wait for the server to start
time.sleep(5)
print("Ollama server started, PID:", process.pid)


Ollama server started, PID: 2675


#Pull LIama 3.1

In [4]:
!ollama pull llama3.1:8b

#Pull the embedding model

In [5]:
!ollama pull nomic-embed-text

# Verify both are installed

In [6]:
!ollama list

NAME                       ID              SIZE      MODIFIED               
nomic-embed-text:latest    0a109f422b47    274 MB    Less than a second ago    
llama3.1:8b                46e0c10c039e    4.9 GB    10 seconds ago            


#Test generation works

In [7]:
import requests
response = requests.post(
    "http://localhost:11434/api/generate",
    json={
        "model": "llama3.1:8b",
        "prompt": "Say 'setup successful' and nothing else.",
        "stream": False,
    }
)
print(response.json()["response"])

Setup successful.


#Install Langchain+dependencies for next steps

In [8]:
!pip install -q langchain langchain-community langchain-ollama faiss-cpu pypdf

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.34.2 which is incompatible.


Connect Langchain to Ollama

In [9]:
from langchain_ollama import OllamaLLM

llm = OllamaLLM(model="llama3.1:8b", temperature=0.2)
result = llm.invoke("In one sentence, what is Retrieval Augmented Generation?")
print(result)

Retrieval Augmented Generation (RAG) is a type of artificial intelligence model that combines the strengths of retrieval-based and generative models by first retrieving relevant information from a large database and then generating new text based on that information.


#PDF Ingestion + Chunking

## Upload a test PDF to colab

In [10]:
from google.colab import files

uploaded = files.upload()
pdf_path = list(uploaded.keys())[0]
print("Uploaded:",pdf_path)

Saving dl based detecting malicious packages in pypl.pdf to dl based detecting malicious packages in pypl.pdf
Saving On the Performance of Malware Detection.pdf to On the Performance of Malware Detection.pdf
Saving Quantum redy crypto.pdf to Quantum redy crypto.pdf
Uploaded: dl based detecting malicious packages in pypl.pdf


#Install the text-splitter package

In [11]:
!pip install -q langchain langchain-community langchain-ollama  langchain-text-splitters faiss-cpu pypdf

##Load the PDF with PyPDFLoader

In [12]:
from langchain_community.document_loaders import PyPDFLoader

loader = PyPDFLoader(pdf_path)
pages = loader.load()

print(f"Loaded {len(pages)} pages")
print("Sample text from page 1:\n", pages[0].page_content[:500])

/tmp/ipykernel_1316/2514516427.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader


Loaded 26 pages
Sample text from page 1:
 eDySec: A Deep Learning-based Explainable Dynamic Analysis
Framework for Detecting Malicious Packages in PyPI Ecosystem
Sk Tanzir Mehedi∗
Queensland University of Technology
Brisbane, QLD, Australia
Raja Jurdak
Queensland University of Technology
Brisbane, QLD, Australia
Chadni Islam
Edith Cowan University
Joondalup, WA, Australia
Abu Bakar Siddique Mahi
Research Graduate School, BUBT
Dhaka, Bangladesh
Gowri Ramachandran
Queensland University of Technology
Brisbane, QLD, Australia
Abstract
The s


##Chunk the text with RecursiveChracterTextSplitter

In [13]:
!pip install -q -U langchain-text-splitters


In [14]:
from langchain_text_splitters import RecursiveCharacterTextSplitter
print("import successful!")

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=200,
    separators=["\n\n", "\n", ". ", " ", ""]
)

chunks = text_splitter.split_documents(pages)

print(f"Split into {len(chunks)} chunks")
print("Sample text from chunk 1:\n", chunks[0].page_content[:300])

import successful!
Split into 148 chunks
Sample text from chunk 1:
 eDySec: A Deep Learning-based Explainable Dynamic Analysis
Framework for Detecting Malicious Packages in PyPI Ecosystem
Sk Tanzir Mehedi∗
Queensland University of Technology
Brisbane, QLD, Australia
Raja Jurdak
Queensland University of Technology
Brisbane, QLD, Australia
Chadni Islam
Edith Cowan Uni


##Quick sanity check -- Inspect a few chunks

In [15]:
for i in range(min(3, len(chunks))):
  print(f"---chunk {i} ---")
  print(chunks[i].page_content[:200])
  print("Metadata:", chunks[i].metadata)
  print()

---chunk 0 ---
eDySec: A Deep Learning-based Explainable Dynamic Analysis
Framework for Detecting Malicious Packages in PyPI Ecosystem
Sk Tanzir Mehedi∗
Queensland University of Technology
Brisbane, QLD, Australia
R
Metadata: {'producer': 'pikepdf 8.15.1', 'creator': 'arXiv GenPDF (tex2pdf:a6404ea)', 'creationdate': '2026-04-30T00:20:09+00:00', 'author': 'Sk Tanzir Mehedi; Raja Jurdak; Chadni Islam; Abu Bakar Siddique Mahi; Gowri Ramachandran', 'doi': 'https://doi.org/10.48550/arXiv.2604.26219', 'keywords': 'Deep learning, malicious detection, software supply chain security, dynamic analysis, stability, explainable AI.', 'license': 'http://creativecommons.org/licenses/by/4.0/', 'moddate': '2026-04-30T00:20:09+00:00', 'ptex.fullbanner': 'This is pdfTeX, Version 3.141592653-2.6-1.40.28 (TeX Live 2025) kpathsea version 6.4.1', 'title': 'eDySec: A Deep Learning-based Explainable Dynamic Analysis Framework for Detecting Malicious Packages in PyPI Ecosystem', 'trapped': '/False', 'arxivid': 

#Embeddings & FAISS vector store , test retrieval

##Connect the nomic-embed-text

In [16]:
from langchain_ollama import OllamaEmbeddings

embeddings = OllamaEmbeddings(
    model="nomic-embed-text"
)

print("Embedding model connected succesfully!")

Embedding model connected succesfully!


##Test the embedding model

In [17]:
test_vector = embeddings.embed_query(
    "What is malicious package detection?"
)

print("Embedding dimension:", len(test_vector))
print("first 10 values:", test_vector[:10])

Embedding dimension: 768
first 10 values: [0.07384906, 0.020637663, -0.15453403, -0.018535705, 0.06982891, -0.04924275, -0.005812537, 0.03157499, -0.0047331247, -0.0016610859]


## Create the FAISS vector database

In [18]:
from langchain_community.vectorstores import FAISS

vectorstore = FAISS.from_documents(
    documents=chunks,
    embedding=embeddings
)

print("FAISS Vector database created successfully!")
print("Number of chunks stored:", len(chunks))

FAISS Vector database created successfully!
Number of chunks stored: 148


#FAISS Test retrieval

In [19]:

# ============================================================
# FIX: RESTART OLLAMA SERVER
# ============================================================

import subprocess
import time
import requests

# Start Ollama without blocking the notebook
ollama_log = open("/tmp/ollama_server.log", "a")

process = subprocess.Popen(
    ["ollama", "serve"],
    stdout=ollama_log,
    stderr=subprocess.STDOUT
)

# Wait until Ollama responds
server_ready = False

for attempt in range(20):
    try:
        response = requests.get(
            "http://127.0.0.1:11434/api/tags",
            timeout=3
        )

        if response.status_code == 200:
            server_ready = True
            break

    except requests.exceptions.RequestException:
        pass

    time.sleep(1)

if server_ready:
    print("Ollama server started successfully!")
    print("Available models:")
    print(requests.get(
        "http://127.0.0.1:11434/api/tags"
    ).json())
else:
    print("Ollama server failed to start.")
    print("Check /tmp/ollama_server.log")


Ollama server failed to start.
Check /tmp/ollama_server.log


In [20]:
import os, time, subprocess, requests

OLLAMA_URL = "http://127.0.0.1:11434"

def ollama_up(timeout=3):
    try:
        return requests.get(f"{OLLAMA_URL}/api/tags", timeout=timeout).status_code == 200
    except requests.RequestException:
        return False

def start_ollama():
    # 1. The old server may just be slow (busy with the GPU), so give it 20 s.
    for _ in range(10):
        if ollama_up(timeout=2):
            print("Ollama server already running and healthy.")
            return True
        time.sleep(2)

    # 2. Still unresponsive: kill stale processes. -x matches the exact
    #    process name, so it can't accidentally kill your notebook kernel.
    print("Server unresponsive. Killing stale Ollama processes...")
    subprocess.run(["pkill", "-9", "-x", "ollama"])
    time.sleep(3)

    # 3. Start a fresh server in the background.
    log_file = open("/tmp/ollama_server.log", "w")
    process = subprocess.Popen(
        ["ollama", "serve"],
        stdout=log_file,
        stderr=subprocess.STDOUT,
        stdin=subprocess.DEVNULL,
        start_new_session=True,
        env=os.environ.copy(),
    )

    # 4. Wait for the API to come up.
    for _ in range(60):
        if ollama_up():
            print("Ollama server is READY!")
            return True
        if process.poll() is not None:
            print("Ollama process exited.")
            break
        time.sleep(2)

    print("\nOllama failed to start. Last log lines:")
    with open("/tmp/ollama_server.log", errors="replace") as f:
        print("".join(f.readlines()[-30:]))
    return False

server_ready = start_ollama()
print("Server ready:", server_ready)
if not server_ready:
    raise RuntimeError("Ollama is not ready. Check the log above.")

Server unresponsive. Killing stale Ollama processes...
Ollama server is READY!
Server ready: True


In [21]:

# ============================================================
# CHECK OLLAMA MODELS AND TEST EMBEDDINGS
# ============================================================

import subprocess
import requests
from langchain_ollama import OllamaLLM, OllamaEmbeddings

required_models = [
    "llama3.1:8b",
    "nomic-embed-text"
]

response = requests.get(
    "http://127.0.0.1:11434/api/tags",
    timeout=10
)
response.raise_for_status()

installed_models = {
    model["name"]
    for model in response.json().get("models", [])
}

for model in required_models:
    if model not in installed_models:
        print(f"Downloading {model}...")
        subprocess.run(
            ["ollama", "pull", model],
            check=True
        )
    else:
        print(f"{model}: already installed")

llm = OllamaLLM(
    model="llama3.1:8b",
    temperature=0.2,
    base_url="http://127.0.0.1:11434"
)

embeddings = OllamaEmbeddings(
    model="nomic-embed-text",
    base_url="http://127.0.0.1:11434"
)

test_embedding = embeddings.embed_query(
    "What problem does eDySec aim to solve?"
)

print("\nEmbedding successful!")
print("Embedding dimension:", len(test_embedding))
print("Ollama setup complete.")


llama3.1:8b: already installed

Embedding successful!
Embedding dimension: 768
Ollama setup complete.


In [22]:

# ============================================================
# TEST EMBEDDING MODEL
# ============================================================

import time

start = time.time()

test_embedding = embeddings.embed_query(
    "What problem does eDySec aim to solve?"
)

print("Embedding successful!")
print("Embedding dimension:", len(test_embedding))
print("Time:", round(time.time() - start, 2), "seconds")


Embedding successful!
Embedding dimension: 768
Time: 0.01 seconds


In [23]:

# ============================================================
# TEST FAISS RETRIEVAL
# ============================================================

question = "What problem does eDySec aim to solve?"

retrieved_docs = vectorstore.similarity_search(
    question,
    k=4
)

print("Retrieved chunks:", len(retrieved_docs))

for i, doc in enumerate(retrieved_docs):
    print(f"\n{'='*60}")
    print(f"Retrieved Chunk {i+1}")
    print("Page:", doc.metadata.get("page_label", "Unknown"))
    print("="*60)
    print(doc.page_content[:500])


Retrieved chunks: 4

Retrieved Chunk 1
Page: 6
Conference Name, Month Year, City, Country Tanzir et al.
between the selected feature subset and the model’s learned be-
havior is assessed throughC=F ∗∩Feat(R 𝑒𝑥𝑝), where a larger
overlap indicates stronger agreement between the selected features
and the features that most strongly influence model predictions.
Algorithm 1 presents the pseudo-code for the proposed eDySec
framework workflow.
4 eDySec Design and Setup
This section describes the experimental design of eDySec, including
the computat

Retrieved Chunk 2
Page: 13
eDySec: A Deep Learning-based Explainable Dynamic Analysis Framework for Detecting Malicious Packages in PyPI Ecosystem Conference Name, Month Year, City, Country
References
[1] Hubert Baniecki and Przemyslaw Biecek. 2024. Adversarial Attacks and Defenses
in Explainable Artificial Intelligence: A Survey.Information Fusion107 (2024),
102303. doi:10.1016/j.inffus.2024.102303
[2] Alessio Benavoli, Giorgio Corani, Janez Demš

# Multi Query Generartion

In [24]:

# ============================================================
# ADVANCED RAG - IMPROVED MULTI-QUERY GENERATION
# ============================================================

def generate_search_queries(question, llm, vectorstore):

    # Retrieve initial context from the uploaded paper
    initial_docs = vectorstore.similarity_search(
        question,
        k=4
    )

    context = "\n\n".join(
        doc.page_content[:700]
        for doc in initial_docs
    )

    prompt = f"""
You are an academic research retrieval assistant.

Generate exactly 3 alternative search queries.

ORIGINAL QUESTION:
{question}

DOCUMENT CONTEXT:
{context}

RULES:
1. Preserve the original question's meaning.
2. Use only terminology supported by the document context.
3. Do not introduce unrelated concepts.
4. Do not answer the question.
5. Do not number the queries.
6. Return exactly 3 queries, one per line.

SEARCH QUERIES:
"""

    response = llm.invoke(prompt)

    queries = [
        line.strip().lstrip("-").strip()
        for line in response.split("\n")
        if line.strip()
    ]

    return [question] + queries[:3]


In [25]:

# ============================================================
# TEST IMPROVED MULTI-QUERY GENERATION
# ============================================================

question = "What problem does eDySec aim to solve?"

search_queries = generate_search_queries(
    question,
    llm,
    vectorstore
)

print("Improved Search Queries:")
print("=" * 60)

for i, query in enumerate(search_queries):
    print(f"{i+1}. {query}")


Improved Search Queries:
1. What problem does eDySec aim to solve?
2. What problem does eDySec aim to solve in the PyPI ecosystem through explainable dynamic analysis?
3. What is the primary challenge that eDySec seeks to address in the context of malicious package detection?
4. What issue does the eDySec framework aim to resolve in the PyPI ecosystem through the integration of explainability techniques?


In [26]:

# ============================================================
# ADVANCED RAG - STEP 2: MULTI-QUERY FAISS RETRIEVAL
# ============================================================

def multi_query_retrieve(search_queries, vectorstore, k=6):
    """
    Retrieve document chunks using multiple search queries
    and remove duplicate results.
    """

    all_docs = []

    # Retrieve relevant chunks for each query
    for i, query in enumerate(search_queries):

        print(f"Searching query {i+1}/{len(search_queries)}...")

        docs = vectorstore.similarity_search(
            query,
            k=k
        )

        all_docs.extend(docs)

    print("\nTotal retrieved chunks:", len(all_docs))

    # Remove duplicate chunks
    unique_docs = []
    seen = set()

    for doc in all_docs:

        key = (
            doc.metadata.get("source"),
            doc.metadata.get("page"),
            doc.page_content
        )

        if key not in seen:
            seen.add(key)
            unique_docs.append(doc)

    print("Unique chunks after deduplication:", len(unique_docs))

    return unique_docs


In [27]:

# ============================================================
# TEST MULTI-QUERY RETRIEVAL
# ============================================================

candidate_docs = multi_query_retrieve(
    search_queries,
    vectorstore,
    k=6
)

print("\n" + "=" * 60)
print("MULTI-QUERY RETRIEVAL RESULTS")
print("=" * 60)

for i, doc in enumerate(candidate_docs):

    print(f"\n--- Candidate {i+1} ---")
    print("Page:", doc.metadata.get("page_label", "Unknown"))
    print(doc.page_content[:300])


Searching query 1/4...
Searching query 2/4...
Searching query 3/4...
Searching query 4/4...

Total retrieved chunks: 24
Unique chunks after deduplication: 15

MULTI-QUERY RETRIEVAL RESULTS

--- Candidate 1 ---
Page: 6
Conference Name, Month Year, City, Country Tanzir et al.
between the selected feature subset and the model’s learned be-
havior is assessed throughC=F ∗∩Feat(R 𝑒𝑥𝑝), where a larger
overlap indicates stronger agreement between the selected features
and the features that most strongly influence model 

--- Candidate 2 ---
Page: 13
eDySec: A Deep Learning-based Explainable Dynamic Analysis Framework for Detecting Malicious Packages in PyPI Ecosystem Conference Name, Month Year, City, Country
References
[1] Hubert Baniecki and Przemyslaw Biecek. 2024. Adversarial Attacks and Defenses
in Explainable Artificial Intelligence: A Su

--- Candidate 3 ---
Page: 15
eDySec: A Deep Learning-based Explainable Dynamic Analysis Framework for Detecting Malicious Packages in PyPI Ecosystem 

#Cross - encoder reranking

Install the reranking model dependencies

In [28]:

# ============================================================
# ADVANCED RAG - STEP 3A: INSTALL RERANKER
# ============================================================

!pip install -q sentence-transformers


Load cross encoder ranker

In [29]:

# ============================================================
# ADVANCED RAG - STEP 3B: LOAD CROSS-ENCODER
# ============================================================

from sentence_transformers import CrossEncoder
import torch

device = "cuda" if torch.cuda.is_available() else "cpu"

reranker = CrossEncoder(
    "cross-encoder/ms-marco-MiniLM-L-6-v2",
    device=device
)

print("Reranker loaded successfully!")
print("Device:", device)


config.json:   0%|          | 0.00/794 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 90.9MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/1.33k [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/711k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/132 [00:00<?, ?B/s]

Reranker loaded successfully!
Device: cuda


Create reranking function

In [30]:


# ============================================================
# ADVANCED RAG - STEP 3C: RERANK RETRIEVED CHUNKS
# ============================================================

def rerank_documents(question, candidate_docs, reranker, top_n=5):
    """
    Rerank retrieved chunks using a Cross-Encoder model.
    Return the top-ranked documents and their scores.
    """

    if not candidate_docs:
        return []

    # Create question-document pairs
    pairs = [
        (question, doc.page_content)
        for doc in candidate_docs
    ]

    # Calculate relevance scores
    scores = reranker.predict(pairs)

    # Combine documents with scores
    scored_docs = list(zip(candidate_docs, scores))

    # Sort by relevance (highest first)
    scored_docs.sort(
        key=lambda item: float(item[1]),
        reverse=True
    )

    # Select top-ranked chunks
    return scored_docs[:top_n]



Test Cross-Encoder Reranking

In [31]:

# ============================================================
# ADVANCED RAG - STEP 3D: TEST RERANKING
# ============================================================

question = "What problem does eDySec aim to solve?"

reranked_docs = rerank_documents(
    question,
    candidate_docs,
    reranker,
    top_n=5
)

print("=" * 65)
print("TOP 5 RERANKED EVIDENCE CHUNKS")
print("=" * 65)

for i, (doc, score) in enumerate(reranked_docs):

    print(f"\n{'='*65}")
    print(f"Rank: {i+1}")
    print(f"Page: {doc.metadata.get('page_label', 'Unknown')}")
    print(f"Relevance score: {float(score):.4f}")
    print("=" * 65)

    print(doc.page_content[:600])


TOP 5 RERANKED EVIDENCE CHUNKS

Rank: 1
Page: 2
Relevance score: 1.4210
techniques into the detection pipeline, enabling near-perfect
stable and transparent interpretation of model decisions by
identifying the features that most influence each prediction.
• Experimental results demonstrate that eDySec significantly
outperforms the state-of-the-art framework. Specifically, it
reduces feature dimensionality by 52.78% while lowering
false positives by 82% and false negatives by 79%. It also
improves accuracy by 3% and maintains a low inference
latency of 170ms per package.
This study is structured as follows: Section 2 reviews existing
methods for malicious package d

Rank: 2
Page: 2
Relevance score: 1.3178
ment, and transparent interpretation of model decisions, creates a
dangerous illusion of security and highlights the need for a more
comprehensive analytical framework.
To address these challenges, this study presents eDySec, an effi-
cient, stable, and explainable DL-basedextendedDyna

# Retrieval Quality Grading

Define retrieval grader

In [32]:

# ============================================================
# ADVANCED RAG - STEP 4: RETRIEVAL QUALITY GRADING
# ============================================================

def grade_retrieval(question, reranked_docs, llm):

    # Handle empty retrieval results.
    if not reranked_docs:
        return "INSUFFICIENT"

    # Prepare the top-ranked evidence for evaluation.
    context = "\n\n".join(
        f"Evidence {i+1}:\n{doc.page_content}"
        for i, (doc, score) in enumerate(reranked_docs)
    )

    # Ask the LLM to evaluate evidence sufficiency.
    prompt = f"""
You are an evidence relevance evaluator for academic research.

USER QUESTION:
{question}

RETRIEVED EVIDENCE:
{context}

TASK:
Determine whether the retrieved evidence contains enough
relevant information to answer the user's question.

RULES:
1. Evaluate only the provided evidence.
2. Do not use external knowledge.
3. Do not answer the question.
4. Return exactly one word:
SUFFICIENT
or
INSUFFICIENT

DECISION:
"""

    response = llm.invoke(prompt).strip().upper()

    # Accept only a valid decision.
    if response == "SUFFICIENT":
        return "SUFFICIENT"

    return "INSUFFICIENT"


In [33]:

# ============================================================
# STEP 4B: TEST RETRIEVAL QUALITY GRADING
# ============================================================

question = "What problem does eDySec aim to solve?"

retrieval_status = grade_retrieval(
    question,
    reranked_docs,
    llm
)

print("=" * 60)
print("RETRIEVAL QUALITY ASSESSMENT")
print("=" * 60)

print("Question:", question)
print("Evidence status:", retrieval_status)

if retrieval_status == "SUFFICIENT":
    print("Decision: Proceed to answer generation.")
else:
    print("Decision: Additional evidence retrieval required.")


RETRIEVAL QUALITY ASSESSMENT
Question: What problem does eDySec aim to solve?
Evidence status: INSUFFICIENT
Decision: Additional evidence retrieval required.


# Rertival quality grader debugging

In [34]:

# ============================================================
# STEP 5A: DEBUG RETRIEVAL QUALITY GRADER
# ============================================================

# Prepare evidence from the reranked documents.
context = "\n\n".join(
    f"Evidence {i+1} (Page {doc.metadata.get('page_label', 'Unknown')}):\n"
    f"{doc.page_content}"
    for i, (doc, score) in enumerate(reranked_docs)
)

debug_prompt = f"""
You are evaluating evidence for scientific question answering.

QUESTION:
{question}

EVIDENCE:
{context}

TASK:
Determine whether the evidence is sufficient to answer
the question using only the supplied passages.

Explain briefly:
1. What relevant information is available?
2. What information is missing, if any?
3. Can the question be answered from this evidence?

Finish with exactly one of these labels:
FINAL_DECISION: SUFFICIENT
FINAL_DECISION: INSUFFICIENT
"""

response = llm.invoke(debug_prompt)

print("=" * 60)
print("RETRIEVAL GRADER DEBUG RESULT")
print("=" * 60)
print(response)


RETRIEVAL GRADER DEBUG RESULT
**1. What relevant information is available?**

The evidence provides information about the eDySec framework, its architecture, and its performance in detecting malicious packages. Specifically, it mentions the following:

* The framework's ability to reduce feature dimensionality, lower false positives and false negatives, and improve accuracy.
* The use of deep learning and explainable AI techniques to provide stable and transparent interpretations of model decisions.
* The framework's architecture, which consists of three interacting components: the user terminal, the detection server, and the sandbox server.
* The results of the experimental evaluation, which demonstrate the framework's performance in detecting malicious packages.

**2. What information is missing, if any?**

The evidence does not provide a clear explanation of the problem that eDySec aims to solve. While it mentions that the framework is designed to detect malicious packages, it does 

In [35]:

# ============================================================
# STEP 5B: INSPECT ORIGINAL PAPER PROBLEM STATEMENT
# ============================================================

print("=" * 65)
print("ORIGINAL PAPER: ABSTRACT AND INTRODUCTION")
print("=" * 65)

# Inspect the first two pages of the uploaded paper.
for page in pages[:2]:

    print("\n" + "=" * 65)
    print("PAGE:", page.metadata.get("page_label", "Unknown"))
    print("=" * 65)

    print(page.page_content[:5000])


ORIGINAL PAPER: ABSTRACT AND INTRODUCTION

PAGE: 1
eDySec: A Deep Learning-based Explainable Dynamic Analysis
Framework for Detecting Malicious Packages in PyPI Ecosystem
Sk Tanzir Mehedi∗
Queensland University of Technology
Brisbane, QLD, Australia
Raja Jurdak
Queensland University of Technology
Brisbane, QLD, Australia
Chadni Islam
Edith Cowan University
Joondalup, WA, Australia
Abu Bakar Siddique Mahi
Research Graduate School, BUBT
Dhaka, Bangladesh
Gowri Ramachandran
Queensland University of Technology
Brisbane, QLD, Australia
Abstract
The security of open-source software repositories is increasingly
threatened by next-gen software supply chain attacks. These at-
tacks include multiphase malware execution, remote access ac-
tivation, and dynamic payload generation. Traditional Machine
Learning (ML) detectors struggle to detect these attacks due to the
high-dimensional and sparse nature of dynamic behavioral data,
including system calls, network traffic, directory access patterns,
a

In [36]:

# ============================================================
# STEP 5C: CORRECTIVE RETRIEVAL FOR INSUFFICIENT EVIDENCE
# ============================================================

def corrective_retrieval(question, vectorstore, reranker, top_n=5):

    # Target missing information using alternative queries.
    corrective_queries = [
        question,
        "What limitations of existing malicious package detection methods does eDySec address?",
        "What challenges motivate the development of eDySec?",
        "What problems with dynamic behavioral data, model stability and explainability does eDySec solve?"
    ]

    all_docs = []

    # Retrieve evidence for each query.
    for query in corrective_queries:
        docs = vectorstore.similarity_search(query, k=6)
        all_docs.extend(docs)

    # Remove duplicate documents.
    unique_docs = []
    seen = set()

    for doc in all_docs:
        key = (
            doc.metadata.get("source"),
            doc.metadata.get("page"),
            doc.page_content
        )

        if key not in seen:
            seen.add(key)
            unique_docs.append(doc)

    # Rerank the retrieved evidence.
    corrected_docs = rerank_documents(
        question,
        unique_docs,
        reranker,
        top_n=top_n
    )

    print("Corrective queries:", len(corrective_queries))
    print("Unique evidence chunks:", len(unique_docs))

    return corrected_docs


# Run corrective retrieval only when evidence is insufficient.
if retrieval_status == "INSUFFICIENT":

    corrected_docs = corrective_retrieval(
        question,
        vectorstore,
        reranker,
        top_n=5
    )

    print("\nCORRECTIVE RETRIEVAL RESULTS")

    for i, (doc, score) in enumerate(corrected_docs):
        print(f"\nRank {i+1}")
        print("Page:", doc.metadata.get("page_label"))
        print("Score:", round(float(score), 4))
        print(doc.page_content[:700])

else:
    corrected_docs = reranked_docs
    print("Original evidence accepted.")


Corrective queries: 4
Unique evidence chunks: 16

CORRECTIVE RETRIEVAL RESULTS

Rank 1
Page: 2
Score: 1.421
techniques into the detection pipeline, enabling near-perfect
stable and transparent interpretation of model decisions by
identifying the features that most influence each prediction.
• Experimental results demonstrate that eDySec significantly
outperforms the state-of-the-art framework. Specifically, it
reduces feature dimensionality by 52.78% while lowering
false positives by 82% and false negatives by 79%. It also
improves accuracy by 3% and maintains a low inference
latency of 170ms per package.
This study is structured as follows: Section 2 reviews existing
methods for malicious package detection. Section 3 presents the
eDySec framework, and Section 4 describes dataset, experimental
des

Rank 2
Page: 2
Score: 1.3178
ment, and transparent interpretation of model decisions, creates a
dangerous illusion of security and highlights the need for a more
comprehensive analytical fra

In [37]:

# ============================================================
# STEP 5D: RE-EVALUATE CORRECTED EVIDENCE
# ============================================================

print("=" * 60)
print("CORRECTIVE RETRIEVAL EVALUATION")
print("=" * 60)

# Evaluate the corrected evidence using the existing grader.
corrected_status = grade_retrieval(
    question,
    corrected_docs,
    llm
)

print("Original evidence status:", retrieval_status)
print("Corrected evidence status:", corrected_status)

if corrected_status == "SUFFICIENT":
    print("Decision: Proceed to grounded answer generation.")
else:
    print("Decision: Evidence remains insufficient.")
    print("Next action: Inspect evidence coverage or abstain.")


CORRECTIVE RETRIEVAL EVALUATION
Original evidence status: INSUFFICIENT
Corrected evidence status: INSUFFICIENT
Decision: Evidence remains insufficient.
Next action: Inspect evidence coverage or abstain.


In [38]:

# ============================================================
# STEP 5E: DIAGNOSE EVIDENCE COVERAGE
# ============================================================

# Diagnostic phrases from the paper's problem statement.
# These are used only to inspect the current pipeline.
problem_phrases = [
    "Traditional Machine Learning",
    "high-dimensional and sparse",
    "false positives and false negatives",
    "explainability analysis",
    "model stability"
]

def find_problem_evidence(documents):
    matches = []

    for doc in documents:
        content = doc.page_content.lower()

        found = [
            phrase
            for phrase in problem_phrases
            if phrase.lower() in content
        ]

        if found:
            matches.append((doc, found))

    return matches


# 1. Search all indexed chunks.
corpus_matches = find_problem_evidence(chunks)

# 2. Search corrective retrieval candidates.
# Reconstruct the candidate pool for diagnosis.
corrective_queries = [
    question,
    "What limitations of existing malicious package detection methods does eDySec address?",
    "What challenges motivate the development of eDySec?",
    "What problems with dynamic behavioral data, model stability and explainability does eDySec solve?"
]

candidate_pool = []

for query in corrective_queries:
    candidate_pool.extend(
        vectorstore.similarity_search(query, k=6)
    )

candidate_matches = find_problem_evidence(candidate_pool)

# 3. Search final top-5 reranked chunks.
top_docs = [doc for doc, score in corrected_docs]
top_matches = find_problem_evidence(top_docs)

print("=" * 60)
print("EVIDENCE COVERAGE DIAGNOSIS")
print("=" * 60)

print("Matching chunks in full corpus:", len(corpus_matches))
print("Matching entries in retrieval pool:", len(candidate_matches))
print("Matching chunks in final top 5:", len(top_matches))

print("\nFINAL TOP-5 MATCH DETAILS")

for i, (doc, phrases) in enumerate(top_matches, 1):
    print(f"\nMatch {i}")
    print("Page:", doc.metadata.get("page_label"))
    print("Matched phrases:", phrases)


EVIDENCE COVERAGE DIAGNOSIS
Matching chunks in full corpus: 18
Matching entries in retrieval pool: 4
Matching chunks in final top 5: 1

FINAL TOP-5 MATCH DETAILS

Match 1
Page: 1
Matched phrases: ['model stability']


In [39]:

# ============================================================
# STEP 5F: DIVERSITY-AWARE EVIDENCE SELECTION
# ============================================================

def select_diverse_evidence(
    question,
    candidate_docs,
    reranker,
    top_n=5,
    relevance_weight=0.7
):
    """
    Select relevant evidence while reducing redundancy.

    This method uses Cross-Encoder relevance scores and
    embedding similarity to select diverse chunks.
    """

    if not candidate_docs:
        return []

    # Score all candidate documents for relevance.
    pairs = [
        (question, doc.page_content)
        for doc in candidate_docs
    ]

    scores = reranker.predict(pairs)

    # Convert scores to a comparable 0-1 scale.
    import numpy as np

    scores = np.asarray(scores, dtype=float)

    score_range = scores.max() - scores.min()

    if score_range > 0:
        normalized_scores = (
            scores - scores.min()
        ) / score_range
    else:
        normalized_scores = np.ones_like(scores)

    # Embed candidate documents for similarity comparison.
    texts = [doc.page_content for doc in candidate_docs]
    vectors = np.asarray(
        embeddings.embed_documents(texts),
        dtype=float
    )

    # Normalize vectors for cosine similarity.
    norms = np.linalg.norm(vectors, axis=1, keepdims=True)
    vectors = vectors / np.maximum(norms, 1e-12)

    selected = []
    remaining = list(range(len(candidate_docs)))

    while remaining and len(selected) < top_n:

        best_idx = None
        best_score = float("-inf")

        for idx in remaining:

            relevance = normalized_scores[idx]

            if selected:
                redundancy = max(
                    float(np.dot(vectors[idx], vectors[j]))
                    for j in selected
                )
            else:
                redundancy = 0.0

            combined_score = (
                relevance_weight * relevance
                - (1 - relevance_weight) * redundancy
            )

            if combined_score > best_score:
                best_score = combined_score
                best_idx = idx

        selected.append(best_idx)
        remaining.remove(best_idx)

    return [
        (candidate_docs[i], float(scores[i]))
        for i in selected
    ]


# Remove duplicate candidates from corrective retrieval.
unique_candidates = []
seen = set()

for doc in candidate_pool:
    key = (
        doc.metadata.get("source"),
        doc.metadata.get("page"),
        doc.page_content
    )

    if key not in seen:
        seen.add(key)
        unique_candidates.append(doc)


diverse_docs = select_diverse_evidence(
    question,
    unique_candidates,
    reranker,
    top_n=5
)

print("=" * 60)
print("DIVERSITY-AWARE EVIDENCE RESULTS")
print("=" * 60)

for i, (doc, score) in enumerate(diverse_docs, 1):
    print(f"\nRank {i}")
    print("Page:", doc.metadata.get("page_label"))
    print("Relevance score:", round(score, 4))
    print(doc.page_content[:500])


DIVERSITY-AWARE EVIDENCE RESULTS

Rank 1
Page: 2
Relevance score: 1.421
techniques into the detection pipeline, enabling near-perfect
stable and transparent interpretation of model decisions by
identifying the features that most influence each prediction.
• Experimental results demonstrate that eDySec significantly
outperforms the state-of-the-art framework. Specifically, it
reduces feature dimensionality by 52.78% while lowering
false positives by 82% and false negatives by 79%. It also
improves accuracy by 3% and maintains a low inference
latency of 170ms per pack

Rank 2
Page: 15
Relevance score: 1.2273
of the eDySec pipeline, including data handling, preprocessing, fea-
ture extraction, model training, and evaluation, were performed
in isolated and controlled environments to reduce operational risk
and ensure safe experimentation. The dataset was used exclusively
for research and defensive purposes to advance malicious pack-
age detection in software supply chains. No personal, pri

In [40]:

# ============================================================
# STEP 5G: DIAGNOSE PROBLEM-STATEMENT RETRIEVAL
# ============================================================

diagnostic_queries = [
    "Traditional machine learning detectors struggle to detect malicious packages",
    "High-dimensional sparse dynamic behavioral data challenges",
    "Limitations of existing methods for malicious PyPI package detection"
]

for i, query in enumerate(diagnostic_queries, 1):

    print("\n" + "=" * 65)
    print(f"DIAGNOSTIC QUERY {i}: {query}")
    print("=" * 65)

    docs = vectorstore.similarity_search(query, k=5)

    for j, doc in enumerate(docs, 1):

        print(f"\nResult {j}")
        print("Page:", doc.metadata.get("page_label", "Unknown"))
        print(doc.page_content[:450])



DIAGNOSTIC QUERY 1: Traditional machine learning detectors struggle to detect malicious packages

Result 1
Page: 1
Learning (ML) detectors struggle to detect these attacks due to the
high-dimensional and sparse nature of dynamic behavioral data,
including system calls, network traffic, directory access patterns,
and dependency logs. As a result, these data characteristics degrade
the performance, stability, and explainability of ML models. These
challenges have made Deep Learning (DL) a promising alternative,
given its success across various domains and its p

Result 2
Page: 1
malicious package detection. Additionally, model stability analysis
and explainable AI techniques are incorporated into the detection
pipeline to enable stable, and transparent interpretations of model
decisions. Experimental results demonstrate that eDySec signifi-
cantly outperforms the state-of-the-art frameworks. Specifically,
it halves feature dimensionality while lowering false positives by
82% and false n

In [41]:

# ============================================================
# STEP 5H: AUTOMATED CORRECTIVE QUERY GENERATION
# ============================================================

def generate_corrective_queries(question, evidence_docs, llm):

    # Use retrieved evidence to identify missing information.
    context = "\n\n".join(
        doc.page_content[:700]
        for doc, score in evidence_docs[:5]
    )

    prompt = f"""
You are a scientific information retrieval assistant.

USER QUESTION:
{question}

CURRENT RETRIEVED EVIDENCE:
{context}

The evidence has been marked as insufficient.

Generate exactly 3 alternative search queries to retrieve
the missing information needed to answer the user question.

INSTRUCTIONS:
- Focus on the underlying research problem and motivation.
- Use terminology supported by the evidence.
- Do not invent facts or assume missing information is true.
- Avoid searching for performance results unless required.
- Do not answer the question.
- Output only 3 search queries, one per line.
- Do not number the queries.

SEARCH QUERIES:
"""

    response = llm.invoke(prompt)

    queries = [
        line.strip().lstrip("-").strip()
        for line in response.splitlines()
        if line.strip()
    ]

    return [question] + queries[:3]


# Test automated corrective query generation.
auto_queries = generate_corrective_queries(
    question,
    corrected_docs,
    llm
)

print("=" * 60)
print("AUTOMATED CORRECTIVE SEARCH QUERIES")
print("=" * 60)

for i, query in enumerate(auto_queries, 1):
    print(f"{i}. {query}")


AUTOMATED CORRECTIVE SEARCH QUERIES
1. What problem does eDySec aim to solve?
2. What are the challenges in malicious package detection that eDySec aims to address?
3. What is the current limitation of existing methods for malicious package detection that eDySec seeks to overcome?
4. How does the lack of transparency and interpretability in existing malicious package detection methods create security risks?


In [42]:

# ============================================================
# STEP 5I: EXECUTE AUTOMATED CORRECTIVE RETRIEVAL
# ============================================================

def automated_corrective_retrieval(
    question,
    search_queries,
    vectorstore,
    reranker,
    top_n=5
):
    all_docs = []

    # Retrieve evidence for each generated query.
    for query in search_queries:
        retrieved = vectorstore.similarity_search(
            query,
            k=6
        )
        all_docs.extend(retrieved)

    # Remove duplicate evidence chunks.
    unique_docs = []
    seen = set()

    for doc in all_docs:
        key = (
            doc.metadata.get("source"),
            doc.metadata.get("page"),
            doc.page_content
        )

        if key not in seen:
            seen.add(key)
            unique_docs.append(doc)

    # Rank documents against the original question.
    ranked_docs = rerank_documents(
        question,
        unique_docs,
        reranker,
        top_n=top_n
    )

    return ranked_docs, unique_docs


auto_corrected_docs, auto_candidate_docs = (
    automated_corrective_retrieval(
        question,
        auto_queries,
        vectorstore,
        reranker,
        top_n=5
    )
)

print("=" * 60)
print("AUTOMATED CORRECTIVE RETRIEVAL RESULTS")
print("=" * 60)

print("Generated queries:", len(auto_queries))
print("Unique candidate chunks:", len(auto_candidate_docs))

for i, (doc, score) in enumerate(auto_corrected_docs, 1):
    print(f"\nRank {i}")
    print("Page:", doc.metadata.get("page_label"))
    print("Relevance score:", round(float(score), 4))
    print(doc.page_content[:500])


AUTOMATED CORRECTIVE RETRIEVAL RESULTS
Generated queries: 4
Unique candidate chunks: 16

Rank 1
Page: 2
Relevance score: 1.421
techniques into the detection pipeline, enabling near-perfect
stable and transparent interpretation of model decisions by
identifying the features that most influence each prediction.
• Experimental results demonstrate that eDySec significantly
outperforms the state-of-the-art framework. Specifically, it
reduces feature dimensionality by 52.78% while lowering
false positives by 82% and false negatives by 79%. It also
improves accuracy by 3% and maintains a low inference
latency of 170ms per pack

Rank 2
Page: 2
Relevance score: 1.3178
ment, and transparent interpretation of model decisions, creates a
dangerous illusion of security and highlights the need for a more
comprehensive analytical framework.
To address these challenges, this study presents eDySec, an effi-
cient, stable, and explainable DL-basedextendedDynamicSecurity
analysis framework for detecting m

In [43]:

# ============================================================
# STEP 5 FIX: CORRECTED EVIDENCE SELECTION
# ============================================================

def build_corrected_evidence(
    question,
    original_docs,
    corrective_candidates,
    vectorstore,
    reranker,
    top_n=7
):
    # Include the original retrieved evidence.
    all_docs = [doc for doc, score in original_docs]

    # Include corrective retrieval candidates.
    all_docs.extend(corrective_candidates)

    # Retrieve additional evidence for the original question.
    all_docs.extend(
        vectorstore.similarity_search(question, k=10)
    )

    # Remove duplicate chunks.
    unique_docs = []
    seen = set()

    for doc in all_docs:
        key = (
            doc.metadata.get("source"),
            doc.metadata.get("page"),
            doc.page_content
        )

        if key not in seen:
            seen.add(key)
            unique_docs.append(doc)

    # Cross-Encoder reranking.
    ranked_docs = rerank_documents(
        question,
        unique_docs,
        reranker,
        top_n=len(unique_docs)
    )

    # Prefer evidence with positive relevance scores.
    # These are raw model scores, not probabilities.
    positive_docs = [
        (doc, score)
        for doc, score in ranked_docs
        if float(score) > 0
    ]

    # Fall back to top-ranked evidence if none scores positive.
    if not positive_docs:
        positive_docs = ranked_docs

    # Select the strongest evidence chunks.
    final_docs = positive_docs[:top_n]

    return final_docs


fixed_docs = build_corrected_evidence(
    question=question,
    original_docs=reranked_docs,
    corrective_candidates=auto_candidate_docs,
    vectorstore=vectorstore,
    reranker=reranker,
    top_n=7
)

print("=" * 60)
print("CORRECTED EVIDENCE SELECTION")
print("=" * 60)

for i, (doc, score) in enumerate(fixed_docs, 1):
    print(f"\nRank {i}")
    print("Page:", doc.metadata.get("page_label"))
    print("Score:", round(float(score), 4))
    print(doc.page_content[:500])


CORRECTED EVIDENCE SELECTION

Rank 1
Page: 2
Score: 1.421
techniques into the detection pipeline, enabling near-perfect
stable and transparent interpretation of model decisions by
identifying the features that most influence each prediction.
• Experimental results demonstrate that eDySec significantly
outperforms the state-of-the-art framework. Specifically, it
reduces feature dimensionality by 52.78% while lowering
false positives by 82% and false negatives by 79%. It also
improves accuracy by 3% and maintains a low inference
latency of 170ms per pack

Rank 2
Page: 2
Score: 1.3178
ment, and transparent interpretation of model decisions, creates a
dangerous illusion of security and highlights the need for a more
comprehensive analytical framework.
To address these challenges, this study presents eDySec, an effi-
cient, stable, and explainable DL-basedextendedDynamicSecurity
analysis framework for detecting malicious packages. Figure 1 out-
lines the overall system architecture of the p

In [44]:

# ============================================================
# STEP 5: VERIFY CORRECTED EVIDENCE
# ============================================================

fixed_status = grade_retrieval(
    question,
    fixed_docs,
    llm
)

print("=" * 60)
print("FINAL EVIDENCE VERIFICATION")
print("=" * 60)

print("Question:", question)
print("Original status:", retrieval_status)
print("Corrected status:", fixed_status)

if fixed_status == "SUFFICIENT":
    print("Grader decision: Evidence is sufficient.")
    print("Next: Manually verify supporting passages.")
else:
    print("Grader decision: Evidence is still insufficient.")
    print("Next: Inspect grader output and evidence coverage.")


FINAL EVIDENCE VERIFICATION
Question: What problem does eDySec aim to solve?
Original status: INSUFFICIENT
Corrected status: INSUFFICIENT
Grader decision: Evidence is still insufficient.
Next: Inspect grader output and evidence coverage.


In [45]:

# ============================================================
# STEP 5K: IMPROVED EVIDENCE SUFFICIENCY GRADER
# ============================================================

import json
import re

def grade_evidence_v2(question, evidence_docs, llm):

    if not evidence_docs:
        return {
            "decision": "INSUFFICIENT",
            "reason": "No evidence retrieved.",
            "supporting_evidence": [],
            "raw_response": ""
        }

    context = "\n\n".join(
        f"[Evidence {i + 1} | Page "
        f"{doc.metadata.get('page_label', 'Unknown')}]\n"
        f"{doc.page_content}"
        for i, (doc, score) in enumerate(evidence_docs)
    )

    prompt = f"""
You are an evidence sufficiency evaluator for scientific
question answering.

QUESTION:
{question}

RETRIEVED EVIDENCE:
{context}

Evaluate whether the provided evidence is sufficient to
answer the question accurately.

Rules:
1. Use ONLY the retrieved evidence.
2. Judge whether the main question can be answered.
3. Do not require unrelated experimental details.
4. Evidence may be distributed across multiple passages.
5. Do not assume missing information.
6. If evidence supports only part of the required answer,
   return INSUFFICIENT.
7. Do not generate a full answer.

Return ONLY a valid JSON object:

{{
  "decision": "SUFFICIENT or INSUFFICIENT",
  "reason": "Brief explanation",
  "supporting_evidence": [1, 2]
}}
"""

    raw_response = llm.invoke(prompt).strip()

    try:
        # Allow JSON enclosed in a Markdown code fence.
        cleaned = re.sub(
            r"^```(?:json)?\\s*|\\s*```$",
            "",
            raw_response,
            flags=re.IGNORECASE
        ).strip()

        result = json.loads(cleaned)

        decision = str(
            result.get("decision", "")
        ).strip().upper()

        if decision not in ["SUFFICIENT", "INSUFFICIENT"]:
            decision = "UNKNOWN"

        supporting = result.get("supporting_evidence", [])

        if not isinstance(supporting, list):
            supporting = []

        valid_supporting = [
            i for i in supporting
            if type(i) is int
            and 1 <= i <= len(evidence_docs)
        ]

        return {
            "decision": decision,
            "reason": str(result.get("reason", "")),
            "supporting_evidence": valid_supporting,
            "raw_response": raw_response
        }

    except (json.JSONDecodeError, TypeError, ValueError):
        return {
            "decision": "UNKNOWN",
            "reason": "Unable to parse grader response.",
            "supporting_evidence": [],
            "raw_response": raw_response
        }


# Test the improved grader on the corrected evidence.
grade_result = grade_evidence_v2(
    question,
    fixed_docs,
    llm
)

print("=" * 60)
print("IMPROVED EVIDENCE SUFFICIENCY EVALUATION")
print("=" * 60)

print("Question:", question)
print("Decision:", grade_result["decision"])
print("Reason:", grade_result["reason"])
print(
    "Supporting evidence:",
    grade_result["supporting_evidence"]
)

if grade_result["decision"] == "UNKNOWN":
    print("\nRaw model response:")
    print(grade_result["raw_response"])


IMPROVED EVIDENCE SUFFICIENCY EVALUATION
Question: What problem does eDySec aim to solve?
Decision: SUFFICIENT
Reason: The evidence clearly states that eDySec aims to solve the problem of malicious package detection, and provides details on its architecture and performance.
Supporting evidence: [1, 2, 4]


# Grounded Answer Generation

In [46]:

# ============================================================
# STEP 6A: GROUNDED ANSWER GENERATION
# ============================================================

def generate_grounded_answer(question, evidence_docs, grade_result, llm):

    # Stop if evidence is insufficient or grading is uncertain.
    if grade_result["decision"] != "SUFFICIENT":
        return {
            "status": "ABSTAINED",
            "answer": "Insufficient verified evidence to answer.",
            "sources": []
        }

    # Build context with evidence IDs and page numbers.
    context_parts = []
    source_pages = set()

    for i, (doc, score) in enumerate(evidence_docs, 1):

        page = doc.metadata.get("page_label", "Unknown")

        context_parts.append(
            f"[Evidence {i} | Page {page}]\n"
            f"{doc.page_content}"
        )

        source_pages.add(str(page))

    context = "\n\n".join(context_parts)

    prompt = f"""
You are a scientific question-answering assistant.

Answer the question using ONLY the evidence provided.

QUESTION:
{question}

EVIDENCE:
{context}

STRICT RULES:
1. Use only information explicitly supported by the evidence.
2. Do not use external knowledge.
3. Do not invent facts, statistics, or explanations.
4. Explain the answer clearly and concisely.
5. Cite supporting evidence using [Evidence N, Page X].
6. Only cite evidence that actually supports the statement.
7. If the evidence cannot support an answer, respond:
   INSUFFICIENT EVIDENCE
8. Do not include unsupported claims.

ANSWER:
"""

    answer = llm.invoke(prompt).strip()

    if "INSUFFICIENT EVIDENCE" in answer.upper():
        return {
            "status": "ABSTAINED",
            "answer": "Insufficient evidence to answer.",
            "sources": []
        }

    return {
        "status": "GENERATED_UNVERIFIED",
        "answer": answer,
        "sources": sorted(source_pages)
    }


# Generate an answer using Step 5K evidence.
answer_result = generate_grounded_answer(
    question=question,
    evidence_docs=fixed_docs,
    grade_result=grade_result,
    llm=llm
)

print("=" * 60)
print("GROUNDED ANSWER GENERATION")
print("=" * 60)

print("Question:", question)
print("Status:", answer_result["status"])

print("\nGenerated Answer:")
print(answer_result["answer"])

print("\nRetrieved source pages:")
print(answer_result["sources"])


GROUNDED ANSWER GENERATION
Question: What problem does eDySec aim to solve?
Status: GENERATED_UNVERIFIED

Generated Answer:
Based on the provided evidence, the problem that eDySec aims to solve is:

**Malicious package detection in the software supply chain, specifically in the PyPI ecosystem.**

This is supported by the following evidence:

* [Evidence 1, Page 2]: "This study presents eDySec, an efficient, stable, and explainable DL-based extended Dynamic Security analysis framework for detecting malicious packages."
* [Evidence 4, Page 1]: "malicious package detection. Additionally, model stability analysis and explainable AI techniques are incorporated into the detection pipeline..."
* [Evidence 5, Page 12]: "This study presents eDySec, an efficient, stable, and explainable DL-based framework for malicious package detection using the dynamic QUT-DV25 dataset."
* [Evidence 6, Page 10]: "Computational efficiency further strengthens this conclusion. Table 5 shows that eDySec achieves a

#Atomic Claim Extraction

In [47]:

# ============================================================
# STEP 7A: ATOMIC CLAIM EXTRACTION
# ============================================================

import json
import re

def extract_atomic_claims(answer, llm):

    prompt = f"""
You are a scientific claim extraction assistant.

Extract the factual claims from the answer below.

ANSWER:
{answer}

RULES:
1. Extract factual claims only.
2. Each claim must express one independently verifiable fact.
3. Split compound statements into separate claims.
4. Preserve the original meaning.
5. Do not introduce new information.
6. Ignore citation markers and introductory phrases.
7. Do not evaluate whether claims are true.
8. Return only a JSON array of strings.

Example:
[
  "The framework detects malicious packages.",
  "The framework uses dynamic behavioral analysis."
]

CLAIMS:
"""

    raw_response = llm.invoke(prompt).strip()

    # Remove optional Markdown code fences.
    cleaned = re.sub(
        r"^```(?:json)?\s*|\s*```$",
        "",
        raw_response,
        flags=re.IGNORECASE
    ).strip()

    try:
        claims = json.loads(cleaned)

        if not isinstance(claims, list):
            raise ValueError("Expected a JSON array.")

        claims = [
            claim.strip()
            for claim in claims
            if isinstance(claim, str) and claim.strip()
        ]

        return claims

    except (json.JSONDecodeError, ValueError):
        print("Claim extraction could not be parsed.")
        print("Raw response:", raw_response)
        return []


# Extract claims from the generated answer.
atomic_claims = extract_atomic_claims(
    answer_result["answer"],
    llm
)

print("=" * 60)
print("ATOMIC CLAIM EXTRACTION")
print("=" * 60)

print("Number of claims:", len(atomic_claims))

for i, claim in enumerate(atomic_claims, 1):
    print(f"\nClaim {i}: {claim}")


Claim extraction could not be parsed.
Raw response: Here are the extracted factual claims in JSON format:

[
  "The problem that eDySec aims to solve is malicious package detection in the software supply chain.",
  "The problem that eDySec aims to solve is specifically in the PyPI ecosystem.",
  "eDySec is an efficient framework.",
  "eDySec is a stable framework.",
  "eDySec is an explainable DL-based framework.",
  "eDySec uses dynamic behavioral analysis.",
  "eDySec uses the dynamic QUT-DV25 dataset.",
  "eDySec achieves a balance between predictive performance and computational cost compared with the DySec baselines.",
  "Computational efficiency is a strength of eDySec."
]
ATOMIC CLAIM EXTRACTION
Number of claims: 0


In [48]:

# ============================================================
# STEP 7A FIX: ROBUST ATOMIC CLAIM EXTRACTION
# ============================================================

import json
import re

def extract_atomic_claims(answer, llm):

    prompt = f"""
You are a scientific claim extraction assistant.

Extract independently verifiable factual claims.

ANSWER:
{answer}

RULES:
1. Extract factual claims only.
2. Each claim must contain one verifiable fact.
3. Split compound statements.
4. Preserve the original meaning.
5. Do not invent new information.
6. Ignore citation markers and introductory phrases.
7. Do not evaluate whether claims are true.
8. Return a JSON array of strings.

CLAIMS:
"""

    raw_response = llm.invoke(prompt).strip()

    # Locate a JSON array even when the model adds extra text.
    decoder = json.JSONDecoder()
    claims = None

    for match in re.finditer(r"\[", raw_response):
        try:
            parsed, _ = decoder.raw_decode(
                raw_response[match.start():]
            )

            if (
                isinstance(parsed, list)
                and all(isinstance(item, str) for item in parsed)
            ):
                claims = parsed
                break

        except json.JSONDecodeError:
            continue

    if claims is None:
        print("ERROR: No valid JSON claim array found.")
        print("Raw response:", raw_response)
        return []

    # Remove empty and duplicate claims.
    cleaned_claims = []
    seen = set()

    for claim in claims:
        claim = claim.strip()
        key = claim.casefold()

        if claim and key not in seen:
            cleaned_claims.append(claim)
            seen.add(key)

    return cleaned_claims


# Run corrected claim extraction.
atomic_claims = extract_atomic_claims(
    answer_result["answer"],
    llm
)

print("=" * 60)
print("CORRECTED ATOMIC CLAIM EXTRACTION")
print("=" * 60)

print("Number of claims:", len(atomic_claims))

for i, claim in enumerate(atomic_claims, 1):
    print(f"\nClaim {i}: {claim}")


CORRECTED ATOMIC CLAIM EXTRACTION
Number of claims: 4

Claim 1: This study presents eDySec, an efficient, stable, and explainable DL-based extended Dynamic Security analysis framework for detecting malicious packages.

Claim 2: malicious package detection. Additionally, model stability analysis and explainable AI techniques are incorporated into the detection pipeline...

Claim 3: This study presents eDySec, an efficient, stable, and explainable DL-based framework for malicious package detection using the dynamic QUT-DV25 dataset.

Claim 4: Computational efficiency further strengthens this conclusion. Table 5 shows that eDySec achieves a balance between predictive performance and computational cost compared with the DySec baselines.


In [49]:

# ============================================================
# STEP 7B: CLAIM-LEVEL EVIDENCE VERIFICATION
# ============================================================

import json
import re

def parse_json_object(raw_response):
    decoder = json.JSONDecoder()

    for match in re.finditer(r"\{", raw_response):
        try:
            result, _ = decoder.raw_decode(
                raw_response[match.start():]
            )

            if isinstance(result, dict):
                return result

        except json.JSONDecodeError:
            continue

    return None


def verify_claim(claim, evidence_docs, llm):

    context = "\n\n".join(
        f"[Evidence {i} | Page "
        f"{doc.metadata.get('page_label', 'Unknown')}]\n"
        f"{doc.page_content}"
        for i, (doc, score) in enumerate(evidence_docs, 1)
    )

    prompt = f"""
You are a scientific evidence verification assistant.

CLAIM:
{claim}

RETRIEVED EVIDENCE:
{context}

TASK:
Determine whether the evidence supports or contradicts
the claim.

RULES:
1. Use ONLY the provided evidence.
2. SUPPORTED means the evidence directly supports the claim.
3. CONTRADICTED means the evidence explicitly conflicts
   with the claim.
4. INSUFFICIENT means neither support nor contradiction
   can be established.
5. Do not use external knowledge.
6. Do not assume a claim is false because evidence is missing.
7. Identify the evidence IDs supporting your decision.
8. Return only a JSON object.

OUTPUT FORMAT:
{{
    "decision": "SUPPORTED",
    "reason": "Brief evidence-based explanation",
    "evidence_ids": [1, 2]
}}
"""

    raw_response = llm.invoke(prompt).strip()
    result = parse_json_object(raw_response)

    if result is None:
        return {
            "decision": "UNKNOWN",
            "reason": "Unable to parse verification response.",
            "evidence_ids": [],
            "raw_response": raw_response
        }

    decision = str(
        result.get("decision", "")
    ).strip().upper()

    allowed = {
        "SUPPORTED",
        "CONTRADICTED",
        "INSUFFICIENT"
    }

    if decision not in allowed:
        decision = "UNKNOWN"

    evidence_ids = result.get("evidence_ids", [])

    if not isinstance(evidence_ids, list):
        evidence_ids = []

    evidence_ids = [
        i for i in evidence_ids
        if type(i) is int
        and 1 <= i <= len(evidence_docs)
    ]

    # A supported or contradicted claim must cite evidence.
    if decision in {"SUPPORTED", "CONTRADICTED"} and not evidence_ids:
        decision = "UNKNOWN"

    return {
        "decision": decision,
        "reason": str(result.get("reason", "")),
        "evidence_ids": evidence_ids,
        "raw_response": raw_response
    }


# Verify every extracted claim.
verification_results = []

print("=" * 60)
print("CLAIM-LEVEL EVIDENCE VERIFICATION")
print("=" * 60)

for i, claim in enumerate(atomic_claims, 1):

    result = verify_claim(
        claim,
        fixed_docs,
        llm
    )

    verification_results.append({
        "claim": claim,
        **result
    })

    print(f"\nClaim {i}: {claim}")
    print("Decision:", result["decision"])
    print("Reason:", result["reason"])
    print("Evidence IDs:", result["evidence_ids"])

    if result["decision"] == "UNKNOWN":
        print("Raw response:", result["raw_response"])


CLAIM-LEVEL EVIDENCE VERIFICATION

Claim 1: This study presents eDySec, an efficient, stable, and explainable DL-based extended Dynamic Security analysis framework for detecting malicious packages.
Decision: SUPPORTED
Reason: The evidence directly supports the claim by presenting experimental results that demonstrate eDySec's efficiency, stability, and explainability in detecting malicious packages.
Evidence IDs: [1, 2, 3, 4, 5, 6, 7]

Claim 2: malicious package detection. Additionally, model stability analysis and explainable AI techniques are incorporated into the detection pipeline...
Decision: SUPPORTED
Reason: The evidence directly supports the claim by demonstrating the effectiveness of eDySec in detecting malicious packages, improving accuracy, reducing false positives and negatives, and maintaining a low inference latency.
Evidence IDs: [1, 4, 5]

Claim 3: This study presents eDySec, an efficient, stable, and explainable DL-based framework for malicious package detection using 

In [50]:

# ============================================================
# STEP 7C: VERIFICATION SUMMARY AND FINAL DECISION
# ============================================================

from collections import Counter

def summarize_verification(verification_results):

    counts = Counter(
        result["decision"]
        for result in verification_results
    )

    total = len(verification_results)
    supported = counts["SUPPORTED"]
    contradicted = counts["CONTRADICTED"]
    insufficient = counts["INSUFFICIENT"]
    unknown = counts["UNKNOWN"]

    if total == 0:
        final_decision = "ABSTAIN"

    elif contradicted > 0:
        final_decision = "REVISE"

    elif insufficient > 0 or unknown > 0:
        final_decision = "ABSTAIN"

    elif supported == total:
        final_decision = "ACCEPT_FOR_REVIEW"

    else:
        final_decision = "ABSTAIN"

    return {
        "total_claims": total,
        "supported": supported,
        "contradicted": contradicted,
        "insufficient": insufficient,
        "unknown": unknown,
        "support_rate": supported / total if total else 0.0,
        "final_decision": final_decision
    }


verification_summary = summarize_verification(
    verification_results
)

print("=" * 60)
print("CLAIM VERIFICATION SUMMARY")
print("=" * 60)

for key, value in verification_summary.items():
    print(f"{key}: {value}")


CLAIM VERIFICATION SUMMARY
total_claims: 4
supported: 4
contradicted: 0
insufficient: 0
unknown: 0
support_rate: 1.0
final_decision: ACCEPT_FOR_REVIEW


In [51]:

# ============================================================
# STEP 7D: INSPECT CLAIM-TO-EVIDENCE CITATIONS
# ============================================================

def inspect_claim_citations(verification_results, evidence_docs):

    print("=" * 60)
    print("CLAIM-TO-EVIDENCE CITATION INSPECTION")
    print("=" * 60)

    for claim_number, result in enumerate(
        verification_results, 1
    ):
        print(f"\n{'=' * 60}")
        print(f"CLAIM {claim_number}")
        print(result["claim"])
        print("Verifier decision:", result["decision"])

        evidence_ids = result.get("evidence_ids", [])

        if not evidence_ids:
            print("No supporting citations provided.")
            continue

        for evidence_id in evidence_ids:

            if not 1 <= evidence_id <= len(evidence_docs):
                print("Invalid evidence ID:", evidence_id)
                continue

            doc, score = evidence_docs[evidence_id - 1]

            print(f"\n  Evidence {evidence_id}")
            print("  Page:", doc.metadata.get("page_label"))
            print("  Passage:")
            print(doc.page_content[:600])


inspect_claim_citations(
    verification_results,
    fixed_docs
)


CLAIM-TO-EVIDENCE CITATION INSPECTION

CLAIM 1
This study presents eDySec, an efficient, stable, and explainable DL-based extended Dynamic Security analysis framework for detecting malicious packages.
Verifier decision: SUPPORTED

  Evidence 1
  Page: 2
  Passage:
techniques into the detection pipeline, enabling near-perfect
stable and transparent interpretation of model decisions by
identifying the features that most influence each prediction.
• Experimental results demonstrate that eDySec significantly
outperforms the state-of-the-art framework. Specifically, it
reduces feature dimensionality by 52.78% while lowering
false positives by 82% and false negatives by 79%. It also
improves accuracy by 3% and maintains a low inference
latency of 170ms per package.
This study is structured as follows: Section 2 reviews existing
methods for malicious package d

  Evidence 2
  Page: 2
  Passage:
ment, and transparent interpretation of model decisions, creates a
dangerous illusion of security a

In [52]:

# ============================================================
# STEP 7E: INDIVIDUAL CITATION VALIDATION
# ============================================================

def validate_claim_citations(
    verification_results,
    evidence_docs,
    llm
):
    validated_results = []

    for claim_number, result in enumerate(
        verification_results, 1
    ):
        claim = result["claim"]
        valid_ids = []
        citation_checks = []

        for evidence_id in result.get("evidence_ids", []):

            # Verify against ONE passage at a time.
            doc, score = evidence_docs[evidence_id - 1]

            prompt = f"""
You are a strict scientific citation validator.

CLAIM:
{claim}

EVIDENCE PASSAGE:
{doc.page_content}

TASK:
Does this single passage directly support ALL factual
parts of the claim?

RULES:
- Use only this passage.
- Do not use external knowledge.
- Do not infer facts from the paper title alone.
- If the passage supports only part of the claim,
  return INSUFFICIENT.
- If the passage contradicts the claim,
  return CONTRADICTED.
- Otherwise, return SUPPORTED.
- Return exactly one label:
  SUPPORTED, CONTRADICTED, or INSUFFICIENT.

DECISION:
"""

            raw = llm.invoke(prompt).strip().upper()

            import re
            match = re.search(
                r"\b(SUPPORTED|CONTRADICTED|INSUFFICIENT)\b",
                raw
            )

            decision = (
                match.group(1)
                if match else "UNKNOWN"
            )

            citation_checks.append({
                "evidence_id": evidence_id,
                "page": doc.metadata.get("page_label"),
                "decision": decision
            })

            if decision == "SUPPORTED":
                valid_ids.append(evidence_id)

        validated_results.append({
            "claim": claim,
            "original_decision": result["decision"],
            "validated_evidence_ids": valid_ids,
            "citation_checks": citation_checks,
            "citation_validated": bool(valid_ids)
        })

        print(f"\nClaim {claim_number}")
        print("Validated evidence IDs:", valid_ids)
        print("Citation checks:", citation_checks)

    return validated_results


validated_citations = validate_claim_citations(
    verification_results,
    fixed_docs,
    llm
)



Claim 1
Validated evidence IDs: []
Citation checks: [{'evidence_id': 1, 'page': '2', 'decision': 'INSUFFICIENT'}, {'evidence_id': 2, 'page': '2', 'decision': 'INSUFFICIENT'}, {'evidence_id': 3, 'page': '9', 'decision': 'INSUFFICIENT'}, {'evidence_id': 4, 'page': '1', 'decision': 'INSUFFICIENT'}, {'evidence_id': 5, 'page': '12', 'decision': 'INSUFFICIENT'}, {'evidence_id': 6, 'page': '10', 'decision': 'INSUFFICIENT'}, {'evidence_id': 7, 'page': '15', 'decision': 'INSUFFICIENT'}]

Claim 2
Validated evidence IDs: []
Citation checks: [{'evidence_id': 1, 'page': '2', 'decision': 'INSUFFICIENT'}, {'evidence_id': 4, 'page': '1', 'decision': 'INSUFFICIENT'}, {'evidence_id': 5, 'page': '12', 'decision': 'INSUFFICIENT'}]

Claim 3
Validated evidence IDs: []
Citation checks: [{'evidence_id': 1, 'page': '2', 'decision': 'INSUFFICIENT'}, {'evidence_id': 2, 'page': '2', 'decision': 'INSUFFICIENT'}, {'evidence_id': 4, 'page': '1', 'decision': 'INSUFFICIENT'}, {'evidence_id': 5, 'page': '12', 'decisio

In [53]:

# ============================================================
# STEP 7E DEBUG: CHECK RAW LLM VERIFICATION RESPONSE
# ============================================================

claim = verification_results[3]["claim"]

# Evidence 5 corresponds to index 4.
doc, score = fixed_docs[4]

prompt = f"""
You are a strict scientific evidence validator.

CLAIM:
{claim}

EVIDENCE:
{doc.page_content}

Does the evidence directly support the claim?

Return exactly one word:
SUPPORTED
CONTRADICTED
INSUFFICIENT

Decision:
"""

raw_response = llm.invoke(prompt)

print("=" * 60)
print("CITATION VALIDATION DEBUG")
print("=" * 60)

print("\nCLAIM:")
print(claim)

print("\nEVIDENCE PAGE:")
print(doc.metadata.get("page_label"))

print("\nRAW LLM RESPONSE:")
print(repr(raw_response))


CITATION VALIDATION DEBUG

CLAIM:
Computational efficiency further strengthens this conclusion. Table 5 shows that eDySec achieves a balance between predictive performance and computational cost compared with the DySec baselines.

EVIDENCE PAGE:
12

RAW LLM RESPONSE:
'INSUFFICIENT\n\nThe claim mentions "computational efficiency" and "Table 5", but the provided evidence does not mention Table 5 or provide any specific information about computational efficiency. The evidence does discuss the performance of eDySec in terms of accuracy, false positives, false negatives, and inference latency, but it does not directly address the claim about computational efficiency.'


In [54]:

# ============================================================
# STEP 7E FIX: CONSISTENT CITATION VALIDATION
# ============================================================

def validate_claim_citations(
    verification_results,
    evidence_docs,
    llm
):
    validated_results = []

    for claim_number, result in enumerate(
        verification_results, 1
    ):
        claim = result["claim"]
        valid_ids = []
        citation_checks = []

        for evidence_id in result.get("evidence_ids", []):

            if (
                type(evidence_id) is not int
                or not 1 <= evidence_id <= len(evidence_docs)
            ):
                continue

            doc, score = evidence_docs[evidence_id - 1]

            prompt = f"""
You are a strict scientific evidence validator.

CLAIM:
{claim}

EVIDENCE:
{doc.page_content}

Does the evidence directly support the claim?

Return exactly one word:
SUPPORTED
CONTRADICTED
INSUFFICIENT

Decision:
"""

            raw_response = llm.invoke(prompt).strip().upper()

            # Accept only an exact decision.
            if raw_response in {
                "SUPPORTED",
                "CONTRADICTED",
                "INSUFFICIENT"
            }:
                decision = raw_response
            else:
                decision = "UNKNOWN"

            citation_checks.append({
                "evidence_id": evidence_id,
                "page": doc.metadata.get("page_label"),
                "decision": decision
            })

            if decision == "SUPPORTED":
                valid_ids.append(evidence_id)

        validated_results.append({
            "claim": claim,
            "original_decision": result["decision"],
            "validated_evidence_ids": valid_ids,
            "citation_checks": citation_checks,
            "citation_validated": bool(valid_ids)
        })

        print(f"\nClaim {claim_number}")
        print("Validated evidence IDs:", valid_ids)

        for check in citation_checks:
            print(
                f"  Evidence {check['evidence_id']} "
                f"(Page {check['page']}): "
                f"{check['decision']}"
            )

    return validated_results


validated_citations = validate_claim_citations(
    verification_results,
    fixed_docs,
    llm
)



Claim 1
Validated evidence IDs: [1, 4, 5, 6, 7]
  Evidence 1 (Page 2): SUPPORTED
  Evidence 2 (Page 2): UNKNOWN
  Evidence 3 (Page 9): UNKNOWN
  Evidence 4 (Page 1): SUPPORTED
  Evidence 5 (Page 12): SUPPORTED
  Evidence 6 (Page 10): SUPPORTED
  Evidence 7 (Page 15): SUPPORTED

Claim 2
Validated evidence IDs: [1, 4]
  Evidence 1 (Page 2): SUPPORTED
  Evidence 4 (Page 1): SUPPORTED
  Evidence 5 (Page 12): UNKNOWN

Claim 3
Validated evidence IDs: [1, 4]
  Evidence 1 (Page 2): SUPPORTED
  Evidence 2 (Page 2): UNKNOWN
  Evidence 4 (Page 1): SUPPORTED
  Evidence 5 (Page 12): UNKNOWN

Claim 4
Validated evidence IDs: []
  Evidence 6 (Page 10): UNKNOWN


In [55]:

# ============================================================
# STEP 7F: CLAIM FILTERING AND EVIDENCE CHECKS
# ============================================================

from collections import Counter

def filter_verified_claims(validated_citations):

    filtered_claims = []

    for number, result in enumerate(validated_citations, 1):

        checks = result.get("citation_checks", [])

        supported_ids = sorted(set(
            c["evidence_id"] for c in checks
            if c["decision"] == "SUPPORTED"
        ))

        contradicted_ids = sorted(set(
            c["evidence_id"] for c in checks
            if c["decision"] == "CONTRADICTED"
        ))

        unknown_ids = sorted(set(
            c["evidence_id"] for c in checks
            if c["decision"] == "UNKNOWN"
        ))

        insufficient_ids = sorted(set(
            c["evidence_id"] for c in checks
            if c["decision"] == "INSUFFICIENT"
        ))

        if contradicted_ids:
            status = "CONFLICT_REVIEW"

        elif supported_ids:
            status = "PROVISIONALLY_SUPPORTED"

        else:
            status = "NEEDS_EVIDENCE"

        filtered_claims.append({
            "claim_number": number,
            "claim": result["claim"],
            "status": status,
            "supported_evidence_ids": supported_ids,
            "contradicted_evidence_ids": contradicted_ids,
            "unknown_evidence_ids": unknown_ids,
            "insufficient_evidence_ids": insufficient_ids
        })

    return filtered_claims


filtered_claims = filter_verified_claims(
    validated_citations
)

print("=" * 60)
print("STEP 7F: CLAIM FILTERING RESULTS")
print("=" * 60)

for item in filtered_claims:

    print(f"\nClaim {item['claim_number']}")
    print("Status:", item["status"])
    print("Supported:", item["supported_evidence_ids"])
    print("Unknown:", item["unknown_evidence_ids"])
    print("Contradicted:", item["contradicted_evidence_ids"])

counts = Counter(
    item["status"] for item in filtered_claims
)

print("\n" + "=" * 60)
print("SUMMARY")
print("=" * 60)
print("Total claims:", len(filtered_claims))
print("Provisionally supported:",
      counts["PROVISIONALLY_SUPPORTED"])
print("Needs evidence:", counts["NEEDS_EVIDENCE"])
print("Conflict review:", counts["CONFLICT_REVIEW"])


STEP 7F: CLAIM FILTERING RESULTS

Claim 1
Status: PROVISIONALLY_SUPPORTED
Supported: [1, 4, 5, 6, 7]
Unknown: [2, 3]
Contradicted: []

Claim 2
Status: PROVISIONALLY_SUPPORTED
Supported: [1, 4]
Unknown: [5]
Contradicted: []

Claim 3
Status: PROVISIONALLY_SUPPORTED
Supported: [1, 4]
Unknown: [2, 5]
Contradicted: []

Claim 4
Status: NEEDS_EVIDENCE
Supported: []
Unknown: [6]
Contradicted: []

SUMMARY
Total claims: 4
Provisionally supported: 3
Needs evidence: 1
Conflict review: 0


In [56]:

# ============================================================
# STEP 7G: SELECTIVE CORRECTIVE RETRIEVAL
# ============================================================

def retrieve_claim_evidence(
    claim,
    vectorstore,
    reranker,
    k=8,
    top_n=4
):
    candidate_docs = vectorstore.similarity_search(
        claim,
        k=k
    )

    if not candidate_docs:
        return []

    # Reuse the existing cross-encoder reranking function.
    return rerank_documents(
        claim,
        candidate_docs,
        reranker,
        top_n=top_n
    )


def run_selective_retrieval(
    filtered_claims,
    vectorstore,
    reranker
):
    corrective_evidence = {}

    targets = [
        item for item in filtered_claims
        if item["status"] == "NEEDS_EVIDENCE"
    ]

    print("Claims requiring corrective retrieval:", len(targets))

    for item in targets:
        number = item["claim_number"]

        evidence = retrieve_claim_evidence(
            item["claim"],
            vectorstore,
            reranker
        )

        corrective_evidence[number] = evidence

        print(
            f"Claim {number}: "
            f"{len(evidence)} passages retrieved"
        )

    return corrective_evidence


corrective_evidence = run_selective_retrieval(
    filtered_claims,
    vectorstore,
    reranker
)

print("\nSTEP 7G COMPLETE")


Claims requiring corrective retrieval: 1
Claim 4: 4 passages retrieved

STEP 7G COMPLETE


In [57]:

# ============================================================
# STEP 7G TEST: UNSUPPORTED CLAIM SCENARIO
# ============================================================

test_claims = [
    {
        "claim_number": 999,
        "claim": (
            "The eDySec framework was developed "
            "specifically for detecting malware "
            "in Android mobile applications."
        ),
        "status": "NEEDS_EVIDENCE",
        "supported_evidence_ids": [],
        "contradicted_evidence_ids": [],
        "unknown_evidence_ids": [],
        "insufficient_evidence_ids": []
    }
]

test_corrective_evidence = run_selective_retrieval(
    test_claims,
    vectorstore,
    reranker
)

print("\nTEST RETRIEVAL RESULTS")

for number, evidence in test_corrective_evidence.items():
    print(f"\nTest Claim {number}")
    print("Retrieved passages:", len(evidence))

    for i, (doc, score) in enumerate(evidence, 1):
        print(
            f"Passage {i} | "
            f"Page: {doc.metadata.get('page_label', 'Unknown')} | "
            f"Relevance score: {float(score):.4f}"
        )


Claims requiring corrective retrieval: 1
Claim 999: 4 passages retrieved

TEST RETRIEVAL RESULTS

Test Claim 999
Retrieved passages: 4
Passage 1 | Page: 2 | Relevance score: 5.0980
Passage 2 | Page: 2 | Relevance score: 4.8551
Passage 3 | Page: 1 | Relevance score: 4.6874
Passage 4 | Page: 15 | Relevance score: 3.7070


In [58]:

# ============================================================
# STEP 7H FIX: DEFINE SINGLE CITATION VALIDATOR
# ============================================================

def validate_single_citation(claim, document, llm):

    prompt = f"""
You are a strict scientific evidence validator.

CLAIM:
{claim}

EVIDENCE:
{document.page_content}

Does this single evidence passage directly support
ALL factual parts of the claim?

RULES:
1. Use ONLY the evidence provided.
2. Do not use external knowledge.
3. SUPPORTED: The passage directly supports the entire claim.
4. CONTRADICTED: The passage explicitly contradicts the claim.
5. INSUFFICIENT: The passage does not establish the full claim.
6. Missing evidence does not mean contradiction.
7. Return exactly ONE word:
SUPPORTED
CONTRADICTED
INSUFFICIENT

Decision:
"""

    raw_response = llm.invoke(prompt).strip()
    decision = raw_response.upper()

    allowed = {
        "SUPPORTED",
        "CONTRADICTED",
        "INSUFFICIENT"
    }

    if decision not in allowed:
        decision = "UNKNOWN"

    return {
        "decision": decision,
        "raw_response": raw_response
    }


print("Single citation validator defined successfully!")


Single citation validator defined successfully!


In [59]:

# ============================================================
# STEP 7H: CORRECTIVE EVIDENCE VALIDATION
# ============================================================

def validate_corrective_evidence(
    filtered_claims,
    corrective_evidence,
    llm
):
    results = {}

    for item in filtered_claims:

        if item["status"] != "NEEDS_EVIDENCE":
            continue

        claim_number = item["claim_number"]
        claim = item["claim"]

        evidence = corrective_evidence.get(
            claim_number, []
        )

        checks = []

        for passage_number, (doc, score) in enumerate(
            evidence, 1
        ):

            validation = validate_single_citation(
                claim,
                doc,
                llm
            )

            checks.append({
                "passage_number": passage_number,
                "page": doc.metadata.get(
                    "page_label", "Unknown"
                ),
                "decision": validation["decision"],
                "score": float(score),
                "raw_response": validation["raw_response"]
            })

        results[claim_number] = checks

        print(f"\nClaim {claim_number}")

        if not checks:
            print("No corrective evidence available.")

        for check in checks:
            print(
                f"  Passage {check['passage_number']} "
                f"(Page {check['page']}): "
                f"{check['decision']}"
            )

    return results


# Original claims: no extra LLM calls if all supported.
corrective_results = validate_corrective_evidence(
    filtered_claims,
    corrective_evidence,
    llm
)

# Negative test claim: validate all 4 retrieved passages.
test_corrective_results = validate_corrective_evidence(
    test_claims,
    test_corrective_evidence,
    llm
)

print("\nSTEP 7H COMPLETE")



Claim 4
  Passage 1 (Page 10): UNKNOWN
  Passage 2 (Page 9): UNKNOWN
  Passage 3 (Page 10): UNKNOWN
  Passage 4 (Page 7): UNKNOWN

Claim 999
  Passage 1 (Page 2): UNKNOWN
  Passage 2 (Page 2): UNKNOWN
  Passage 3 (Page 1): UNKNOWN
  Passage 4 (Page 15): UNKNOWN

STEP 7H COMPLETE


In [60]:

# ============================================================
# STEP 7I: FINAL CLAIM DECISIONS
# ============================================================

def build_final_claim_results(
    filtered_claims,
    corrective_results
):
    final_results = []

    for item in filtered_claims:

        number = item["claim_number"]

        extra_checks = corrective_results.get(
            number, []
        )

        extra_supported = [
            c for c in extra_checks
            if c["decision"] == "SUPPORTED"
        ]

        extra_contradicted = [
            c for c in extra_checks
            if c["decision"] == "CONTRADICTED"
        ]

        initial_supported = item.get(
            "supported_evidence_ids", []
        )

        initial_contradicted = item.get(
            "contradicted_evidence_ids", []
        )

        if initial_contradicted or extra_contradicted:
            final_status = "CONFLICT_REVIEW"

        elif initial_supported or extra_supported:
            final_status = "PROVISIONALLY_SUPPORTED"

        else:
            final_status = "UNVERIFIED"

        final_results.append({
            "claim_number": number,
            "claim": item["claim"],
            "status": final_status,
            "initial_supported_evidence_ids":
                initial_supported,
            "corrective_supported_passages": [
                {
                    "passage_number": c["passage_number"],
                    "page": c["page"]
                }
                for c in extra_supported
            ],
            "corrective_checks": extra_checks
        })

    return final_results


# Original research question.
final_claim_results = build_final_claim_results(
    filtered_claims,
    corrective_results
)

# Negative test case.
test_final_claim_results = build_final_claim_results(
    test_claims,
    test_corrective_results
)

print("=" * 60)
print("ORIGINAL QUESTION: FINAL CLAIM RESULTS")
print("=" * 60)

for item in final_claim_results:
    print(
        f"Claim {item['claim_number']}: "
        f"{item['status']}"
    )

print("\n" + "=" * 60)
print("NEGATIVE TEST: FINAL CLAIM RESULTS")
print("=" * 60)

for item in test_final_claim_results:
    print(
        f"Claim {item['claim_number']}: "
        f"{item['status']}"
    )

print("\nSTEP 7I COMPLETE")


ORIGINAL QUESTION: FINAL CLAIM RESULTS
Claim 1: PROVISIONALLY_SUPPORTED
Claim 2: PROVISIONALLY_SUPPORTED
Claim 3: PROVISIONALLY_SUPPORTED
Claim 4: UNVERIFIED

NEGATIVE TEST: FINAL CLAIM RESULTS
Claim 999: UNVERIFIED

STEP 7I COMPLETE


In [61]:

# ============================================================
# STEP 7J: ANSWER DECISION AND ABSTENTION
# ============================================================

def decide_answer_action(final_claim_results):

    if not final_claim_results:
        return "ABSTAIN"

    statuses = [
        item["status"]
        for item in final_claim_results
    ]

    if "CONFLICT_REVIEW" in statuses:
        return "REVIEW_CONFLICT"

    supported_count = statuses.count(
        "PROVISIONALLY_SUPPORTED"
    )

    if supported_count == len(statuses):
        return "ACCEPT_FOR_REVIEW"

    if supported_count > 0:
        return "REVISE"

    return "ABSTAIN"


original_action = decide_answer_action(
    final_claim_results
)

test_action = decide_answer_action(
    test_final_claim_results
)

print("=" * 60)
print("STEP 7J: ANSWER DECISIONS")
print("=" * 60)

print("Original question:", original_action)
print("Negative test:", test_action)

if original_action == "ACCEPT_FOR_REVIEW":
    print(
        "\nOriginal answer: All extracted claims "
        "have provisional supporting evidence."
    )

if test_action == "ABSTAIN":
    print(
        "\nNegative test: Insufficient evidence. "
        "System correctly routes to abstention."
    )

elif test_action == "REVIEW_CONFLICT":
    print(
        "\nNegative test: Conflicting evidence "
        "requires further review."
    )

elif test_action == "ACCEPT_FOR_REVIEW":
    print(
        "\nWARNING: Negative test was provisionally "
        "accepted. Inspect supporting passages."
    )

print("\nSTEP 7J COMPLETE")


STEP 7J: ANSWER DECISIONS
Original question: REVISE
Negative test: ABSTAIN

Negative test: Insufficient evidence. System correctly routes to abstention.

STEP 7J COMPLETE


In [62]:

# ============================================================
# STEP 7K: SAVE VERIFICATION RESULTS
# ============================================================

import json
import os
from datetime import datetime, timezone

output_dir = "/content/rag_verification_results"
os.makedirs(output_dir, exist_ok=True)


def make_json_safe(value):
    if isinstance(value, dict):
        return {
            str(k): make_json_safe(v)
            for k, v in value.items()
        }

    if isinstance(value, (list, tuple)):
        return [make_json_safe(v) for v in value]

    if isinstance(value, (str, int, float, bool)) or value is None:
        return value

    return str(value)


report = {
    "timestamp_utc": datetime.now(
        timezone.utc
    ).isoformat(),

    "question": question,

    "generated_answer": answer_result["answer"],

    "original": {
        "action": original_action,
        "claim_results": final_claim_results
    },

    "negative_test": {
        "claim": test_claims[0]["claim"],
        "action": test_action,
        "claim_results": test_final_claim_results
    },

    "verification_note": (
        "Evidence decisions are produced by an LLM "
        "and remain provisional until independently evaluated."
    )
}

output_path = os.path.join(
    output_dir,
    "step7_verification_report.json"
)

with open(output_path, "w", encoding="utf-8") as f:
    json.dump(
        make_json_safe(report),
        f,
        indent=2,
        ensure_ascii=False
    )

print("Results saved successfully:")
print(output_path)


Results saved successfully:
/content/rag_verification_results/step7_verification_report.json


In [63]:

# ============================================================
# STEP 7L: DECISION ROUTING UNIT TESTS
# ============================================================

test_cases = [
    (
        "All supported",
        ["PROVISIONALLY_SUPPORTED",
         "PROVISIONALLY_SUPPORTED"],
        "ACCEPT_FOR_REVIEW"
    ),
    (
        "Partially supported",
        ["PROVISIONALLY_SUPPORTED", "UNVERIFIED"],
        "REVISE"
    ),
    (
        "All unsupported",
        ["UNVERIFIED", "UNVERIFIED"],
        "ABSTAIN"
    ),
    (
        "Conflicting evidence",
        ["PROVISIONALLY_SUPPORTED", "CONFLICT_REVIEW"],
        "REVIEW_CONFLICT"
    )
]

print("=" * 60)
print("STEP 7L: UNIT TEST RESULTS")
print("=" * 60)

for name, statuses, expected in test_cases:

    sample_results = [
        {"status": status}
        for status in statuses
    ]

    actual = decide_answer_action(sample_results)

    passed = actual == expected

    print(
        f"{'PASS' if passed else 'FAIL'} | "
        f"{name} | Actual: {actual}"
    )

    assert passed, f"Test failed: {name}"

print("\nAll decision routing unit tests passed.")


STEP 7L: UNIT TEST RESULTS
PASS | All supported | Actual: ACCEPT_FOR_REVIEW
PASS | Partially supported | Actual: REVISE
PASS | All unsupported | Actual: ABSTAIN
PASS | Conflicting evidence | Actual: REVIEW_CONFLICT

All decision routing unit tests passed.


In [64]:

# BACKUP STEP 7 RESULTS TO GOOGLE DRIVE

from google.colab import drive
import os
import shutil

drive.mount("/content/drive")

backup_dir = "/content/drive/MyDrive/Evidence_Verified_RAG/results"
os.makedirs(backup_dir, exist_ok=True)

source = "/content/rag_verification_results/step7_verification_report.json"
destination = os.path.join(
    backup_dir,
    "step7_verification_report.json"
)

shutil.copy2(source, destination)

print("Backup completed:", destination)


Mounted at /content/drive
Backup completed: /content/drive/MyDrive/Evidence_Verified_RAG/results/step7_verification_report.json


# Step 8: Configure corrective retrrival

In [65]:

# ============================================================
# STEP 8A: CORRECTIVE RETRIEVAL CONFIGURATION
# ============================================================

MAX_RETRIES = 2
RETRIEVAL_K = 8
TOP_N = 4

print("=" * 60)
print("STEP 8A: CONFIGURATION")
print("=" * 60)

print("Maximum retrieval attempts:", MAX_RETRIES)
print("Candidate documents per query:", RETRIEVAL_K)
print("Top passages for verification:", TOP_N)

print("\nSTEP 8A COMPLETE")


STEP 8A: CONFIGURATION
Maximum retrieval attempts: 2
Candidate documents per query: 8
Top passages for verification: 4

STEP 8A COMPLETE


In [66]:

# ============================================================
# STEP 8B: GENERATE CORRECTIVE SEARCH QUERIES
# ============================================================

def generate_corrective_queries(claim, attempt):

    if attempt == 1:
        return [
            claim,
            f"Research evidence supporting: {claim}"
        ]

    return [
        claim,
        f"Evidence, methodology and findings related to: {claim}",
        f"Does the research paper establish the following claim: {claim}"
    ]


print("=" * 60)
print("STEP 8B: QUERY GENERATION TEST")
print("=" * 60)

sample_claim = test_claims[0]["claim"]

for attempt in range(1, MAX_RETRIES + 1):
    print(f"\nAttempt {attempt}")

    queries = generate_corrective_queries(
        sample_claim,
        attempt
    )

    for query in queries:
        print("-", query)

print("\nSTEP 8B COMPLETE")


STEP 8B: QUERY GENERATION TEST

Attempt 1
- The eDySec framework was developed specifically for detecting malware in Android mobile applications.
- Research evidence supporting: The eDySec framework was developed specifically for detecting malware in Android mobile applications.

Attempt 2
- The eDySec framework was developed specifically for detecting malware in Android mobile applications.
- Evidence, methodology and findings related to: The eDySec framework was developed specifically for detecting malware in Android mobile applications.
- Does the research paper establish the following claim: The eDySec framework was developed specifically for detecting malware in Android mobile applications.

STEP 8B COMPLETE


In [67]:

# ============================================================
# STEP 8C: RETRIEVE AND RERANK CORRECTIVE EVIDENCE
# ============================================================

def retrieve_corrective_passages(
    claim,
    attempt,
    vectorstore,
    reranker,
    k=8,
    top_n=4
):

    queries = generate_corrective_queries(
        claim,
        attempt
    )

    all_docs = []

    for query in queries:

        docs = vectorstore.similarity_search(
            query,
            k=k
        )

        all_docs.extend(docs)

    # Remove duplicate passages.
    unique_docs = []
    seen = set()

    for doc in all_docs:

        key = (
            doc.metadata.get("source"),
            doc.metadata.get("page"),
            doc.page_content
        )

        if key not in seen:
            seen.add(key)
            unique_docs.append(doc)

    if not unique_docs:
        return []

    # Reuse the existing cross-encoder.
    ranked_docs = rerank_documents(
        claim,
        unique_docs,
        reranker,
        top_n=top_n
    )

    return ranked_docs


print("STEP 8C: Retrieval function defined successfully.")


STEP 8C: Retrieval function defined successfully.


In [68]:

# ============================================================
# STEP 8D: EVIDENCE VERIFICATION WITH RETRIES
# ============================================================

def corrective_verify_claim(
    claim,
    vectorstore,
    reranker,
    llm,
    max_retries=2
):

    history = []
    seen_passages = set()

    supported = []
    contradicted = []

    for attempt in range(1, max_retries + 1):

        print(f"\nRetrieval attempt {attempt}/{max_retries}")

        passages = retrieve_corrective_passages(
            claim,
            attempt,
            vectorstore,
            reranker,
            k=RETRIEVAL_K,
            top_n=TOP_N
        )

        new_passages = []

        for doc, score in passages:

            key = (
                doc.metadata.get("source"),
                doc.metadata.get("page"),
                doc.page_content
            )

            if key not in seen_passages:
                seen_passages.add(key)
                new_passages.append((doc, score))

        print("New passages:", len(new_passages))

        for doc, score in new_passages:

            validation = validate_single_citation(
                claim,
                doc,
                llm
            )

            result = {
                "attempt": attempt,
                "page": doc.metadata.get(
                    "page_label", "Unknown"
                ),
                "decision": validation["decision"],
                "score": float(score),
                "raw_response": validation["raw_response"]
            }

            history.append(result)

            if result["decision"] == "SUPPORTED":
                supported.append(result)

            elif result["decision"] == "CONTRADICTED":
                contradicted.append(result)

            print(
                "Page:", result["page"],
                "| Decision:", result["decision"]
            )

        # Stop only if supporting evidence was found
        # and no contradiction was identified.
        if supported and not contradicted:
            print("Supporting evidence found.")
            break

    if contradicted:
        status = "CONFLICT_REVIEW"

    elif supported:
        status = "PROVISIONALLY_SUPPORTED"

    else:
        status = "UNVERIFIED"

    return {
        "claim": claim,
        "status": status,
        "attempts_used": attempt,
        "supported_passages": supported,
        "contradicted_passages": contradicted,
        "verification_history": history
    }


print("STEP 8D: Corrective verification function ready.")


STEP 8D: Corrective verification function ready.


In [69]:

# ============================================================
# STEP 8E: NEGATIVE CLAIM TEST
# ============================================================

print("=" * 60)
print("STEP 8E: NEGATIVE CLAIM TEST")
print("=" * 60)

negative_claim = test_claims[0]["claim"]

negative_result = corrective_verify_claim(
    negative_claim,
    vectorstore,
    reranker,
    llm,
    max_retries=MAX_RETRIES
)

print("\n" + "=" * 60)
print("FINAL NEGATIVE TEST RESULT")
print("=" * 60)

print("Claim:", negative_result["claim"])
print("Status:", negative_result["status"])
print("Attempts used:", negative_result["attempts_used"])

print(
    "Supporting passages:",
    len(negative_result["supported_passages"])
)

print(
    "Contradicted passages:",
    len(negative_result["contradicted_passages"])
)

print("\nSTEP 8E COMPLETE")


STEP 8E: NEGATIVE CLAIM TEST

Retrieval attempt 1/2
New passages: 4
Page: 2 | Decision: UNKNOWN
Page: 2 | Decision: UNKNOWN
Page: 1 | Decision: UNKNOWN
Page: 15 | Decision: UNKNOWN

Retrieval attempt 2/2
New passages: 0

FINAL NEGATIVE TEST RESULT
Claim: The eDySec framework was developed specifically for detecting malware in Android mobile applications.
Status: UNVERIFIED
Attempts used: 2
Supporting passages: 0
Contradicted passages: 0

STEP 8E COMPLETE


In [70]:

# ============================================================
# STEP 8F: SELECTIVE CORRECTIVE VERIFICATION
# ============================================================

def run_improved_corrective_verification(
    filtered_claims,
    vectorstore,
    reranker,
    llm
):

    results = {}

    targets = [
        item for item in filtered_claims
        if item["status"] == "NEEDS_EVIDENCE"
    ]

    print("Claims requiring corrective verification:", len(targets))

    for item in targets:

        number = item["claim_number"]

        results[number] = corrective_verify_claim(
            item["claim"],
            vectorstore,
            reranker,
            llm,
            max_retries=MAX_RETRIES
        )

    return results


improved_corrective_results = (
    run_improved_corrective_verification(
        filtered_claims,
        vectorstore,
        reranker,
        llm
    )
)

print("\nSTEP 8F COMPLETE")


Claims requiring corrective verification: 1

Retrieval attempt 1/2
New passages: 4
Page: 10 | Decision: UNKNOWN
Page: 9 | Decision: UNKNOWN
Page: 10 | Decision: UNKNOWN
Page: 9 | Decision: UNKNOWN

Retrieval attempt 2/2
New passages: 1
Page: 7 | Decision: UNKNOWN

STEP 8F COMPLETE


In [71]:

# ============================================================
# STEP 8G: DEBUG UNKNOWN VERIFICATION DECISIONS
# ============================================================

print("=" * 60)
print("STEP 8G: RAW VERIFICATION RESPONSES")
print("=" * 60)

for i, result in enumerate(
    negative_result["verification_history"], 1
):
    print(f"\nPassage {i}")
    print("Page:", result["page"])
    print("Parsed decision:", result["decision"])
    print("Raw LLM response:")
    print(repr(result["raw_response"]))
    print("-" * 60)

print("\nSTEP 8G COMPLETE")


STEP 8G: RAW VERIFICATION RESPONSES

Passage 1
Page: 2
Parsed decision: UNKNOWN
Raw LLM response:
'INSUFFICIENT\n\nThe passage mentions that eDySec is an "efficient, stable, and explainable DL-based extended Dynamic Security analysis framework for detecting malicious packages", which supports the claim that eDySec is a framework for detecting malware. However, it does not directly state that eDySec was "developed specifically for detecting malware in Android mobile applications". The passage only mentions "malicious packages" in general, without specifying the platform (Android) or the type of malware (mobile applications).'
------------------------------------------------------------

Passage 2
Page: 2
Parsed decision: UNKNOWN
Raw LLM response:
'INSUFFICIENT\n\nThe passage does not explicitly state that the eDySec framework was developed specifically for detecting malware in Android mobile applications. It mentions that the framework is used for detecting "malicious packages", but it 

In [72]:

# ============================================================
# STEP 8H: ROBUST VERIFICATION DECISION PARSER
# ============================================================

import re

def parse_verification_decision(raw_response):

    text = raw_response.strip().upper()

    allowed = {
        "SUPPORTED",
        "CONTRADICTED",
        "INSUFFICIENT"
    }

    # Case 1: Exact one-word response.
    if text in allowed:
        return text

    # Case 2: Explicit labelled decision.
    matches = re.findall(
        r"(?m)^\s*(?:FINAL_DECISION|DECISION|LABEL)"
        r"\s*:\s*(SUPPORTED|CONTRADICTED|INSUFFICIENT)\s*$",
        text
    )

    if len(set(matches)) == 1 and matches:
        return matches[0]

    # Case 3: A single standalone label on the first line.
    first_line = text.splitlines()[0].strip() if text else ""

    if first_line in allowed:
        return first_line

    # Ambiguous responses must not be guessed.
    return "UNKNOWN"


# Test the parser.
test_responses = [
    "SUPPORTED",
    "Decision: INSUFFICIENT",
    "FINAL_DECISION: CONTRADICTED",
    "INSUFFICIENT\nThe evidence does not establish the claim.",
    "The evidence may support some aspects of the claim."
]

print("=" * 60)
print("STEP 8H: PARSER TEST")
print("=" * 60)

for response in test_responses:
    print(
        repr(response),
        "=>",
        parse_verification_decision(response)
    )

print("\nSTEP 8H COMPLETE")


STEP 8H: PARSER TEST
'SUPPORTED' => SUPPORTED
'Decision: INSUFFICIENT' => INSUFFICIENT
'FINAL_DECISION: CONTRADICTED' => CONTRADICTED
'INSUFFICIENT\nThe evidence does not establish the claim.' => INSUFFICIENT
'The evidence may support some aspects of the claim.' => UNKNOWN

STEP 8H COMPLETE


In [73]:

# ============================================================
# STEP 8I: RECHECK EXISTING NEGATIVE TEST RESULTS
# ============================================================

rechecked_history = []

for item in negative_result["verification_history"]:

    updated = item.copy()

    updated["decision"] = parse_verification_decision(
        item["raw_response"]
    )

    rechecked_history.append(updated)

    print(
        "Page:", updated["page"],
        "| Previous:", item["decision"],
        "| Rechecked:", updated["decision"]
    )

supported = [
    item for item in rechecked_history
    if item["decision"] == "SUPPORTED"
]

contradicted = [
    item for item in rechecked_history
    if item["decision"] == "CONTRADICTED"
]

if contradicted:
    rechecked_status = "CONFLICT_REVIEW"
elif supported:
    rechecked_status = "PROVISIONALLY_SUPPORTED"
else:
    rechecked_status = "UNVERIFIED"

print("\n" + "=" * 60)
print("RECHECKED FINAL STATUS:", rechecked_status)
print("Supported passages:", len(supported))
print("Contradicted passages:", len(contradicted))
print("UNKNOWN passages:", sum(
    item["decision"] == "UNKNOWN"
    for item in rechecked_history
))
print("=" * 60)


Page: 2 | Previous: UNKNOWN | Rechecked: INSUFFICIENT
Page: 2 | Previous: UNKNOWN | Rechecked: INSUFFICIENT
Page: 1 | Previous: UNKNOWN | Rechecked: INSUFFICIENT
Page: 15 | Previous: UNKNOWN | Rechecked: INSUFFICIENT

RECHECKED FINAL STATUS: UNVERIFIED
Supported passages: 0
Contradicted passages: 0
UNKNOWN passages: 0


# Step 9: Evidence-Based Answer Revision and Abstention

In [74]:

# ============================================================
# STEP 9A: PREPARE FINAL CLAIM DECISIONS
# ============================================================

def prepare_final_claim_decisions(
    final_claim_results,
    improved_corrective_results
):
    prepared = []

    for item in final_claim_results:

        result = item.copy()
        number = result["claim_number"]

        # Apply updated corrective verification if available.
        if number in improved_corrective_results:

            corrective = improved_corrective_results[number]

            result["status"] = corrective["status"]

            result["corrective_checks"] = (
                corrective["verification_history"]
            )

            result["corrective_supported_passages"] = [
                {
                    "page": p["page"],
                    "attempt": p["attempt"]
                }
                for p in corrective["supported_passages"]
            ]

        prepared.append(result)

    return prepared


step9_claim_results = prepare_final_claim_decisions(
    final_claim_results,
    improved_corrective_results
)

print("=" * 60)
print("STEP 9A: FINAL CLAIM DECISIONS")
print("=" * 60)

for item in step9_claim_results:
    print(
        f"Claim {item['claim_number']}: "
        f"{item['status']}"
    )

print("\nSTEP 9A COMPLETE")


STEP 9A: FINAL CLAIM DECISIONS
Claim 1: PROVISIONALLY_SUPPORTED
Claim 2: PROVISIONALLY_SUPPORTED
Claim 3: PROVISIONALLY_SUPPORTED
Claim 4: UNVERIFIED

STEP 9A COMPLETE


In [75]:

# ============================================================
# STEP 9B: PREPARE VERIFIED CLAIMS WITH CITATIONS
# ============================================================

def prepare_verified_claims(claim_results, evidence_docs):

    verified_claims = []

    for item in claim_results:

        if item["status"] != "PROVISIONALLY_SUPPORTED":
            continue

        evidence = []

        # Original validated supporting citations.
        for evidence_id in item.get(
            "initial_supported_evidence_ids", []
        ):

            if not isinstance(evidence_id, int):
                continue

            if not 1 <= evidence_id <= len(evidence_docs):
                continue

            doc, score = evidence_docs[evidence_id - 1]

            evidence.append({
                "evidence_id": evidence_id,
                "page": doc.metadata.get(
                    "page_label", "Unknown"
                ),
                "text": doc.page_content,
                "source": "initial"
            })

        # Corrective retrieval citations.
        # The earlier Step 8 results contain page metadata,
        # but not the actual passage text. Do not treat a page
        # number alone as verified supporting evidence.

        if not evidence:
            continue

        verified_claims.append({
            "claim_number": item["claim_number"],
            "claim": item["claim"],
            "evidence": evidence
        })

    return verified_claims


step9_verified_claims = prepare_verified_claims(
    step9_claim_results,
    fixed_docs
)

print("=" * 60)
print("STEP 9B: VERIFIED CLAIMS")
print("=" * 60)

for item in step9_verified_claims:

    print(f"\nClaim {item['claim_number']}")
    print("Text:", item["claim"])

    pages = sorted(set(
        str(e["page"]) for e in item["evidence"]
    ))

    print("Supporting pages:", pages)

print("\nClaims with usable supporting evidence:",
      len(step9_verified_claims))

print("\nSTEP 9B COMPLETE")


STEP 9B: VERIFIED CLAIMS

Claim 1
Text: This study presents eDySec, an efficient, stable, and explainable DL-based extended Dynamic Security analysis framework for detecting malicious packages.
Supporting pages: ['1', '10', '12', '15', '2']

Claim 2
Text: malicious package detection. Additionally, model stability analysis and explainable AI techniques are incorporated into the detection pipeline...
Supporting pages: ['1', '2']

Claim 3
Text: This study presents eDySec, an efficient, stable, and explainable DL-based framework for malicious package detection using the dynamic QUT-DV25 dataset.
Supporting pages: ['1', '2']

Claims with usable supporting evidence: 3

STEP 9B COMPLETE


In [76]:

# ============================================================
# STEP 9C: GENERATE EVIDENCE-GROUNDED FINAL ANSWER
# ============================================================

def generate_verified_answer(question, verified_claims, llm):

    if not verified_claims:
        return {
            "status": "ABSTAIN",
            "answer": (
                "Insufficient verified evidence is available "
                "to answer this question reliably."
            )
        }

    evidence_blocks = []

    for item in verified_claims:

        claim_number = item["claim_number"]

        for ev in item["evidence"]:

            evidence_blocks.append(
                f"Claim {claim_number}\n"
                f"Verified claim: {item['claim']}\n"
                f"Citation: [E{ev['evidence_id']}, "
                f"Page {ev['page']}]\n"
                f"Evidence passage:\n{ev['text']}"
            )

    evidence_context = "\n\n".join(evidence_blocks)

    prompt = f"""
You are a careful scientific research assistant.

QUESTION:
{question}

PROVISIONALLY VERIFIED CLAIMS AND EVIDENCE:
{evidence_context}

TASK:
Write a clear, concise answer to the question.

STRICT RULES:
1. Use ONLY the supplied verified claims and evidence.
2. Do not introduce additional factual claims.
3. Do not invent citations or page numbers.
4. Cite factual statements using the provided
   [E<number>, Page <number>] citation format.
5. Do not include unsupported information.
6. If the evidence cannot answer the question,
   respond with:
   INSUFFICIENT_EVIDENCE
7. Write in natural academic English.

FINAL ANSWER:
"""

    answer = llm.invoke(prompt).strip()

    if answer.upper().startswith("INSUFFICIENT_EVIDENCE"):
        return {
            "status": "ABSTAIN",
            "answer": (
                "Insufficient verified evidence is available "
                "to answer this question reliably."
            )
        }

    return {
        "status": "DRAFT_FOR_REVIEW",
        "answer": answer
    }


print("STEP 9C: Answer generation function defined.")


STEP 9C: Answer generation function defined.


In [77]:

# ============================================================
# STEP 9D: ORIGINAL QUESTION TEST
# ============================================================

print("=" * 60)
print("STEP 9D: VERIFIED ANSWER GENERATION")
print("=" * 60)

step9_answer_result = generate_verified_answer(
    question,
    step9_verified_claims,
    llm
)

print("\nQUESTION:")
print(question)

print("\nSTATUS:")
print(step9_answer_result["status"])

print("\nGENERATED ANSWER:")
print(step9_answer_result["answer"])

print("\nSTEP 9D COMPLETE")


STEP 9D: VERIFIED ANSWER GENERATION

QUESTION:
What problem does eDySec aim to solve?

STATUS:
DRAFT_FOR_REVIEW

GENERATED ANSWER:
The problem that eDySec aims to solve is malicious package detection in software supply chain security. 

Verified claim: This study presents eDySec, an efficient, stable, and explainable DL-based extended Dynamic Security analysis framework for detecting malicious packages.
Citation: [E1, Page 2]

Evidence passage:
techniques into the detection pipeline, enabling near-perfect
stable and transparent interpretation of model decisions by
identifying the features that most influence each prediction.
• Experimental results demonstrate that eDySec significantly
outperforms the state-of-the-art framework. Specifically, it
reduces feature dimensionality by 52.78% while lowering
false positives by 82% and false negatives by 79%. It also
improves accuracy by 3% and maintains a low inference
latency of 170ms per package.

Verified claim: malicious package detection. 

In [78]:

# ============================================================
# STEP 9E: ABSTENTION TEST
# ============================================================

unsupported_claims = []

step9_abstention_result = generate_verified_answer(
    "Was eDySec developed for Android malware detection?",
    unsupported_claims,
    llm
)

print("=" * 60)
print("STEP 9E: ABSTENTION TEST")
print("=" * 60)

print("Status:", step9_abstention_result["status"])
print("Answer:", step9_abstention_result["answer"])

assert step9_abstention_result["status"] == "ABSTAIN"

print("\nPASS: System abstains when no verified evidence exists.")
print("STEP 9E COMPLETE")


STEP 9E: ABSTENTION TEST
Status: ABSTAIN
Answer: Insufficient verified evidence is available to answer this question reliably.

PASS: System abstains when no verified evidence exists.
STEP 9E COMPLETE


In [79]:

# ============================================================
# STEP 9F: SAVE ANSWER REVISION RESULTS
# ============================================================

import os
import json

output_dir = "/content/rag_verification_results"
os.makedirs(output_dir, exist_ok=True)

step9_report = {
    "question": question,
    "verified_claim_count": len(step9_verified_claims),
    "claim_results": step9_claim_results,
    "generated_answer": step9_answer_result,
    "abstention_test": step9_abstention_result
}

# Convert any non-JSON-native values safely.
output_path = os.path.join(
    output_dir,
    "step9_answer_revision_report.json"
)

with open(output_path, "w", encoding="utf-8") as f:
    json.dump(
        step9_report,
        f,
        indent=2,
        ensure_ascii=False,
        default=str
    )

print("=" * 60)
print("STEP 9F: RESULTS SAVED")
print("=" * 60)
print("File:", output_path)
print("STEP 9F COMPLETE")


STEP 9F: RESULTS SAVED
File: /content/rag_verification_results/step9_answer_revision_report.json
STEP 9F COMPLETE


In [80]:

# ============================================================
# STEP 9G: CONSTRUCT CITATION-GROUNDED ANSWER
# ============================================================

def construct_cited_answer(verified_claims):

    if not verified_claims:
        return {
            "status": "ABSTAIN",
            "answer": (
                "Insufficient verified evidence is available "
                "to answer this question reliably."
            ),
            "citations": []
        }

    sentences = []
    citation_records = []

    for item in verified_claims:

        claim = item["claim"].strip().rstrip(".")

        references = []
        seen = set()

        for ev in item["evidence"]:

            citation = (
                f"[E{ev['evidence_id']}, "
                f"Page {ev['page']}]"
            )

            if citation not in seen:
                seen.add(citation)
                references.append(citation)

                citation_records.append({
                    "claim_number": item["claim_number"],
                    "evidence_id": ev["evidence_id"],
                    "page": ev["page"]
                })

        if references:
            sentences.append(
                claim + ". " + " ".join(references)
            )

    if not sentences:
        return {
            "status": "ABSTAIN",
            "answer": (
                "Insufficient verified evidence is available "
                "to answer this question reliably."
            ),
            "citations": []
        }

    return {
        "status": "CITED_CLAIMS_FOR_REVIEW",
        "answer": "\n\n".join(sentences),
        "citations": citation_records
    }


step9_cited_result = construct_cited_answer(
    step9_verified_claims
)

print("=" * 60)
print("STEP 9G: CITATION-GROUNDED ANSWER")
print("=" * 60)

print("Status:", step9_cited_result["status"])
print("\nAnswer:\n")
print(step9_cited_result["answer"])

print("\nSTEP 9G COMPLETE")


STEP 9G: CITATION-GROUNDED ANSWER
Status: CITED_CLAIMS_FOR_REVIEW

Answer:

This study presents eDySec, an efficient, stable, and explainable DL-based extended Dynamic Security analysis framework for detecting malicious packages. [E1, Page 2] [E4, Page 1] [E5, Page 12] [E6, Page 10] [E7, Page 15]

malicious package detection. Additionally, model stability analysis and explainable AI techniques are incorporated into the detection pipeline. [E1, Page 2] [E4, Page 1]

This study presents eDySec, an efficient, stable, and explainable DL-based framework for malicious package detection using the dynamic QUT-DV25 dataset. [E1, Page 2] [E4, Page 1]

STEP 9G COMPLETE


In [81]:

# ============================================================
# STEP 9H: CITATION INTEGRITY TEST
# ============================================================

import re

def check_citation_integrity(answer_result, evidence_docs):

    answer = answer_result["answer"]

    found_citations = re.findall(
        r"\[E(\d+),\s*Page\s+([^\]]+)\]",
        answer
    )

    checks = []

    for evidence_id_text, page in found_citations:

        evidence_id = int(evidence_id_text)

        valid_id = (
            1 <= evidence_id <= len(evidence_docs)
        )

        valid_page = False

        if valid_id:
            doc, score = evidence_docs[evidence_id - 1]

            actual_page = str(
                doc.metadata.get("page_label", "Unknown")
            )

            valid_page = (
                actual_page == page.strip()
            )

        checks.append({
            "evidence_id": evidence_id,
            "page": page,
            "valid_id": valid_id,
            "valid_page": valid_page
        })

    all_valid = (
        len(found_citations) > 0
        and all(
            item["valid_id"] and item["valid_page"]
            for item in checks
        )
    )

    return {
        "citations_found": len(found_citations),
        "checks": checks,
        "all_valid": all_valid
    }


step9_citation_check = check_citation_integrity(
    step9_cited_result,
    fixed_docs
)

print("=" * 60)
print("STEP 9H: CITATION INTEGRITY")
print("=" * 60)

print(
    "Citations found:",
    step9_citation_check["citations_found"]
)

print(
    "All citation references valid:",
    step9_citation_check["all_valid"]
)

for item in step9_citation_check["checks"]:
    print(
        f"E{item['evidence_id']} | "
        f"Page {item['page']} | "
        f"Valid ID: {item['valid_id']} | "
        f"Valid page: {item['valid_page']}"
    )

assert step9_citation_check["all_valid"], (
    "Citation integrity check failed."
)

print("\nSTEP 9H COMPLETE")


STEP 9H: CITATION INTEGRITY
Citations found: 9
All citation references valid: True
E1 | Page 2 | Valid ID: True | Valid page: True
E4 | Page 1 | Valid ID: True | Valid page: True
E5 | Page 12 | Valid ID: True | Valid page: True
E6 | Page 10 | Valid ID: True | Valid page: True
E7 | Page 15 | Valid ID: True | Valid page: True
E1 | Page 2 | Valid ID: True | Valid page: True
E4 | Page 1 | Valid ID: True | Valid page: True
E1 | Page 2 | Valid ID: True | Valid page: True
E4 | Page 1 | Valid ID: True | Valid page: True

STEP 9H COMPLETE


In [82]:

# ============================================================
# STEP 9I: SAVE CITATION-GROUNDED RESULTS
# ============================================================

import json
import os

output_dir = "/content/rag_verification_results"
os.makedirs(output_dir, exist_ok=True)

step9_final_report = {
    "question": question,
    "claim_count": len(step9_verified_claims),
    "draft_answer": step9_answer_result,
    "cited_answer": step9_cited_result,
    "citation_integrity": step9_citation_check,
    "abstention_test": step9_abstention_result
}

output_path = os.path.join(
    output_dir,
    "step9_final_verified_answer.json"
)

with open(output_path, "w", encoding="utf-8") as f:
    json.dump(
        step9_final_report,
        f,
        indent=2,
        ensure_ascii=False
    )

print("=" * 60)
print("STEP 9I: FINAL REPORT SAVED")
print("=" * 60)
print("File:", output_path)
print("STEP 9I COMPLETE")


STEP 9I: FINAL REPORT SAVED
File: /content/rag_verification_results/step9_final_verified_answer.json
STEP 9I COMPLETE


In [84]:
# ============================================================
# STEP 9J: CORRECT AND REVERIFY CLAIM 6
# ============================================================

# Candidate correction (not yet verified)
corrected_claim_6 = (
    "The PyPI ecosystem faces security threats "
    "from malicious software packages and "
    "software supply chain attacks."
)

# Find the original Claim 6 safely with a default fallback
claim_6 = next(
    (item for item in step9_verified_claims
     if item["claim_number"] == 6),
    None
)

print("=" * 60)
print("STEP 9J: CLAIM 6 CORRECTION")
print("=" * 60)

if claim_6 is None:
    print("\nCLAIM 6 WAS NOT FOUND IN THE VERIFIED CLAIMS list. Skipping reverification.")
    claim_6_correction_status = "UNVERIFIED"
    verification_checks = []
else:
    print("\nORIGINAL CLAIM:")
    print(claim_6["claim"])

    print("\nCORRECTED CANDIDATE:")
    print(corrected_claim_6)

    print("\nREVERIFYING AGAINST ORIGINAL EVIDENCE...")

    verification_checks = []

    for ev in claim_6["evidence"]:

        evidence_id = ev["evidence_id"]

        # Evidence IDs correspond to 1-based positions in fixed_docs
        doc, score = fixed_docs[evidence_id - 1]

        result = validate_single_citation(
            corrected_claim_6,
            doc,
            llm
        )

        # Use improved parser from Step 8H
        decision = parse_verification_decision(
            result["raw_response"]
        )

        verification_checks.append({
            "evidence_id": evidence_id,
            "page": ev["page"],
            "decision": decision
        })

        print(
            f"E{evidence_id} | "
            f"Page {ev['page']} | "
            f"Decision: {decision}"
        )

    supported = any(
        item["decision"] == "SUPPORTED"
        for item in verification_checks
    )

    contradicted = any(
        item["decision"] == "CONTRADICTED"
        for item in verification_checks
    )

    if supported and not contradicted:
        claim_6_correction_status = "PROVISIONALLY_SUPPORTED"
    elif contradicted:
        claim_6_correction_status = "CONFLICT_REVIEW"
    else:
        claim_6_correction_status = "UNVERIFIED"

print("\nCORRECTION STATUS:")
print(claim_6_correction_status)

print("\nSTEP 9J COMPLETE")

STEP 9J: CLAIM 6 CORRECTION

CLAIM 6 WAS NOT FOUND IN THE VERIFIED CLAIMS list. Skipping reverification.

CORRECTION STATUS:
UNVERIFIED

STEP 9J COMPLETE


In [85]:

# ============================================================
# STEP 9K: UPDATE CLAIM 6 SAFELY
# ============================================================

import copy

step9_corrected_claims = copy.deepcopy(
    step9_verified_claims
)

for item in step9_corrected_claims:

    if item["claim_number"] != 6:
        continue

    if claim_6_correction_status == "PROVISIONALLY_SUPPORTED":

        supported_ids = {
            check["evidence_id"]
            for check in verification_checks
            if check["decision"] == "SUPPORTED"
        }

        item["claim"] = corrected_claim_6

        item["evidence"] = [
            ev for ev in item["evidence"]
            if ev["evidence_id"] in supported_ids
        ]

    else:
        # Exclude the problematic claim if correction
        # cannot be supported by the retrieved evidence.
        item["evidence"] = []

step9_corrected_claims = [
    item for item in step9_corrected_claims
    if item["evidence"]
]

step9_corrected_answer = construct_cited_answer(
    step9_corrected_claims
)

print("=" * 60)
print("STEP 9K: UPDATED CITATION-GROUNDED ANSWER")
print("=" * 60)

print("Status:", step9_corrected_answer["status"])

print("\nUPDATED ANSWER:\n")
print(step9_corrected_answer["answer"])

updated_citation_check = check_citation_integrity(
    step9_corrected_answer,
    fixed_docs
)

print(
    "\nCitation integrity:",
    updated_citation_check["all_valid"]
)

assert updated_citation_check["all_valid"]

print("\nSTEP 9K COMPLETE")


STEP 9K: UPDATED CITATION-GROUNDED ANSWER
Status: CITED_CLAIMS_FOR_REVIEW

UPDATED ANSWER:

This study presents eDySec, an efficient, stable, and explainable DL-based extended Dynamic Security analysis framework for detecting malicious packages. [E1, Page 2] [E4, Page 1] [E5, Page 12] [E6, Page 10] [E7, Page 15]

malicious package detection. Additionally, model stability analysis and explainable AI techniques are incorporated into the detection pipeline. [E1, Page 2] [E4, Page 1]

This study presents eDySec, an efficient, stable, and explainable DL-based framework for malicious package detection using the dynamic QUT-DV25 dataset. [E1, Page 2] [E4, Page 1]

Citation integrity: True

STEP 9K COMPLETE


In [86]:

# ============================================================
# STEP 9L: SAVE CORRECTED FINAL ANSWER
# ============================================================

import os
import json

output_dir = "/content/rag_verification_results"
os.makedirs(output_dir, exist_ok=True)

step9_corrected_report = {
    "question": question,
    "original_claim_count": len(step9_verified_claims),
    "retained_claim_count": len(step9_corrected_claims),
    "claim_6_correction": {
        "original_claim": claim_6["claim"],
        "corrected_candidate": corrected_claim_6,
        "status": claim_6_correction_status,
        "verification_checks": verification_checks
    },
    "final_answer": step9_corrected_answer,
    "citation_integrity": updated_citation_check,
    "abstention_test": step9_abstention_result
}

output_path = os.path.join(
    output_dir,
    "step9_corrected_final_report.json"
)

with open(output_path, "w", encoding="utf-8") as f:
    json.dump(
        step9_corrected_report,
        f,
        indent=2,
        ensure_ascii=False
    )

print("=" * 60)
print("STEP 9L: CORRECTED REPORT SAVED")
print("=" * 60)

print("Retained claims:", len(step9_corrected_claims))
print("Claim 6 status:", claim_6_correction_status)
print("Citation integrity:", updated_citation_check["all_valid"])
print("Report:", output_path)

print("\nSTEP 9L COMPLETE")


TypeError: 'NoneType' object is not subscriptable

#step 10: Define LangGraph State

In [87]:

# ============================================================
# STEP 10A: DEFINE LANGGRAPH WORKFLOW STATE
# ============================================================

from typing import TypedDict, List, Dict, Any
from langgraph.graph import StateGraph, START, END


class RAGState(TypedDict, total=False):

    # User input
    question: str

    # Retrieved and reranked evidence
    retrieved_docs: List[Any]

    # Initial LLM answer
    initial_answer: str

    # Atomic claims extracted from the answer
    atomic_claims: List[Dict[str, Any]]

    # Initial claim verification
    verification_results: List[Dict[str, Any]]

    # Citation validation and claim filtering
    validated_citations: List[Dict[str, Any]]
    filtered_claims: List[Dict[str, Any]]

    # Corrective evidence retrieval
    corrective_results: Dict[str, Any]

    # Final claim decisions
    final_claim_results: List[Dict[str, Any]]

    # Supported claims with citations
    verified_claims: List[Dict[str, Any]]

    # Final output
    final_answer: str
    final_status: str

    # Debugging and execution tracking
    execution_log: List[str]


print("=" * 60)
print("STEP 10A: LANGGRAPH STATE DEFINED")
print("=" * 60)

print("State name: RAGState")
print("Number of state fields:",
      len(RAGState.__annotations__))

print("\nState fields:")

for field in RAGState.__annotations__:
    print("-", field)

print("\nSTEP 10A COMPLETE")



STEP 10A: LANGGRAPH STATE DEFINED
State name: RAGState
Number of state fields: 13

State fields:
- question
- retrieved_docs
- initial_answer
- atomic_claims
- verification_results
- validated_citations
- filtered_claims
- corrective_results
- final_claim_results
- verified_claims
- final_answer
- final_status
- execution_log

STEP 10A COMPLETE


In [88]:

# ============================================================
# STEP 10B: RETRIEVAL AND RERANKING NODE
# ============================================================

def retrieval_node(state: RAGState):

    question = state["question"]

    # Retrieve candidate passages using FAISS
    retrieved_docs = vectorstore.similarity_search(
        question,
        k=8
    )

    # Rerank the retrieved passages
    reranked_docs = rerank_documents(
        question,
        retrieved_docs,
        reranker,
        top_n=4
    )

    # Update LangGraph state
    return {
        "retrieved_docs": reranked_docs,
        "execution_log": state.get(
            "execution_log", []
        ) + ["Retrieval and reranking completed"]
    }


print("=" * 60)
print("STEP 10B: RETRIEVAL NODE CREATED")
print("=" * 60)

print("Node name: retrieval_node")
print("FAISS retrieval: Top 8")
print("Cross-Encoder reranking: Top 4")

print("\nSTEP 10B COMPLETE")


STEP 10B: RETRIEVAL NODE CREATED
Node name: retrieval_node
FAISS retrieval: Top 8
Cross-Encoder reranking: Top 4

STEP 10B COMPLETE


In [89]:

# ============================================================
# STEP 10C: TEST RETRIEVAL NODE
# ============================================================

test_state = {
    "question": "What problem does eDySec aim to solve?",
    "execution_log": []
}

retrieval_test = retrieval_node(test_state)

print("=" * 60)
print("STEP 10C: RETRIEVAL NODE TEST")
print("=" * 60)

print("Retrieved passages:",
      len(retrieval_test["retrieved_docs"]))

for i, (doc, score) in enumerate(
    retrieval_test["retrieved_docs"], start=1
):
    print(f"\nEvidence {i}")
    print("Page:", doc.metadata.get("page_label", "Unknown"))
    print("Reranking score:", round(float(score), 4))
    print("Preview:", doc.page_content[:180])

print("\nExecution log:")
print(retrieval_test["execution_log"])

assert len(retrieval_test["retrieved_docs"]) > 0

print("\nSTEP 10C COMPLETE")


STEP 10C: RETRIEVAL NODE TEST
Retrieved passages: 4

Evidence 1
Page: 9
Reranking score: 0.9496
Preview: better trade-off between attack sensitivity and false-alarm control.
This pattern is particularly important in malicious package detec-
tion, where false negatives allow threats to

Evidence 2
Page: 10
Reranking score: 0.1753
Preview: tendency to over-flag benign samples when considered in isolation.
TCP and Opensnoop demonstrate intermediate behavior. These
findings provide a feature-level explanation for the o

Evidence 3
Page: 15
Reranking score: 0.0006
Preview: eDySec: A Deep Learning-based Explainable Dynamic Analysis Framework for Detecting Malicious Packages in PyPI Ecosystem Conference Name, Month Year, City, Country
Appendix
This app

Evidence 4
Page: 13
Reranking score: -0.4885
Preview: eDySec: A Deep Learning-based Explainable Dynamic Analysis Framework for Detecting Malicious Packages in PyPI Ecosystem Conference Name, Month Year, City, Country
References
[1] Hu

Executi

In [90]:

# ============================================================
# STEP 10D: INITIAL ANSWER GENERATION NODE
# ============================================================

def answer_generation_node(state: RAGState):

    question = state["question"]
    retrieved_docs = state.get("retrieved_docs", [])

    # Handle questions with no retrieved evidence.
    if not retrieved_docs:
        return {
            "initial_answer": "Insufficient evidence.",
            "execution_log": state.get("execution_log", [])
            + ["Answer generation skipped: no evidence"]
        }

    # Build an evidence context from the reranked passages.
    evidence_parts = []

    for i, (doc, score) in enumerate(retrieved_docs, start=1):

        page = doc.metadata.get("page_label", "Unknown")

        evidence_parts.append(
            f"[E{i} | Page {page}]\n{doc.page_content}"
        )

    evidence_context = "\n\n".join(evidence_parts)

    # Instruct the model to use only retrieved evidence.
    prompt = f"""
You are a research question-answering assistant.

Answer the question using ONLY the evidence provided below.

Rules:
1. Do not invent facts or results.
2. Do not use outside knowledge.
3. Use evidence citations such as [E1] or [E2].
4. If the evidence is insufficient, say:
   "Insufficient evidence to answer the question."
5. Keep the answer clear and concise.

QUESTION:
{question}

EVIDENCE:
{evidence_context}

ANSWER:
"""

    initial_answer = llm.invoke(prompt).strip()

    return {
        "initial_answer": initial_answer,
        "execution_log": state.get("execution_log", [])
        + ["Initial answer generation completed"]
    }


print("=" * 60)
print("STEP 10D: ANSWER GENERATION NODE CREATED")
print("=" * 60)
print("Node name: answer_generation_node")
print("Model: Llama 3.1 8B via Ollama")
print("Input: Retrieved evidence")
print("Output: Initial answer")
print()
print("STEP 10D COMPLETE")


STEP 10D: ANSWER GENERATION NODE CREATED
Node name: answer_generation_node
Model: Llama 3.1 8B via Ollama
Input: Retrieved evidence
Output: Initial answer

STEP 10D COMPLETE


In [91]:

# ============================================================
# STEP 10E: TEST ANSWER GENERATION NODE
# ============================================================

print("=" * 60)
print("STEP 10E: ANSWER GENERATION NODE TEST")
print("=" * 60)

# Reuse the actual evidence from Step 10C.
generation_test_state = {
    "question": test_state["question"],
    "retrieved_docs": retrieval_test["retrieved_docs"],
    "execution_log": retrieval_test["execution_log"]
}

generation_test = answer_generation_node(
    generation_test_state
)

print("\nQUESTION:")
print(generation_test_state["question"])

print("\nGENERATED INITIAL ANSWER:")
print(generation_test["initial_answer"])

print("\nEXECUTION LOG:")
for item in generation_test["execution_log"]:
    print("-", item)

assert isinstance(generation_test["initial_answer"], str)
assert generation_test["initial_answer"].strip()

print("\nSTEP 10E COMPLETE")


STEP 10E: ANSWER GENERATION NODE TEST

QUESTION:
What problem does eDySec aim to solve?

GENERATED INITIAL ANSWER:
According to the evidence [E1] and [E2], eDySec aims to solve the problem of malicious package detection in the PyPI ecosystem. Specifically, it aims to improve the trade-off between attack sensitivity and false-alarm control, reducing the number of false negatives (allowing threats to evade detection) and false positives (undermining the practical usability of the detector).

EXECUTION LOG:
- Retrieval and reranking completed
- Initial answer generation completed

STEP 10E COMPLETE


In [92]:

# ============================================================
# STEP 10F: ATOMIC CLAIM EXTRACTION NODE
# ============================================================

import json
import re

def atomic_claim_extraction_node(state: RAGState):

    initial_answer = state.get("initial_answer", "").strip()

    if (
        not initial_answer
        or initial_answer.lower().startswith("insufficient evidence")
    ):
        return {
            "atomic_claims": [],
            "execution_log": state.get("execution_log", [])
            + ["Claim extraction skipped: no answer"]
        }

    prompt = f"""
You are an atomic factual claim extractor.

Break the following answer into individual factual claims.

Rules:
1. Each claim must express one factual statement.
2. Preserve the meaning of the original answer.
3. Do not introduce new information.
4. Do not verify the claims.
5. Return ONLY a valid JSON array.
6. Each item must have the key "claim".

Example:
[
  {{"claim": "The framework detects malicious packages."}},
  {{"claim": "The framework uses dynamic analysis."}}
]

ANSWER:
{initial_answer}
"""

    raw_output = llm.invoke(prompt).strip()

    # Remove optional Markdown code fences.
    cleaned = re.sub(
        r"^```(?:json)?\s*|\s*```$",
        "",
        raw_output,
        flags=re.IGNORECASE
    ).strip()

    try:
        parsed = json.loads(cleaned)

        if not isinstance(parsed, list):
            raise ValueError("Expected a JSON array.")

        claims = []

        for item in parsed:
            if not isinstance(item, dict):
                raise ValueError("Each claim must be an object.")

            claim_text = item.get("claim")

            if not isinstance(claim_text, str):
                raise ValueError("Claim text must be a string.")

            claim_text = claim_text.strip()

            if not claim_text:
                raise ValueError("Empty claim detected.")

            claims.append({"claim": claim_text})

        if not claims:
            raise ValueError("No claims extracted.")

    except (json.JSONDecodeError, ValueError) as error:
        raise ValueError(
            "Atomic claim extraction failed. "
            f"Reason: {error}. Raw output: {raw_output[:500]}"
        ) from error

    return {
        "atomic_claims": claims,
        "execution_log": state.get("execution_log", [])
        + [f"Atomic claim extraction completed: {len(claims)} claims"]
    }


print("=" * 60)
print("STEP 10F: ATOMIC CLAIM EXTRACTION NODE CREATED")
print("=" * 60)
print("Node name: atomic_claim_extraction_node")
print("Input: Initial answer")
print("Output: Atomic claims")
print()
print("STEP 10F COMPLETE")


STEP 10F: ATOMIC CLAIM EXTRACTION NODE CREATED
Node name: atomic_claim_extraction_node
Input: Initial answer
Output: Atomic claims

STEP 10F COMPLETE


In [93]:

# ============================================================
# STEP 10G: TEST ATOMIC CLAIM EXTRACTION NODE
# ============================================================

claim_test_state = {
    "question": generation_test_state["question"],
    "initial_answer": generation_test["initial_answer"],
    "execution_log": generation_test["execution_log"]
}

claim_test = atomic_claim_extraction_node(claim_test_state)

print("=" * 60)
print("STEP 10G: ATOMIC CLAIM EXTRACTION TEST")
print("=" * 60)

print("\nINITIAL ANSWER:")
print(claim_test_state["initial_answer"])

print("\nEXTRACTED CLAIMS:")

for i, item in enumerate(claim_test["atomic_claims"], start=1):
    print(f"{i}. {item['claim']}")

print("\nTOTAL CLAIMS:", len(claim_test["atomic_claims"]))

print("\nEXECUTION LOG:")
for entry in claim_test["execution_log"]:
    print("-", entry)

assert len(claim_test["atomic_claims"]) > 0
assert all(
    isinstance(item.get("claim"), str)
    and item["claim"].strip()
    for item in claim_test["atomic_claims"]
)

print("\nSTEP 10G COMPLETE")


STEP 10G: ATOMIC CLAIM EXTRACTION TEST

INITIAL ANSWER:
According to the evidence [E1] and [E2], eDySec aims to solve the problem of malicious package detection in the PyPI ecosystem. Specifically, it aims to improve the trade-off between attack sensitivity and false-alarm control, reducing the number of false negatives (allowing threats to evade detection) and false positives (undermining the practical usability of the detector).

EXTRACTED CLAIMS:
1. eDySec aims to solve the problem of malicious package detection in the PyPI ecosystem.
2. eDySec aims to improve the trade-off between attack sensitivity and false-alarm control.
3. eDySec aims to reduce the number of false negatives.
4. eDySec aims to reduce the number of false positives.

TOTAL CLAIMS: 4

EXECUTION LOG:
- Retrieval and reranking completed
- Initial answer generation completed
- Atomic claim extraction completed: 4 claims

STEP 10G COMPLETE


In [94]:

# ============================================================
# STEP 10H: EVIDENCE VERIFICATION NODE
# ============================================================

def evidence_verification_node(state: RAGState):

    claims = state.get("atomic_claims", [])
    retrieved_docs = state.get("retrieved_docs", [])

    verification_results = []

    for claim_item in claims:

        claim_text = claim_item["claim"]

        evidence_checks = []

        for i, (doc, score) in enumerate(retrieved_docs, start=1):

            evidence_id = f"E{i}"
            page = doc.metadata.get("page_label", "Unknown")

            # Reuse the existing citation verification function.
            result = validate_single_citation(
                claim_text,
                doc,
                llm
            )

            # Use the improved parser from Step 8H.
            decision = parse_verification_decision(
                result.get("raw_response", "")
            )

            evidence_checks.append({
                "evidence_id": evidence_id,
                "page": page,
                "decision": decision,
                "reranking_score": float(score),
                "raw_response": result.get("raw_response", "")
            })

        supported = [
            item for item in evidence_checks
            if item["decision"] == "SUPPORTED"
        ]

        contradicted = [
            item for item in evidence_checks
            if item["decision"] == "CONTRADICTED"
        ]

        if supported and contradicted:
            status = "CONFLICT_REVIEW"

        elif supported:
            status = "PROVISIONALLY_SUPPORTED"

        elif contradicted:
            status = "CONFLICT_REVIEW"

        else:
            status = "NEEDS_EVIDENCE"

        verification_results.append({
            "claim": claim_text,
            "status": status,
            "evidence_checks": evidence_checks,
            "supported_evidence": [
                item["evidence_id"] for item in supported
            ],
            "contradicted_evidence": [
                item["evidence_id"] for item in contradicted
            ]
        })

    return {
        "verification_results": verification_results,
        "execution_log": state.get("execution_log", [])
        + [
            f"Evidence verification completed: "
            f"{len(verification_results)} claims"
        ]
    }


print("=" * 60)
print("STEP 10H: EVIDENCE VERIFICATION NODE CREATED")
print("=" * 60)
print("Node name: evidence_verification_node")
print("Input: Atomic claims + retrieved evidence")
print("Output: Claim-level verification decisions")
print()
print("STEP 10H COMPLETE")


STEP 10H: EVIDENCE VERIFICATION NODE CREATED
Node name: evidence_verification_node
Input: Atomic claims + retrieved evidence
Output: Claim-level verification decisions

STEP 10H COMPLETE


In [95]:

# ============================================================
# STEP 10I: TEST EVIDENCE VERIFICATION NODE
# ============================================================

verification_test_state = {
    "question": test_state["question"],
    "retrieved_docs": retrieval_test["retrieved_docs"],
    "initial_answer": generation_test["initial_answer"],
    "atomic_claims": claim_test["atomic_claims"],
    "execution_log": claim_test["execution_log"]
}

verification_test = evidence_verification_node(
    verification_test_state
)

print("=" * 60)
print("STEP 10I: EVIDENCE VERIFICATION TEST")
print("=" * 60)

for i, result in enumerate(
    verification_test["verification_results"], start=1
):
    print(f"\nCLAIM {i}: {result['claim']}")
    print("Status:", result["status"])
    print("Supported evidence:", result["supported_evidence"])
    print("Contradicted evidence:", result["contradicted_evidence"])

    for evidence in result["evidence_checks"]:
        print(
            f"  {evidence['evidence_id']} "
            f"(Page {evidence['page']}): "
            f"{evidence['decision']}"
        )

print("\nEXECUTION LOG:")
for entry in verification_test["execution_log"]:
    print("-", entry)

assert len(verification_test["verification_results"]) == len(
    claim_test["atomic_claims"]
)

print("\nSTEP 10I COMPLETE")


STEP 10I: EVIDENCE VERIFICATION TEST

CLAIM 1: eDySec aims to solve the problem of malicious package detection in the PyPI ecosystem.
Status: NEEDS_EVIDENCE
Supported evidence: []
Contradicted evidence: []
  E1 (Page 9): INSUFFICIENT
  E2 (Page 10): INSUFFICIENT
  E3 (Page 15): INSUFFICIENT
  E4 (Page 13): INSUFFICIENT

CLAIM 2: eDySec aims to improve the trade-off between attack sensitivity and false-alarm control.
Status: NEEDS_EVIDENCE
Supported evidence: []
Contradicted evidence: []
  E1 (Page 9): INSUFFICIENT
  E2 (Page 10): INSUFFICIENT
  E3 (Page 15): INSUFFICIENT
  E4 (Page 13): INSUFFICIENT

CLAIM 3: eDySec aims to reduce the number of false negatives.
Status: NEEDS_EVIDENCE
Supported evidence: []
Contradicted evidence: []
  E1 (Page 9): INSUFFICIENT
  E2 (Page 10): INSUFFICIENT
  E3 (Page 15): INSUFFICIENT
  E4 (Page 13): INSUFFICIENT

CLAIM 4: eDySec aims to reduce the number of false positives.
Status: NEEDS_EVIDENCE
Supported evidence: []
Contradicted evidence: []
  E1 (Pa

In [96]:

# ============================================================
# STEP 10J: CLAIM FILTERING AND CORRECTIVE ROUTING NODE
# ============================================================

def claim_filtering_node(state: RAGState):

    verification_results = state.get(
        "verification_results", []
    )

    filtered_claims = []

    for result in verification_results:

        status = result["status"]

        if status == "PROVISIONALLY_SUPPORTED":
            action = "RETAIN_FOR_REVIEW"

        elif status == "NEEDS_EVIDENCE":
            action = "CORRECTIVE_RETRIEVAL"

        elif status == "CONFLICT_REVIEW":
            action = "REVIEW_CONFLICT"

        else:
            action = "ABSTAIN"

        filtered_claims.append({
            **result,
            "action": action
        })

    return {
        "filtered_claims": filtered_claims,
        "execution_log": state.get("execution_log", [])
        + ["Claim filtering and routing completed"]
    }


print("=" * 60)
print("STEP 10J: CLAIM FILTERING NODE CREATED")
print("=" * 60)
print("Node name: claim_filtering_node")
print("Input: Verification results")
print("Output: Filtered claims and routing actions")
print()
print("STEP 10J COMPLETE")


STEP 10J: CLAIM FILTERING NODE CREATED
Node name: claim_filtering_node
Input: Verification results
Output: Filtered claims and routing actions

STEP 10J COMPLETE


In [97]:

# ============================================================
# STEP 10K: TEST CLAIM FILTERING NODE
# ============================================================

filtering_test_state = {
    "verification_results": verification_test[
        "verification_results"
    ],
    "execution_log": verification_test["execution_log"]
}

filtering_test = claim_filtering_node(
    filtering_test_state
)

print("=" * 60)
print("STEP 10K: CLAIM FILTERING TEST")
print("=" * 60)

for i, item in enumerate(
    filtering_test["filtered_claims"], start=1
):
    print(f"\nCLAIM {i}: {item['claim']}")
    print("Status:", item["status"])
    print("Action:", item["action"])

print("\nEXECUTION LOG:")
for entry in filtering_test["execution_log"]:
    print("-", entry)

assert len(filtering_test["filtered_claims"]) == len(
    verification_test["verification_results"]
)

print("\nSTEP 10K COMPLETE")


STEP 10K: CLAIM FILTERING TEST

CLAIM 1: eDySec aims to solve the problem of malicious package detection in the PyPI ecosystem.
Status: NEEDS_EVIDENCE
Action: CORRECTIVE_RETRIEVAL

CLAIM 2: eDySec aims to improve the trade-off between attack sensitivity and false-alarm control.
Status: NEEDS_EVIDENCE
Action: CORRECTIVE_RETRIEVAL

CLAIM 3: eDySec aims to reduce the number of false negatives.
Status: NEEDS_EVIDENCE
Action: CORRECTIVE_RETRIEVAL

CLAIM 4: eDySec aims to reduce the number of false positives.
Status: NEEDS_EVIDENCE
Action: CORRECTIVE_RETRIEVAL

EXECUTION LOG:
- Retrieval and reranking completed
- Initial answer generation completed
- Atomic claim extraction completed: 4 claims
- Evidence verification completed: 4 claims
- Claim filtering and routing completed

STEP 10K COMPLETE


In [98]:

# ============================================================
# STEP 10L: CORRECTIVE RETRIEVAL NODE
# ============================================================

def corrective_retrieval_node(state: RAGState):

    filtered_claims = state.get("filtered_claims", [])

    corrective_results = {}

    for item in filtered_claims:

        if item.get("action") != "CORRECTIVE_RETRIEVAL":
            continue

        claim_text = item["claim"]

        # Reuse the Step 8 corrective verification pipeline.
        result = corrective_verify_claim(
            claim=claim_text,
            vectorstore=vectorstore,
            reranker=reranker,
            llm=llm,
            max_retries=2
        )

        corrective_results[claim_text] = result

    return {
        "corrective_results": corrective_results,
        "execution_log": state.get("execution_log", [])
        + [
            f"Corrective retrieval completed: "
            f"{len(corrective_results)} claims processed"
        ]
    }


print("=" * 60)
print("STEP 10L: CORRECTIVE RETRIEVAL NODE CREATED")
print("=" * 60)
print("Node name: corrective_retrieval_node")
print("Input: Claims requiring additional evidence")
print("Output: Corrective verification results")
print()
print("STEP 10L COMPLETE")


STEP 10L: CORRECTIVE RETRIEVAL NODE CREATED
Node name: corrective_retrieval_node
Input: Claims requiring additional evidence
Output: Corrective verification results

STEP 10L COMPLETE


In [99]:

# ============================================================
# STEP 10M: TEST CORRECTIVE RETRIEVAL NODE
# ============================================================

corrective_test_state = {
    "filtered_claims": filtering_test["filtered_claims"],
    "execution_log": filtering_test["execution_log"]
}

corrective_test = corrective_retrieval_node(
    corrective_test_state
)

print("=" * 60)
print("STEP 10M: CORRECTIVE RETRIEVAL TEST")
print("=" * 60)

for claim, result in corrective_test["corrective_results"].items():

    print("\nCLAIM:", claim)
    print("Final status:", result.get("status"))
    print("Attempts used:", result.get("attempts_used"))
    print(
        "Supported passages:",
        len(result.get("supported_passages", []))
    )
    print(
        "Contradicted passages:",
        len(result.get("contradicted_passages", []))
    )

print("\nEXECUTION LOG:")
for entry in corrective_test["execution_log"]:
    print("-", entry)

expected_count = sum(
    item.get("action") == "CORRECTIVE_RETRIEVAL"
    for item in filtering_test["filtered_claims"]
)

assert len(corrective_test["corrective_results"]) == expected_count

print("\nSTEP 10M COMPLETE")



Retrieval attempt 1/2
New passages: 4
Page: 1 | Decision: UNKNOWN
Page: 11 | Decision: UNKNOWN
Page: 3 | Decision: UNKNOWN
Page: 15 | Decision: UNKNOWN

Retrieval attempt 2/2
New passages: 0

Retrieval attempt 1/2
New passages: 4
Page: 9 | Decision: UNKNOWN
Page: 9 | Decision: UNKNOWN
Page: 12 | Decision: UNKNOWN
Page: 2 | Decision: UNKNOWN

Retrieval attempt 2/2
New passages: 0

Retrieval attempt 1/2
New passages: 4
Page: 9 | Decision: UNKNOWN
Page: 2 | Decision: UNKNOWN
Page: 9 | Decision: UNKNOWN
Page: 7 | Decision: UNKNOWN

Retrieval attempt 2/2
New passages: 0

Retrieval attempt 1/2
New passages: 4
Page: 9 | Decision: UNKNOWN
Page: 2 | Decision: UNKNOWN
Page: 9 | Decision: UNKNOWN
Page: 7 | Decision: UNKNOWN

Retrieval attempt 2/2
New passages: 0
STEP 10M: CORRECTIVE RETRIEVAL TEST

CLAIM: eDySec aims to solve the problem of malicious package detection in the PyPI ecosystem.
Final status: UNVERIFIED
Attempts used: 2
Supported passages: 0
Contradicted passages: 0

CLAIM: eDySec ai

In [100]:

# ============================================================
# STEP 10N: DIAGNOSE UNKNOWN VERIFICATION DECISIONS
# ============================================================

print("=" * 60)
print("STEP 10N: UNKNOWN DECISION DIAGNOSTIC")
print("=" * 60)

for claim, result in corrective_test["corrective_results"].items():

    print("\nCLAIM:", claim)
    print("Final status:", result.get("status"))

    history = result.get("verification_history", [])

    print("History entries:", len(history))

    if not history:
        print("No verification history available.")
        continue

    # Inspect only the first entry to avoid excessive output.
    first_entry = history[0]

    print("\nFirst history entry type:")
    print(type(first_entry).__name__)

    print("\nFirst history entry:")
    print(str(first_entry)[:1800])

    print("\n" + "-" * 60)

print("\nSTEP 10N COMPLETE")


STEP 10N: UNKNOWN DECISION DIAGNOSTIC

CLAIM: eDySec aims to solve the problem of malicious package detection in the PyPI ecosystem.
Final status: UNVERIFIED
History entries: 4

First history entry type:
dict

First history entry:
{'attempt': 1, 'page': '1', 'decision': 'UNKNOWN', 'score': 6.950473785400391, 'raw_response': 'INSUFFICIENT\n\nThe passage supports the claim that eDySec aims to solve the problem of malicious package detection in the PyPI ecosystem, but it does not directly support the fact that eDySec is a "Deep Learning-based Explainable Dynamic Analysis Framework". The passage mentions that eDySec is a "Deep Learning-based" framework, but it does not explicitly state that it is also "Explainable" and "Dynamic Analysis".'}

------------------------------------------------------------

CLAIM: eDySec aims to improve the trade-off between attack sensitivity and false-alarm control.
Final status: UNVERIFIED
History entries: 4

First history entry type:
dict

First history ent

In [101]:

# ============================================================
# STEP 10O: REPAIR CORRECTIVE VERIFICATION DECISIONS
# ============================================================

import copy

# Preserve original results for reproducibility.
corrective_test_repaired = copy.deepcopy(corrective_test)

valid_decisions = {
    "SUPPORTED",
    "CONTRADICTED",
    "INSUFFICIENT"
}

total_repaired = 0

for claim, result in corrective_test_repaired[
    "corrective_results"
].items():

    history = result.get("verification_history", [])

    for entry in history:

        old_decision = entry.get("decision", "UNKNOWN")
        raw_response = entry.get("raw_response", "")

        new_decision = parse_verification_decision(
            raw_response
        )

        if new_decision not in valid_decisions:
            new_decision = "UNKNOWN"

        entry["decision"] = new_decision

        if old_decision != new_decision:
            total_repaired += 1

    supported = [
        entry for entry in history
        if entry["decision"] == "SUPPORTED"
    ]

    contradicted = [
        entry for entry in history
        if entry["decision"] == "CONTRADICTED"
    ]

    insufficient = [
        entry for entry in history
        if entry["decision"] == "INSUFFICIENT"
    ]

    # Do not invent supported passages from raw text.
    # Only preserve passages already recorded by the pipeline.
    if supported and contradicted:
        final_status = "CONFLICT_REVIEW"

    elif supported:
        final_status = "PROVISIONALLY_SUPPORTED"

    elif contradicted:
        final_status = "CONFLICT_REVIEW"

    else:
        final_status = "UNVERIFIED"

    result["status"] = final_status

    print("\nCLAIM:", claim)
    print("SUPPORTED:", len(supported))
    print("CONTRADICTED:", len(contradicted))
    print("INSUFFICIENT:", len(insufficient))
    print("UNKNOWN:", sum(
        e["decision"] == "UNKNOWN" for e in history
    ))
    print("Repaired status:", final_status)

print("\n" + "=" * 60)
print("STEP 10O: PARSER REPAIR RESULTS")
print("=" * 60)
print("Decisions repaired:", total_repaired)
print("No additional LLM calls made.")

print("\nSTEP 10O COMPLETE")



CLAIM: eDySec aims to solve the problem of malicious package detection in the PyPI ecosystem.
SUPPORTED: 0
CONTRADICTED: 0
INSUFFICIENT: 4
UNKNOWN: 0
Repaired status: UNVERIFIED

CLAIM: eDySec aims to improve the trade-off between attack sensitivity and false-alarm control.
SUPPORTED: 0
CONTRADICTED: 0
INSUFFICIENT: 4
UNKNOWN: 0
Repaired status: UNVERIFIED

CLAIM: eDySec aims to reduce the number of false negatives.
SUPPORTED: 0
CONTRADICTED: 0
INSUFFICIENT: 4
UNKNOWN: 0
Repaired status: UNVERIFIED

CLAIM: eDySec aims to reduce the number of false positives.
SUPPORTED: 0
CONTRADICTED: 0
INSUFFICIENT: 4
UNKNOWN: 0
Repaired status: UNVERIFIED

STEP 10O: PARSER REPAIR RESULTS
Decisions repaired: 16
No additional LLM calls made.

STEP 10O COMPLETE


In [102]:

# ============================================================
# STEP 10P: FINAL CLAIM DECISION NODE
# ============================================================

def final_claim_decision_node(state: RAGState):

    filtered_claims = state.get("filtered_claims", [])
    corrective_results = state.get("corrective_results", {})

    final_claim_results = []

    for item in filtered_claims:

        claim = item["claim"]
        initial_status = item["status"]
        action = item["action"]

        corrective = corrective_results.get(claim, {})
        corrective_status = corrective.get("status")

        if initial_status == "CONFLICT_REVIEW":
            final_status = "CONFLICT_REVIEW"

        elif corrective_status == "CONFLICT_REVIEW":
            final_status = "CONFLICT_REVIEW"

        elif initial_status == "PROVISIONALLY_SUPPORTED":
            final_status = "PROVISIONALLY_SUPPORTED"

        elif corrective_status == "PROVISIONALLY_SUPPORTED":
            final_status = "PROVISIONALLY_SUPPORTED"

        else:
            final_status = "UNVERIFIED"

        if final_status == "PROVISIONALLY_SUPPORTED":
            final_action = "RETAIN_FOR_REVIEW"

        elif final_status == "CONFLICT_REVIEW":
            final_action = "REVIEW_CONFLICT"

        else:
            final_action = "EXCLUDE_FROM_ANSWER"

        final_claim_results.append({
            "claim": claim,
            "initial_status": initial_status,
            "initial_action": action,
            "corrective_status": corrective_status,
            "final_status": final_status,
            "final_action": final_action,
            "initial_supported_evidence": item.get(
                "supported_evidence", []
            ),
            "corrective_supported_passages": corrective.get(
                "supported_passages", []
            )
        })

    return {
        "final_claim_results": final_claim_results,
        "execution_log": state.get("execution_log", [])
        + ["Final claim decisions completed"]
    }


print("=" * 60)
print("STEP 10P: FINAL CLAIM DECISION NODE CREATED")
print("=" * 60)
print("Node name: final_claim_decision_node")
print("Input: Initial + corrective verification results")
print("Output: Final claim decisions")
print()
print("STEP 10P COMPLETE")


STEP 10P: FINAL CLAIM DECISION NODE CREATED
Node name: final_claim_decision_node
Input: Initial + corrective verification results
Output: Final claim decisions

STEP 10P COMPLETE


In [103]:

# ============================================================
# STEP 10Q: TEST FINAL CLAIM DECISION NODE
# ============================================================

final_decision_test_state = {
    "filtered_claims": filtering_test["filtered_claims"],
    "corrective_results": corrective_test_repaired[
        "corrective_results"
    ],
    "execution_log": corrective_test["execution_log"]
}

final_decision_test = final_claim_decision_node(
    final_decision_test_state
)

print("=" * 60)
print("STEP 10Q: FINAL CLAIM DECISION TEST")
print("=" * 60)

for i, item in enumerate(
    final_decision_test["final_claim_results"], start=1
):
    print(f"\nCLAIM {i}: {item['claim']}")
    print("Initial status:", item["initial_status"])
    print("Corrective status:", item["corrective_status"])
    print("Final status:", item["final_status"])
    print("Final action:", item["final_action"])

print("\nEXECUTION LOG:")
for entry in final_decision_test["execution_log"]:
    print("-", entry)

# FIX: Dynamically assert the length matches the number of input filtered claims rather than hardcoding 3
assert len(final_decision_test["final_claim_results"]) == len(filtering_test["filtered_claims"])

print("\nSTEP 10Q COMPLETE")


STEP 10Q: FINAL CLAIM DECISION TEST

CLAIM 1: eDySec aims to solve the problem of malicious package detection in the PyPI ecosystem.
Initial status: NEEDS_EVIDENCE
Corrective status: UNVERIFIED
Final status: UNVERIFIED
Final action: EXCLUDE_FROM_ANSWER

CLAIM 2: eDySec aims to improve the trade-off between attack sensitivity and false-alarm control.
Initial status: NEEDS_EVIDENCE
Corrective status: UNVERIFIED
Final status: UNVERIFIED
Final action: EXCLUDE_FROM_ANSWER

CLAIM 3: eDySec aims to reduce the number of false negatives.
Initial status: NEEDS_EVIDENCE
Corrective status: UNVERIFIED
Final status: UNVERIFIED
Final action: EXCLUDE_FROM_ANSWER

CLAIM 4: eDySec aims to reduce the number of false positives.
Initial status: NEEDS_EVIDENCE
Corrective status: UNVERIFIED
Final status: UNVERIFIED
Final action: EXCLUDE_FROM_ANSWER

EXECUTION LOG:
- Retrieval and reranking completed
- Initial answer generation completed
- Atomic claim extraction completed: 4 claims
- Evidence verification co

In [104]:

# ============================================================
# STEP 10R: FINAL ANSWER AND ABSTENTION NODE
# ============================================================

def final_answer_node(state: RAGState):

    final_claim_results = state.get(
        "final_claim_results", []
    )

    retrieved_docs = state.get("retrieved_docs", [])

    accepted_claims = []
    cited_sentences = []

    for item in final_claim_results:

        if item.get("final_status") != "PROVISIONALLY_SUPPORTED":
            continue

        claim = item["claim"]

        # Evidence IDs must correspond to actual retrieved passages.
        valid_evidence_ids = []

        for evidence_id in item.get(
            "initial_supported_evidence", []
        ):
            if not isinstance(evidence_id, str):
                continue

            if not evidence_id.startswith("E"):
                continue

            try:
                index = int(evidence_id[1:]) - 1
            except ValueError:
                continue

            if 0 <= index < len(retrieved_docs):
                valid_evidence_ids.append(evidence_id)

        # Corrective evidence requires a separate evidence registry.
        # Do not cite corrective passages using initial E IDs.

        if not valid_evidence_ids:
            continue

        unique_ids = list(dict.fromkeys(valid_evidence_ids))

        citations = " ".join(
            f"[{evidence_id}]"
            for evidence_id in unique_ids
        )

        cited_sentences.append(
            f"{claim} {citations}"
        )

        accepted_claims.append({
            "claim": claim,
            "evidence_ids": unique_ids
        })

    if cited_sentences:

        final_answer = "\n".join(cited_sentences)
        final_status = "DRAFT_FOR_REVIEW"

    else:

        final_answer = (
            "Insufficient verified evidence to provide "
            "a reliable answer to this question."
        )

        final_status = "ABSTAIN"

    return {
        "verified_claims": accepted_claims,
        "final_answer": final_answer,
        "final_status": final_status,
        "execution_log": state.get("execution_log", [])
        + [f"Final answer decision completed: {final_status}"]
    }


print("=" * 60)
print("STEP 10R: FINAL ANSWER NODE CREATED")
print("=" * 60)
print("Node name: final_answer_node")
print("Input: Final claim decisions + evidence")
print("Output: Final answer or abstention")
print()
print("STEP 10R COMPLETE")


STEP 10R: FINAL ANSWER NODE CREATED
Node name: final_answer_node
Input: Final claim decisions + evidence
Output: Final answer or abstention

STEP 10R COMPLETE


In [105]:

# ============================================================
# STEP 10S: TEST FINAL ANSWER AND ABSTENTION
# ============================================================

final_answer_test_state = {
    "question": test_state["question"],
    "retrieved_docs": retrieval_test["retrieved_docs"],
    "final_claim_results": final_decision_test[
        "final_claim_results"
    ],
    "execution_log": final_decision_test["execution_log"]
}

final_answer_test = final_answer_node(
    final_answer_test_state
)

print("=" * 60)
print("STEP 10S: FINAL ANSWER AND ABSTENTION TEST")
print("=" * 60)

print("\nQUESTION:")
print(final_answer_test_state["question"])

print("\nFINAL STATUS:")
print(final_answer_test["final_status"])

print("\nFINAL ANSWER:")
print(final_answer_test["final_answer"])

print("\nACCEPTED CLAIMS:")
print(len(final_answer_test["verified_claims"]))

print("\nEXECUTION LOG:")
for entry in final_answer_test["execution_log"]:
    print("-", entry)

assert final_answer_test["final_status"] == "ABSTAIN"
assert len(final_answer_test["verified_claims"]) == 0

print("\nSTEP 10S COMPLETE")


STEP 10S: FINAL ANSWER AND ABSTENTION TEST

QUESTION:
What problem does eDySec aim to solve?

FINAL STATUS:
ABSTAIN

FINAL ANSWER:
Insufficient verified evidence to provide a reliable answer to this question.

ACCEPTED CLAIMS:
0

EXECUTION LOG:
- Retrieval and reranking completed
- Initial answer generation completed
- Atomic claim extraction completed: 4 claims
- Evidence verification completed: 4 claims
- Claim filtering and routing completed
- Corrective retrieval completed: 4 claims processed
- Final claim decisions completed
- Final answer decision completed: ABSTAIN

STEP 10S COMPLETE


In [106]:

# ============================================================
# STEP 10T: BUILD COMPLETE LANGGRAPH WORKFLOW
# ============================================================

from langgraph.graph import StateGraph, START, END

workflow = StateGraph(RAGState)

# Register the nodes created in previous steps.
workflow.add_node("retrieval", retrieval_node)

workflow.add_node(
    "answer_generation",
    answer_generation_node
)

workflow.add_node(
    "claim_extraction",
    atomic_claim_extraction_node
)

workflow.add_node(
    "evidence_verification",
    evidence_verification_node
)

workflow.add_node(
    "claim_filtering",
    claim_filtering_node
)

workflow.add_node(
    "corrective_retrieval",
    corrective_retrieval_node
)

workflow.add_node(
    "final_claim_decisions",
    final_claim_decision_node
)

workflow.add_node(
    "final_answer",
    final_answer_node
)

# Connect nodes in execution order.
workflow.add_edge(START, "retrieval")

workflow.add_edge(
    "retrieval",
    "answer_generation"
)

workflow.add_edge(
    "answer_generation",
    "claim_extraction"
)

workflow.add_edge(
    "claim_extraction",
    "evidence_verification"
)

workflow.add_edge(
    "evidence_verification",
    "claim_filtering"
)

workflow.add_edge(
    "claim_filtering",
    "corrective_retrieval"
)

workflow.add_edge(
    "corrective_retrieval",
    "final_claim_decisions"
)

workflow.add_edge(
    "final_claim_decisions",
    "final_answer"
)

workflow.add_edge("final_answer", END)

# Compile the graph.
rag_graph = workflow.compile()

print("=" * 60)
print("STEP 10T: LANGGRAPH WORKFLOW COMPILED")
print("=" * 60)

print("Total nodes: 8")
print("Entry node: retrieval")
print("Final node: final_answer")
print("Graph compiled:", rag_graph is not None)

print("\nSTEP 10T COMPLETE")


STEP 10T: LANGGRAPH WORKFLOW COMPILED
Total nodes: 8
Entry node: retrieval
Final node: final_answer
Graph compiled: True

STEP 10T COMPLETE


In [107]:

# ============================================================
# STEP 10U: VERIFY LANGGRAPH STRUCTURE
# ============================================================

print("=" * 60)
print("STEP 10U: LANGGRAPH STRUCTURE TEST")
print("=" * 60)

graph_structure = rag_graph.get_graph()

print("\nREGISTERED NODES:")
for node_name in graph_structure.nodes:
    print("-", node_name)

print("\nGRAPH EDGES:")
for edge in graph_structure.edges:
    print(f"{edge.source} -> {edge.target}")

expected_nodes = {
    "retrieval",
    "answer_generation",
    "claim_extraction",
    "evidence_verification",
    "claim_filtering",
    "corrective_retrieval",
    "final_claim_decisions",
    "final_answer"
}

assert expected_nodes.issubset(
    set(graph_structure.nodes)
)

print("\nAll 8 required nodes are present.")
print("STEP 10U COMPLETE")


STEP 10U: LANGGRAPH STRUCTURE TEST

REGISTERED NODES:
- __start__
- retrieval
- answer_generation
- claim_extraction
- evidence_verification
- claim_filtering
- corrective_retrieval
- final_claim_decisions
- final_answer
- __end__

GRAPH EDGES:
__start__ -> retrieval
answer_generation -> claim_extraction
claim_extraction -> evidence_verification
claim_filtering -> corrective_retrieval
corrective_retrieval -> final_claim_decisions
evidence_verification -> claim_filtering
final_claim_decisions -> final_answer
retrieval -> answer_generation
final_answer -> __end__

All 8 required nodes are present.
STEP 10U COMPLETE


In [108]:

# ============================================================
# STEP 10V: PERMANENT CORRECTIVE PARSER FIX
# ============================================================

# Preserve the original corrective verification function.
original_corrective_verify_claim = corrective_verify_claim

def corrected_corrective_verify_claim(
    claim,
    vectorstore,
    reranker,
    llm,
    max_retries=2
):
    result = original_corrective_verify_claim(
        claim=claim,
        vectorstore=vectorstore,
        reranker=reranker,
        llm=llm,
        max_retries=max_retries
    )

    valid_decisions = {
        "SUPPORTED",
        "CONTRADICTED",
        "INSUFFICIENT"
    }

    history = result.get("verification_history", [])

    for entry in history:

        raw_response = entry.get("raw_response", "")

        decision = parse_verification_decision(
            raw_response
        )

        if decision not in valid_decisions:
            decision = "UNKNOWN"

        entry["decision"] = decision

    supported = [
        entry for entry in history
        if entry["decision"] == "SUPPORTED"
    ]

    contradicted = [
        entry for entry in history
        if entry["decision"] == "CONTRADICTED"
    ]

    if supported and contradicted:
        result["status"] = "CONFLICT_REVIEW"

    elif supported:
        result["status"] = "PROVISIONALLY_SUPPORTED"

    elif contradicted:
        result["status"] = "CONFLICT_REVIEW"

    else:
        result["status"] = "UNVERIFIED"

    return result


print("=" * 60)
print("STEP 10V: CORRECTIVE PARSER WRAPPER CREATED")
print("=" * 60)
print("Wrapper: corrected_corrective_verify_claim")
print("Original function preserved.")
print()
print("STEP 10V COMPLETE")


STEP 10V: CORRECTIVE PARSER WRAPPER CREATED
Wrapper: corrected_corrective_verify_claim
Original function preserved.

STEP 10V COMPLETE


In [109]:

# ============================================================
# STEP 10W: UPDATE CORRECTIVE RETRIEVAL NODE
# ============================================================

def corrective_retrieval_node(state: RAGState):

    filtered_claims = state.get("filtered_claims", [])
    corrective_results = {}

    for item in filtered_claims:

        if item.get("action") != "CORRECTIVE_RETRIEVAL":
            continue

        claim_text = item["claim"]

        result = corrected_corrective_verify_claim(
            claim=claim_text,
            vectorstore=vectorstore,
            reranker=reranker,
            llm=llm,
            max_retries=2
        )

        corrective_results[claim_text] = result

    return {
        "corrective_results": corrective_results,
        "execution_log": state.get("execution_log", [])
        + [
            f"Corrective retrieval completed: "
            f"{len(corrective_results)} claims processed"
        ]
    }

print("=" * 60)
print("STEP 10W: CORRECTIVE NODE UPDATED")
print("=" * 60)
print("Uses corrected parser wrapper.")
print("STEP 10W COMPLETE")


STEP 10W: CORRECTIVE NODE UPDATED
Uses corrected parser wrapper.
STEP 10W COMPLETE


In [110]:

# ============================================================
# STEP 10X: RECOMPILE UPDATED LANGGRAPH
# ============================================================

updated_workflow = StateGraph(RAGState)

nodes = {
    "retrieval": retrieval_node,
    "answer_generation": answer_generation_node,
    "claim_extraction": atomic_claim_extraction_node,
    "evidence_verification": evidence_verification_node,
    "claim_filtering": claim_filtering_node,
    "corrective_retrieval": corrective_retrieval_node,
    "final_claim_decisions": final_claim_decision_node,
    "final_answer": final_answer_node
}

for name, function in nodes.items():
    updated_workflow.add_node(name, function)

node_order = list(nodes.keys())

updated_workflow.add_edge(START, node_order[0])

for current, next_node in zip(
    node_order[:-1],
    node_order[1:]
):
    updated_workflow.add_edge(current, next_node)

updated_workflow.add_edge(node_order[-1], END)

rag_graph = updated_workflow.compile()

assert rag_graph is not None

print("=" * 60)
print("STEP 10X: UPDATED LANGGRAPH COMPILED")
print("=" * 60)
print("Nodes:", len(nodes))
print("Corrective parser wrapper connected.")
print("STEP 10X COMPLETE")


STEP 10X: UPDATED LANGGRAPH COMPILED
Nodes: 8
Corrective parser wrapper connected.
STEP 10X COMPLETE


In [111]:

# ============================================================
# STEP 10Y: FIRST END-TO-END LANGGRAPH EXECUTION
# ============================================================

import time

print("=" * 60)
print("STEP 10Y: END-TO-END LANGGRAPH EXECUTION")
print("=" * 60)

test_question = "What problem does eDySec aim to solve?"

initial_state = {
    "question": test_question,
    "execution_log": []
}

start_time = time.perf_counter()

# Execute all 8 nodes automatically.
end_to_end_result = rag_graph.invoke(initial_state)

elapsed_time = time.perf_counter() - start_time

print("\nQUESTION:")
print(test_question)

print("\nINITIAL ANSWER:")
print(end_to_end_result.get("initial_answer", ""))

print("\nEXTRACTED CLAIMS:")
for i, item in enumerate(
    end_to_end_result.get("atomic_claims", []),
    start=1
):
    print(f"{i}. {item['claim']}")

print("\nFINAL CLAIM DECISIONS:")
for i, item in enumerate(
    end_to_end_result.get("final_claim_results", []),
    start=1
):
    print(f"\nClaim {i}: {item['claim']}")
    print("Status:", item["final_status"])
    print("Action:", item["final_action"])

print("\nFINAL STATUS:")
print(end_to_end_result.get("final_status"))

print("\nFINAL ANSWER:")
print(end_to_end_result.get("final_answer"))

print("\nEXECUTION LOG:")
for entry in end_to_end_result.get("execution_log", []):
    print("-", entry)

print("\nEXECUTION TIME:")
print(f"{elapsed_time:.2f} seconds")

assert end_to_end_result.get("final_status") in {
    "ABSTAIN",
    "DRAFT_FOR_REVIEW"
}

assert end_to_end_result.get("final_answer")

assert len(end_to_end_result.get("execution_log", [])) == 8

print("\nSTEP 10Y COMPLETE")


STEP 10Y: END-TO-END LANGGRAPH EXECUTION

Retrieval attempt 1/2
New passages: 4
Page: 1 | Decision: UNKNOWN
Page: 11 | Decision: UNKNOWN
Page: 3 | Decision: UNKNOWN
Page: 15 | Decision: UNKNOWN

Retrieval attempt 2/2
New passages: 0

Retrieval attempt 1/2
New passages: 4
Page: 9 | Decision: UNKNOWN
Page: 9 | Decision: UNKNOWN
Page: 1 | Decision: UNKNOWN
Page: 2 | Decision: UNKNOWN

Retrieval attempt 2/2
New passages: 0

Retrieval attempt 1/2
New passages: 4
Page: 2 | Decision: UNKNOWN
Page: 9 | Decision: UNKNOWN
Page: 7 | Decision: UNKNOWN
Page: 9 | Decision: UNKNOWN

Retrieval attempt 2/2
New passages: 0

Retrieval attempt 1/2
New passages: 4
Page: 2 | Decision: UNKNOWN
Page: 7 | Decision: UNKNOWN
Page: 10 | Decision: UNKNOWN
Page: 9 | Decision: UNKNOWN

Retrieval attempt 2/2
New passages: 0

Retrieval attempt 1/2
New passages: 4
Page: 9 | Decision: UNKNOWN
Page: 2 | Decision: UNKNOWN
Page: 2 | Decision: UNKNOWN
Page: 7 | Decision: UNKNOWN

Retrieval attempt 2/2
New passages: 0

QUEST

In [112]:

# ============================================================
# STEP 10: CONSOLIDATED DEBUG FIX
# ============================================================

import inspect
import re
from langgraph.graph import StateGraph, START, END

# 1. Robust decision parser
def parse_verification_decision(raw_response):
    if not isinstance(raw_response, str):
        return "UNKNOWN"

    text = raw_response.strip().upper()

    match = re.search(
        r"(?m)^\s*(?:DECISION\s*:\s*)?"
        r"(SUPPORTED|CONTRADICTED|INSUFFICIENT)\b",
        text
    )

    return match.group(1) if match else "UNKNOWN"


# 2. Patch the original citation validator's returned decision.
# This fixes callers that use validate_single_citation().
original_validate_single_citation = validate_single_citation

def validate_single_citation(claim, doc, llm):
    result = original_validate_single_citation(
        claim, doc, llm
    )

    result["decision"] = parse_verification_decision(
        result.get("raw_response", "")
    )

    return result


# 3. Corrective node: use original corrective function.
# Its internal behavior depends on how it calls the validator.
def corrective_retrieval_node(state: RAGState):

    results = {}

    for item in state.get("filtered_claims", []):

        if item.get("action") != "CORRECTIVE_RETRIEVAL":
            continue

        claim = item["claim"]

        result = original_corrective_verify_claim(
            claim=claim,
            vectorstore=vectorstore,
            reranker=reranker,
            llm=llm,
            max_retries=2
        )

        history = result.get("verification_history", [])

        for entry in history:
            entry["decision"] = parse_verification_decision(
                entry.get("raw_response", "")
            )

        supported = [
            entry for entry in history
            if entry["decision"] == "SUPPORTED"
        ]

        contradicted = [
            entry for entry in history
            if entry["decision"] == "CONTRADICTED"
        ]

        if supported or contradicted:
            # Do not mark a conflicting claim as supported.
            result["status"] = (
                "CONFLICT_REVIEW"
                if contradicted
                else "PROVISIONALLY_SUPPORTED"
            )
        else:
            result["status"] = "UNVERIFIED"

        results[claim] = result

    return {
        "corrective_results": results,
        "execution_log": state.get("execution_log", [])
        + [f"Corrective retrieval completed: {len(results)} claims processed"]
    }


# 4. Rebuild the complete graph.
workflow_fixed = StateGraph(RAGState)

node_functions = {
    "retrieval": retrieval_node,
    "answer_generation": answer_generation_node,
    "claim_extraction": atomic_claim_extraction_node,
    "evidence_verification": evidence_verification_node,
    "claim_filtering": claim_filtering_node,
    "corrective_retrieval": corrective_retrieval_node,
    "final_claim_decisions": final_claim_decision_node,
    "final_answer": final_answer_node
}

for name, fn in node_functions.items():
    workflow_fixed.add_node(name, fn)

order = list(node_functions)

workflow_fixed.add_edge(START, order[0])

for a, b in zip(order[:-1], order[1:]):
    workflow_fixed.add_edge(a, b)

workflow_fixed.add_edge(order[-1], END)

rag_graph = workflow_fixed.compile()


# 5. Quick checks without Ollama calls.
assert parse_verification_decision(
    "INSUFFICIENT\n\nThe evidence does not support the claim."
) == "INSUFFICIENT"

assert parse_verification_decision(
    "SUPPORTED\nThe passage directly supports the claim."
) == "SUPPORTED"

assert parse_verification_decision(
    "CONTRADICTED\nThe passage contradicts the claim."
) == "CONTRADICTED"

assert len(node_functions) == 8

print("=" * 60)
print("CONSOLIDATED DEBUG FIX COMPLETE")
print("=" * 60)
print("Decision parser: PASS")
print("Citation validator wrapper: INSTALLED")
print("Corrective node: UPDATED")
print("LangGraph: RECOMPILED")
print("Nodes:", len(node_functions))
print("No additional LLM calls made.")


CONSOLIDATED DEBUG FIX COMPLETE
Decision parser: PASS
Citation validator wrapper: INSTALLED
Corrective node: UPDATED
LangGraph: RECOMPILED
Nodes: 8
No additional LLM calls made.


In [113]:

# ============================================================
# STEP 10: FINAL END-TO-END TEST
# ============================================================

import time
from collections import Counter

question = "What problem does eDySec aim to solve?"

start = time.perf_counter()

result = rag_graph.invoke({
    "question": question,
    "execution_log": []
})

elapsed = time.perf_counter() - start

print("\n" + "=" * 65)
print("FINAL LANGGRAPH TEST REPORT")
print("=" * 65)

print("\nQUESTION:", question)

print("\nINITIAL ANSWER:")
print(result.get("initial_answer", ""))

print("\nCLAIM VERIFICATION SUMMARY:")

for i, claim in enumerate(
    result.get("final_claim_results", []), 1
):
    print(f"{i}. {claim['claim']}")
    print("   Initial:", claim["initial_status"])
    print("   Corrective:", claim["corrective_status"])
    print("   Final:", claim["final_status"])

print("\nCORRECTIVE DECISION COUNTS:")

all_decisions = []

for item in result.get("corrective_results", {}).values():
    for entry in item.get("verification_history", []):
        all_decisions.append(
            entry.get("decision", "UNKNOWN")
        )

print(dict(Counter(all_decisions)))

print("\nFINAL STATUS:", result.get("final_status"))
print("FINAL ANSWER:", result.get("final_answer"))

print("\nEXECUTION TIME:", round(elapsed, 2), "seconds")
print("NODES EXECUTED:", len(result.get("execution_log", [])))

assert len(result.get("execution_log", [])) == 8
assert result.get("final_status") in {
    "ABSTAIN", "DRAFT_FOR_REVIEW"
}
assert result.get("final_answer", "").strip()

print("\nTEST EXECUTION: PASS")

if "UNKNOWN" in all_decisions:
    print("WARNING: Some decisions still have parsing issues.")

if result.get("final_status") == "ABSTAIN":
    print(
        "WARNING: Check whether this is an unnecessary "
        "abstention for an answerable question."
    )

if result.get("final_status") == "DRAFT_FOR_REVIEW":
    print(
        "WARNING: Citation integrity and semantic support "
        "still require validation."
    )

print("\nFINAL TEST COMPLETE")



Retrieval attempt 1/2
New passages: 4
Page: 1 | Decision: INSUFFICIENT
Page: 11 | Decision: INSUFFICIENT
Page: 3 | Decision: INSUFFICIENT
Page: 13 | Decision: INSUFFICIENT

Retrieval attempt 2/2
New passages: 0

Retrieval attempt 1/2
New passages: 4
Page: 9 | Decision: INSUFFICIENT
Page: 9 | Decision: INSUFFICIENT
Page: 10 | Decision: INSUFFICIENT
Page: 2 | Decision: INSUFFICIENT

Retrieval attempt 2/2
New passages: 0

Retrieval attempt 1/2
New passages: 4
Page: 2 | Decision: INSUFFICIENT
Page: 9 | Decision: INSUFFICIENT
Page: 9 | Decision: INSUFFICIENT
Page: 7 | Decision: INSUFFICIENT

Retrieval attempt 2/2
New passages: 0

Retrieval attempt 1/2
New passages: 4
Page: 2 | Decision: INSUFFICIENT
Page: 9 | Decision: INSUFFICIENT
Page: 9 | Decision: INSUFFICIENT
Page: 7 | Decision: INSUFFICIENT

Retrieval attempt 2/2
New passages: 0

FINAL LANGGRAPH TEST REPORT

QUESTION: What problem does eDySec aim to solve?

INITIAL ANSWER:
The problem that eDySec aims to solve is malicious package de

In [114]:
import inspect

print("=== CITATION VALIDATOR ===")
print(inspect.getsource(original_validate_single_citation))

print("\n=== CORRECTIVE VERIFICATION ===")
print(inspect.getsource(original_corrective_verify_claim))

=== CITATION VALIDATOR ===
def validate_single_citation(claim, document, llm):

    prompt = f"""
You are a strict scientific evidence validator.

CLAIM:
{claim}

EVIDENCE:
{document.page_content}

Does this single evidence passage directly support
ALL factual parts of the claim?

RULES:
1. Use ONLY the evidence provided.
2. Do not use external knowledge.
3. SUPPORTED: The passage directly supports the entire claim.
4. CONTRADICTED: The passage explicitly contradicts the claim.
5. INSUFFICIENT: The passage does not establish the full claim.
6. Missing evidence does not mean contradiction.
7. Return exactly ONE word:
SUPPORTED
CONTRADICTED
INSUFFICIENT

Decision:
"""

    raw_response = llm.invoke(prompt).strip()
    decision = raw_response.upper()

    allowed = {
        "SUPPORTED",
        "CONTRADICTED",
        "INSUFFICIENT"
    }

    if decision not in allowed:
        decision = "UNKNOWN"

    return {
        "decision": decision,
        "raw_response": raw_response
    }




In [115]:

# ============================================================
# STEP 10 — FINAL VERIFICATION PATCH
# No LLM calls in this cell
# ============================================================

import re

def parse_verification_decision(response):
    match = re.match(
        r"^\s*(?:Decision\s*:\s*)?"
        r"(SUPPORTED|CONTRADICTED|INSUFFICIENT)\b",
        str(response),
        flags=re.IGNORECASE
    )
    return match.group(1).upper() if match else "UNKNOWN"


def validate_single_citation(claim, document, llm):

    prompt = f"""
You are a scientific evidence verifier.

CLAIM:
{claim}

EVIDENCE:
{document.page_content}

Decide whether this evidence supports the claim.

SUPPORTED:
The passage explicitly supports or clearly entails
the complete claim, including faithful paraphrases.

CONTRADICTED:
The passage explicitly establishes the opposite.

INSUFFICIENT:
The passage does not establish the complete claim.

RULES:
- Use only the provided evidence.
- Do not use external knowledge.
- Do not confuse topic similarity with factual support.
- Partial support is insufficient.
- Missing evidence is not contradiction.

First line must be exactly one of:
SUPPORTED
CONTRADICTED
INSUFFICIENT
"""

    raw = str(llm.invoke(prompt)).strip()

    return {
        "decision": parse_verification_decision(raw),
        "raw_response": raw
    }


def corrective_verify_claim(
    claim, vectorstore, reranker, llm, max_retries=2
):
    history = []
    seen = set()
    supported = []
    contradicted = []

    for attempt in range(1, max_retries + 1):

        passages = retrieve_corrective_passages(
            claim, attempt, vectorstore, reranker,
            k=RETRIEVAL_K, top_n=TOP_N
        )

        new_count = 0

        for doc, score in passages:
            key = (
                doc.metadata.get("source"),
                doc.metadata.get("page"),
                doc.page_content
            )

            if key in seen:
                continue

            seen.add(key)
            new_count += 1

            validation = validate_single_citation(
                claim, doc, llm
            )

            entry = {
                "attempt": attempt,
                "page": doc.metadata.get(
                    "page_label", "Unknown"
                ),
                "decision": validation["decision"],
                "score": float(score),
                "raw_response": validation["raw_response"],
                "document": doc
            }

            history.append(entry)

            if entry["decision"] == "SUPPORTED":
                supported.append(entry)
            elif entry["decision"] == "CONTRADICTED":
                contradicted.append(entry)

        print(
            f"Attempt {attempt}: "
            f"{new_count} new passages"
        )

        if supported and not contradicted:
            break

        if new_count == 0:
            break

    status = (
        "CONFLICT_REVIEW" if contradicted
        else "PROVISIONALLY_SUPPORTED" if supported
        else "UNVERIFIED"
    )

    return {
        "claim": claim,
        "status": status,
        "attempts_used": attempt,
        "supported_passages": supported,
        "contradicted_passages": contradicted,
        "verification_history": history
    }


# Quick checks
assert parse_verification_decision(
    "SUPPORTED\nExplanation"
) == "SUPPORTED"

assert parse_verification_decision(
    "INSUFFICIENT\nExplanation"
) == "INSUFFICIENT"

assert parse_verification_decision(
    "CONTRADICTED\nExplanation"
) == "CONTRADICTED"

print("STEP 10 VERIFICATION PATCH: PASS")
print("No LLM calls made.")


STEP 10 VERIFICATION PATCH: PASS
No LLM calls made.


In [116]:
# ============================================================
# STEP 10 — CONNECT PATCH AND RECOMPILE LANGGRAPH
# ============================================================

from langgraph.graph import StateGraph, START, END

def corrective_retrieval_node(state: RAGState):
    results = {}

    for item in state.get("filtered_claims", []):
        if item.get("action") != "CORRECTIVE_RETRIEVAL":
            continue

        claim = item["claim"]

        results[claim] = corrective_verify_claim(
            claim=claim,
            vectorstore=vectorstore,
            reranker=reranker,
            llm=llm,
            max_retries=2
        )

    return {
        "corrective_results": results,
        "execution_log": state.get("execution_log", [])
        + [f"Corrective retrieval completed: {len(results)} claims"]
    }


nodes = {
    "retrieval": retrieval_node,
    "answer_generation": answer_generation_node,
    "claim_extraction": atomic_claim_extraction_node,
    "evidence_verification": evidence_verification_node,
    "claim_filtering": claim_filtering_node,
    "corrective_retrieval": corrective_retrieval_node,
    "final_claim_decisions": final_claim_decision_node,
    "final_answer": final_answer_node
}

workflow = StateGraph(RAGState)

for name, function in nodes.items():
    workflow.add_node(name, function)

order = list(nodes.keys())

workflow.add_edge(START, order[0])

for a, b in zip(order[:-1], order[1:]):
    workflow.add_edge(a, b)

workflow.add_edge(order[-1], END)

rag_graph = workflow.compile()

# Robust validation: check that our 8 custom nodes are correctly registered
assert set(nodes.keys()).issubset(set(rag_graph.nodes))

print("STEP 10 GRAPH RECOMPILED: PASS")
print("Updated corrective verification connected.")
print("No LLM calls made.")

STEP 10 GRAPH RECOMPILED: PASS
Updated corrective verification connected.
No LLM calls made.


In [117]:

# ============================================================
# STEP 10 — FINAL END-TO-END TEST
# ============================================================

import time

test_question = "What problem does eDySec aim to solve?"

start_time = time.time()

try:
    result = rag_graph.invoke({
        "question": test_question,
        "execution_log": []
    })

    elapsed = time.time() - start_time

    print("=" * 60)
    print("STEP 10 — FINAL END-TO-END TEST")
    print("=" * 60)

    print("\nQUESTION:")
    print(test_question)

    print("\nINITIAL ANSWER:")
    print(result.get("initial_answer", "Not available"))

    print("\nFINAL ANSWER:")
    print(result.get("final_answer", "Not available"))

    print("\nFINAL STATUS:")
    print(result.get("final_status", "Not available"))

    print("\nATOMIC CLAIMS:")
    for claim in result.get("atomic_claims", []):
        print("-", claim)

    print("\nCORRECTIVE VERIFICATION:")
    corrective = result.get("corrective_results", {})
    if isinstance(corrective, dict):
        for claim, info in corrective.items():
            print("Claim:", claim)
            print("Status:", info.get("status"))
            print("Attempts:", info.get("attempts_used"))
            print("Supported passages:",
                  len(info.get("supported_passages", [])))
            print("-" * 40)
    else:
        print(corrective)

    print("\nEXECUTION LOG:")
    for item in result.get("execution_log", []):
        print("-", item)

    print("\nEXECUTION TIME:", round(elapsed, 2), "seconds")

    print("\nTEST RESULT: GRAPH EXECUTED")

except Exception as error:
    print("TEST FAILED")
    print(type(error).__name__, ":", str(error))


Attempt 1: 4 new passages
Attempt 2: 0 new passages
Attempt 1: 4 new passages
Attempt 2: 0 new passages
STEP 10 — FINAL END-TO-END TEST

QUESTION:
What problem does eDySec aim to solve?

INITIAL ANSWER:
According to [E1], eDySec aims to solve the problem of malicious package detection, where false negatives allow threats to evade detection and false positives undermine the practical usability of the detector.

FINAL ANSWER:
False positives undermine the practical usability of the detector. [E1]

FINAL STATUS:
DRAFT_FOR_REVIEW

ATOMIC CLAIMS:
- {'claim': 'eDySec aims to solve the problem of malicious package detection.'}
- {'claim': 'False negatives allow threats to evade detection.'}
- {'claim': 'False positives undermine the practical usability of the detector.'}

CORRECTIVE VERIFICATION:
Claim: eDySec aims to solve the problem of malicious package detection.
Status: UNVERIFIED
Attempts: 2
Supported passages: 0
----------------------------------------
Claim: False negatives allow thre

In [118]:

# ============================================================
# STEP 10 — DIAGNOSE VERIFICATION DECISIONS
# No LLM calls
# ============================================================

corrective = result.get("corrective_results", {})

print("CORRECTIVE VERIFICATION DIAGNOSTIC")
print("=" * 60)

for claim, info in list(corrective.items())[:3]:
    print("\nCLAIM:", claim)
    print("STATUS:", info.get("status"))

    for entry in info.get("verification_history", [])[:2]:
        print("\nPage:", entry.get("page"))
        print("Decision:", entry.get("decision"))
        print("Raw response:",
              str(entry.get("raw_response", ""))[:700])

    print("-" * 60)

print("\nFINAL CLAIM DECISIONS")
for item in result.get("final_claim_results", []):
    print(
        item.get("final_status"),
        "|",
        item.get("claim")
    )


CORRECTIVE VERIFICATION DIAGNOSTIC

CLAIM: eDySec aims to solve the problem of malicious package detection.
STATUS: UNVERIFIED

Page: 1
Decision: INSUFFICIENT
Raw response: INSUFFICIENT

The passage mentions that eDySec "significantly outperforms the state-of-the-art frameworks" and provides specific metrics (e.g., 82% reduction in false positives, 79% reduction in false negatives, 3% improvement in accuracy). However, it does not explicitly state that eDySec aims to solve the problem of malicious package detection. The claim is a statement of purpose, while the evidence provides results and analysis.

Page: 2
Decision: INSUFFICIENT
Raw response: INSUFFICIENT

The passage describes a system architecture for detecting malicious packages, but it does not explicitly state that the system, eDySec, aims to solve the problem of malicious package detection. The claim is a general statement, while the evidence provides a specific implementation of a solution.
----------------------------------

In [119]:

# ============================================================
# STEP 10 — FAITHFUL EVIDENCE VERIFIER PATCH
# One targeted LLM test only
# ============================================================

import re

def validate_single_citation(claim, document, llm):

    prompt = f"""
You are checking whether a research-paper passage
supports one factual claim.

CLAIM:
{claim}

EVIDENCE:
{document.page_content}

DECISION RULES:
- SUPPORTED: The evidence directly states the claim,
  faithfully paraphrases it, or clearly entails it.
- CONTRADICTED: The evidence explicitly conflicts
  with the claim.
- INSUFFICIENT: The evidence does not establish the
  claim or leaves a material detail unsupported.

IMPORTANT:
- Exact wording is NOT required.
- A specific result can support a broader claim.
  Example: "reduces false negatives by 79%" supports
  "addresses false negatives".
- Do not require the paper to use the words "aims to"
  if its stated objective clearly establishes the aim.
- Do not introduce outside facts.
- Reject unsupported numbers, comparisons, causal
  explanations, and additional details.
- Missing evidence is not contradiction.

Return exactly:
DECISION: SUPPORTED or CONTRADICTED or INSUFFICIENT
REASON: One short explanation.

DECISION:
"""

    raw = str(llm.invoke(prompt)).strip()

    match = re.search(
        r"(?:DECISION\s*:\s*)?"
        r"\b(SUPPORTED|CONTRADICTED|INSUFFICIENT)\b",
        raw.upper()
    )

    decision = match.group(1) if match else "UNKNOWN"

    return {
        "decision": decision,
        "raw_response": raw
    }


# Test the claim previously rejected by the verifier.
claim = "eDySec addresses the issue of false negatives."

evidence = next(
    doc for doc in chunks
    if "false negatives by 79%" in doc.page_content
)

test = validate_single_citation(claim, evidence, llm)

print("CLAIM:", claim)
print("PAGE:", evidence.metadata.get("page_label"))
print("DECISION:", test["decision"])
print("RAW RESPONSE:", test["raw_response"])


CLAIM: eDySec addresses the issue of false negatives.
PAGE: 1
DECISION: SUPPORTED
RAW RESPONSE: DECISION: SUPPORTED
REASON: The evidence states that eDySec "halves false negatives by 79%".


In [120]:

# ============================================================
# STEP 10 — NEGATIVE VERIFICATION TEST
# ============================================================

false_claim = (
    "eDySec completely eliminates all false negatives "
    "and guarantees 100% detection accuracy."
)

# Reuse the same evidence passage from the positive test.
negative_test = validate_single_citation(
    false_claim,
    evidence,
    llm
)

print("CLAIM:", false_claim)
print("PAGE:", evidence.metadata.get("page_label"))
print("DECISION:", negative_test["decision"])
print("RAW RESPONSE:", negative_test["raw_response"])

if negative_test["decision"] == "INSUFFICIENT":
    print("\nNEGATIVE TEST: PASS")
elif negative_test["decision"] == "CONTRADICTED":
    print("\nNEGATIVE TEST: REJECTED — inspect contradiction reasoning")
else:
    print("\nNEGATIVE TEST: REVIEW REQUIRED")


CLAIM: eDySec completely eliminates all false negatives and guarantees 100% detection accuracy.
PAGE: 1
DECISION: INSUFFICIENT
RAW RESPONSE: DECISION: INSUFFICIENT
REASON: The evidence shows significant improvement in detection accuracy, but does not guarantee 100% detection accuracy, which is the claim.

NEGATIVE TEST: PASS


In [121]:

# ============================================================
# STEP 10 — CORRECTIVE CITATION MAPPING PATCH
from langgraph.graph import StateGraph, START, END

def final_answer_node(state: RAGState):

    final_claim_results = state.get("final_claim_results", [])
    retrieved_docs = state.get("retrieved_docs", [])

    # Initial evidence registry
    evidence_registry = {}

    for i, item in enumerate(retrieved_docs, start=1):
        doc = item[0] if isinstance(item, tuple) else item

        if hasattr(doc, "page_content"):
            evidence_registry[f"E{i}"] = doc

    # Corrective evidence registry
    corrective_registry = {}
    cited_sentences = []
    accepted_claims = []

    def document_key(doc):
        return (
            doc.metadata.get("source"),
            doc.metadata.get("page"),
            doc.page_content
        )

    corrective_ids = {}

    for item in final_claim_results:

        if item.get("final_status") != "PROVISIONALLY_SUPPORTED":
            continue

        claim = item["claim"]
        valid_ids = []

        # 1. Validate initial evidence IDs.
        for eid in item.get("initial_supported_evidence", []):
            if eid in evidence_registry:
                valid_ids.append(eid)

        # 2. Register corrective evidence documents.
        for passage in item.get(
            "corrective_supported_passages", []
        ):
            doc = passage.get("document")

            if doc is None or not hasattr(doc, "page_content"):
                continue

            key = document_key(doc)

            if key not in corrective_ids:
                cid = f"C{len(corrective_ids) + 1}"
                corrective_ids[key] = cid
                corrective_registry[cid] = doc

            valid_ids.append(corrective_ids[key])

        valid_ids = list(dict.fromkeys(valid_ids))

        # Never output an uncited claim.
        if not valid_ids:
            continue

        citation_text = " ".join(
            f"[{eid}]" for eid in valid_ids
        )

        cited_sentences.append(
            f"{claim} {citation_text}"
        )

        accepted_claims.append({
            "claim": claim,
            "evidence_ids": valid_ids
        })

    if cited_sentences:
        final_answer = "\n".join(cited_sentences)
        final_status = "DRAFT_FOR_REVIEW"
    else:
        final_answer = (
            "Insufficient verified evidence to provide "
            "a reliable answer to this question."
        )
        final_status = "ABSTAIN"

    return {
        "verified_claims": accepted_claims,
        "final_answer": final_answer,
        "final_status": final_status,
        "execution_log": state.get("execution_log", [])
        + [f"Final answer decision completed: {final_status}"]
    }


# Recompile graph using updated final_answer_node.
workflow = StateGraph(RAGState)

nodes = {
    "retrieval": retrieval_node,
    "answer_generation": answer_generation_node,
    "claim_extraction": atomic_claim_extraction_node,
    "evidence_verification": evidence_verification_node,
    "claim_filtering": claim_filtering_node,
    "corrective_retrieval": corrective_retrieval_node,
    "final_claim_decisions": final_claim_decision_node,
    "final_answer": final_answer_node
}

for name, fn in nodes.items():
    workflow.add_node(name, fn)

order = list(nodes)

workflow.add_edge(START, order[0])

for current, next_node in zip(order[:-1], order[1:]):
    workflow.add_edge(current, next_node)

workflow.add_edge(order[-1], END)

rag_graph = workflow.compile()

print("CITATION MAPPING PATCH: LOADED")
print("GRAPH RECOMPILED: PASS")
print("No LLM calls made.")

# Targeted citation test using the previously tested
# supported claim and its evidence document.
# This fixture checks citation plumbing, not verifier accuracy.
test_state = {
    "retrieved_docs": [],
    "final_claim_results": [{
        "claim": "eDySec addresses the issue of false negatives.",
        "final_status": "PROVISIONALLY_SUPPORTED",
        "initial_supported_evidence": [],
        "corrective_supported_passages": [{
            "document": evidence,
            "decision": "SUPPORTED"
        }]
    }],
    "execution_log": []
}

citation_test = final_answer_node(test_state)

print("\nCITATION TEST:")
print(citation_test["final_answer"])
print("Status:", citation_test["final_status"])

assert "[C1]" in citation_test["final_answer"]
assert citation_test["final_status"] == "DRAFT_FOR_REVIEW"

print("\nCORRECTIVE CITATION MAPPING: PASS")


CITATION MAPPING PATCH: LOADED
GRAPH RECOMPILED: PASS
No LLM calls made.

CITATION TEST:
eDySec addresses the issue of false negatives. [C1]
Status: DRAFT_FOR_REVIEW

CORRECTIVE CITATION MAPPING: PASS


# Step 11: Map Reduce Summarization

In [123]:

# STEP 11 — CELL 1: INSPECT EXISTING RAG WORKFLOW


# Inspect the existing LangGraph nodes.
print("EXISTING LANGGRAPH NODES:")

for node_name in rag_graph.nodes:
    print("-", node_name)

# Inspect the available RAG state fields.
print("\nEXISTING RAG STATE FIELDS:")

for field_name in RAGState.__annotations__:
    print("-", field_name)

# Confirm the graph and required nodes exist.
required_nodes = [
    "retrieval",
    "answer_generation",
    "claim_extraction",
    "evidence_verification",
    "claim_filtering",
    "corrective_retrieval",
    "final_claim_decisions",
    "final_answer"
]

missing_nodes = [
    node for node in required_nodes
    if node not in rag_graph.nodes
]

assert not missing_nodes, (
    f"Missing workflow nodes: {missing_nodes}"
)

print("\nWORKFLOW STRUCTURE CHECK: PASS")



EXISTING LANGGRAPH NODES:
- __start__
- retrieval
- answer_generation
- claim_extraction
- evidence_verification
- claim_filtering
- corrective_retrieval
- final_claim_decisions
- final_answer

EXISTING RAG STATE FIELDS:
- question
- retrieved_docs
- initial_answer
- atomic_claims
- verification_results
- validated_citations
- filtered_claims
- corrective_results
- final_claim_results
- verified_claims
- final_answer
- final_status
- execution_log

WORKFLOW STRUCTURE CHECK: PASS


In [124]:
# PREPARE MAP-REDUCE DOCUMENT CHUNKS


from langchain_core.documents import Document

def prepare_summary_chunks(retrieved_docs):
    """
    Prepare retrieved documents for Map-Reduce summarization
    while preserving source metadata and evidence IDs.
    """

    prepared_chunks = []

    for index, item in enumerate(retrieved_docs, start=1):

        # Support both Document and (Document, score) formats.
        doc = item[0] if isinstance(item, tuple) else item

        if not isinstance(doc, Document):
            continue

        content = doc.page_content.strip()

        # Skip empty document chunks.
        if not content:
            continue

        metadata = dict(doc.metadata or {})

        prepared_chunks.append({
            "evidence_id": f"E{index}",
            "content": content,
            "source": metadata.get("source", "unknown"),
            "page": metadata.get("page", "unknown"),
            "metadata": metadata,
            "document": doc
        })

    return prepared_chunks


# Test using a synthetic document.
# This test does not claim that retrieval was executed.
test_docs = [
    Document(
        page_content=(
            "Retrieval-Augmented Generation uses retrieved "
            "documents to provide context for language models."
        ),
        metadata={
            "source": "synthetic_test_document",
            "page": 1
        }
    )
]

prepared_test = prepare_summary_chunks(test_docs)

assert len(prepared_test) == 1
assert prepared_test[0]["evidence_id"] == "E1"
assert prepared_test[0]["source"] == "synthetic_test_document"
assert prepared_test[0]["page"] == 1

print("MAP-REDUCE CHUNK PREPARATION: PASS")
print("Prepared chunks:", len(prepared_test))
print("Evidence ID:", prepared_test[0]["evidence_id"])
print("Source:", prepared_test[0]["source"])
print("Page:", prepared_test[0]["page"])



MAP-REDUCE CHUNK PREPARATION: PASS
Prepared chunks: 1
Evidence ID: E1
Source: synthetic_test_document
Page: 1


In [125]:
# MAP SUMMARIZATION


from langchain_core.prompts import ChatPromptTemplate

map_prompt = ChatPromptTemplate.from_messages([
    ("system", """
You are a research-paper summarization assistant.

Summarize only the information explicitly present in the
provided academic document chunk.

Rules:
1. Preserve important research methods, datasets, and results.
2. Preserve numerical values and experimental conditions.
3. Do not invent facts or add external knowledge.
4. If information is incomplete, state the limitation.
5. Keep the summary concise and academically accurate.
6. Do not invent citations or page numbers.
"""),
    ("human", """
Evidence ID: {evidence_id}
Source: {source}
Page: {page}

Document chunk:
{content}

Write a factual summary of this chunk.
""")
])


def map_summarize_chunks(prepared_chunks, llm):
    """
    Summarize each prepared document chunk independently
    while preserving its original evidence metadata.
    """

    mapped_summaries = []

    for chunk in prepared_chunks:

        messages = map_prompt.format_messages(
            evidence_id=chunk["evidence_id"],
            source=chunk["source"],
            page=chunk["page"],
            content=chunk["content"]
        )

        # One LLM call per document chunk.
        response = llm.invoke(messages)

        summary = (
            response.content
            if isinstance(response.content, str)
            else str(response.content)
        )

        mapped_summaries.append({
            "evidence_id": chunk["evidence_id"],
            "summary": summary.strip(),
            "source": chunk["source"],
            "page": chunk["page"],
            "document": chunk["document"]
        })

    return mapped_summaries


# Validate function and prompt without running the LLM.
assert callable(map_summarize_chunks)
assert "content" in map_prompt.input_variables
assert "evidence_id" in map_prompt.input_variables

print("MAP SUMMARIZATION MODULE: LOADED")
print("Prompt validation: PASS")



MAP SUMMARIZATION MODULE: LOADED
Prompt validation: PASS


In [126]:
#TEST MAP SUMMARIZATION


from langchain_ollama import ChatOllama

# Reuse the existing Ollama server and model.
map_llm = ChatOllama(
    model="llama3.1:8b",
    temperature=0
)

# Use an actual research-paper chunk from the existing
# FAISS vector database, rather than synthetic test data.
test_retrieved_docs = vectorstore.similarity_search(
    "What is the main objective and methodology of eDySec?",
    k=1
)

assert len(test_retrieved_docs) > 0, (
    "No research-paper chunks were retrieved."
)

# Prepare the retrieved chunk with its evidence metadata.
real_prepared_chunks = prepare_summary_chunks(
    test_retrieved_docs
)

assert real_prepared_chunks, (
    "No valid chunks were prepared."
)

# Generate a summary using the Ollama LLM.
mapped_test = map_summarize_chunks(
    real_prepared_chunks,
    map_llm
)

assert len(mapped_test) == 1
assert mapped_test[0]["summary"]
assert mapped_test[0]["evidence_id"] == real_prepared_chunks[0]["evidence_id"]
assert mapped_test[0]["source"] == real_prepared_chunks[0]["source"]
assert mapped_test[0]["page"] == real_prepared_chunks[0]["page"]

print("MAP SUMMARIZATION TEST: PASS")
print("\nEvidence ID:", mapped_test[0]["evidence_id"])
print("Source:", mapped_test[0]["source"])
print("Page:", mapped_test[0]["page"])
print("\nGENERATED SUMMARY:")
print(mapped_test[0]["summary"])
print("\nNote: Factual grounding still requires verification.")


MAP SUMMARIZATION TEST: PASS

Evidence ID: E1
Source: dl based detecting malicious packages in pypl.pdf
Page: 5

GENERATED SUMMARY:
Here is a factual summary of the provided document chunk:

The eDySec framework assesses the relationship between selected features and model behavior using the formula C=F ∗∩Feat(R 𝑒𝑥𝑝). The framework's workflow is presented in Algorithm 1. The experimental design of eDySec involves a computational environment with a 13th Gen Intel Core i9-13900K processor, 128 GB RAM, and an NVIDIA RTX A6000 GPU with 48 GB memory. The experiments were conducted using both dedicated hardware and a controlled software environment.

Note: Factual grounding still requires verification.


In [127]:

# REDUCE SUMMARIZATION


from langchain_core.prompts import ChatPromptTemplate

reduce_prompt = ChatPromptTemplate.from_messages([
    ("system", """
You are an academic research summarization assistant.

Combine the provided chunk summaries into one coherent
research-paper summary.

Rules:
1. Use only the supplied chunk summaries.
2. Preserve methods, datasets, results, and limitations.
3. Preserve numerical values and experimental conditions.
4. Do not invent facts or unsupported conclusions.
5. Remove repeated information.
6. Do not silently resolve conflicting statements.
7. Retain evidence IDs beside factual statements.
8. If information is insufficient, state the limitation.
9. Do not invent citations or page numbers.
"""),
    ("human", """
Research-paper chunk summaries:

{mapped_context}

Produce a concise, evidence-referenced summary with:
- Research objective
- Methodology
- Experimental setup
- Key findings
- Limitations

Only include sections supported by the supplied context.
""")
])


def reduce_summaries(mapped_summaries, llm):
    """
    Combine mapped summaries while preserving their
    evidence references and original source metadata.
    """

    if not mapped_summaries:
        return {
            "summary": "",
            "evidence_registry": {},
            "status": "NO_EVIDENCE"
        }

    evidence_registry = {}
    context_parts = []

    for item in mapped_summaries:
        eid = item["evidence_id"]

        evidence_registry[eid] = {
            "source": item["source"],
            "page": item["page"],
            "document": item["document"]
        }

        context_parts.append(
            f"[{eid}]\n"
            f"Source: {item['source']}\n"
            f"Page: {item['page']}\n"
            f"Summary: {item['summary']}"
        )

    mapped_context = "\n\n".join(context_parts)

    messages = reduce_prompt.format_messages(
        mapped_context=mapped_context
    )

    # One LLM call for the supplied summary batch.
    response = llm.invoke(messages)

    combined_summary = (
        response.content
        if isinstance(response.content, str)
        else str(response.content)
    )

    return {
        "summary": combined_summary.strip(),
        "evidence_registry": evidence_registry,
        "status": "DRAFT_FOR_REVIEW"
    }


# Validate without calling the LLM.
assert callable(reduce_summaries)
assert "mapped_context" in reduce_prompt.input_variables

print("REDUCE SUMMARIZATION MODULE: LOADED")
print("Prompt validation: PASS")



REDUCE SUMMARIZATION MODULE: LOADED
Prompt validation: PASS


In [128]:

#MULTI-CHUNK MAP-REDUCE TEST


import re

# Retrieve multiple chunks from the existing FAISS database.
summary_test_docs = vectorstore.similarity_search(
    "Explain the objective, methodology, experimental setup "
    "and findings of the eDySec framework.",
    k=3
)

# Prepare chunks with source metadata.
summary_chunks = prepare_summary_chunks(summary_test_docs)

assert len(summary_chunks) == 3, (
    f"Expected 3 valid chunks, got {len(summary_chunks)}"
)

# MAP: Summarize each chunk independently.
mapped_results = map_summarize_chunks(
    summary_chunks,
    map_llm
)

assert len(mapped_results) == len(summary_chunks)

print("MAP STAGE: PASS")
print("Mapped summaries:", len(mapped_results))

# REDUCE: Combine the individual summaries.
reduced_result = reduce_summaries(
    mapped_results,
    map_llm
)

assert reduced_result["summary"].strip()
assert reduced_result["status"] == "DRAFT_FOR_REVIEW"

# Check the evidence registry.
registry = reduced_result["evidence_registry"]

expected_ids = {
    item["evidence_id"] for item in mapped_results
}

assert set(registry) == expected_ids

# Inspect citation markers used by the LLM.
cited_ids = set(
    re.findall(r"\[(E\d+)\]", reduced_result["summary"])
)

unknown_ids = cited_ids - set(registry)

assert not unknown_ids, (
    f"Unknown evidence IDs in summary: {unknown_ids}"
)

print("\nREDUCE STAGE: PASS")
print("Registered evidence IDs:", sorted(registry))
print("Cited evidence IDs:", sorted(cited_ids))

print("\nCOMBINED RESEARCH SUMMARY:\n")
print(reduced_result["summary"])

print("\nFINAL STATUS:", reduced_result["status"])
print(
    "Note: Citation IDs were checked against the registry; "
    "factual support has not yet been verified."
)


MAP STAGE: PASS
Mapped summaries: 3

REDUCE STAGE: PASS
Registered evidence IDs: ['E1', 'E2', 'E3']
Cited evidence IDs: []

COMBINED RESEARCH SUMMARY:

Here is a concise, evidence-referenced summary of the research paper:

**Research Objective:**
The eDySec framework aims to detect malicious packages in the PyPI ecosystem using a deep learning-based approach (E2).

**Methodology:**
The eDySec framework assesses the relationship between selected features and model behavior using the formula C=F ∗∩Feat(R 𝑒𝑥𝑝) (E1). The framework's workflow is presented in Algorithm 1 (E1). The framework consists of four phases: data preparation, feature selection, deep learning model selection and evaluation, and stability and explainability analysis (E2).

**Experimental Setup:**
The experiments were conducted using a computational environment with a 13th Gen Intel Core i9-13900K processor, 128 GB RAM, and an NVIDIA RTX A6000 GPU with 48 GB memory (E1). The experiments were conducted using both dedicate

In [129]:

#MAP-REDUCE CITATION FORMAT PATCH


import re

def validate_summary_citations(reduced_result):

    summary = reduced_result["summary"]
    registry = reduced_result["evidence_registry"]

    # Normalize citations such as (E1) to [E1].
    normalized = re.sub(
        r"\((E\d+)\)",
        r"[\1]",
        summary
    )

    # Extract normalized citation IDs.
    cited_ids = set(
        re.findall(r"\[(E\d+)\]", normalized)
    )

    # Check citations against the original registry.
    unknown_ids = cited_ids - set(registry)

    if unknown_ids:
        raise ValueError(
            f"Unknown evidence IDs: {sorted(unknown_ids)}"
        )

    if not cited_ids:
        raise ValueError(
            "No evidence citations found in summary."
        )

    return {
        **reduced_result,
        "summary": normalized,
        "cited_ids": sorted(cited_ids),
        "citation_format_status": "PASS",
        "status": "DRAFT_FOR_REVIEW"
    }


# Validate the existing Reduce output.
validated_summary = validate_summary_citations(
    reduced_result
)

print("CITATION FORMAT VALIDATION: PASS")
print("Cited IDs:", validated_summary["cited_ids"])
print("Status:", validated_summary["status"])

print("\nNORMALIZED SUMMARY:\n")
print(validated_summary["summary"])

print(
    "\nNote: Citation format is valid, "
    "but factual support is not yet verified."
)


CITATION FORMAT VALIDATION: PASS
Cited IDs: ['E1', 'E2']
Status: DRAFT_FOR_REVIEW

NORMALIZED SUMMARY:

Here is a concise, evidence-referenced summary of the research paper:

**Research Objective:**
The eDySec framework aims to detect malicious packages in the PyPI ecosystem using a deep learning-based approach [E2].

**Methodology:**
The eDySec framework assesses the relationship between selected features and model behavior using the formula C=F ∗∩Feat(R 𝑒𝑥𝑝) [E1]. The framework's workflow is presented in Algorithm 1 [E1]. The framework consists of four phases: data preparation, feature selection, deep learning model selection and evaluation, and stability and explainability analysis [E2].

**Experimental Setup:**
The experiments were conducted using a computational environment with a 13th Gen Intel Core i9-13900K processor, 128 GB RAM, and an NVIDIA RTX A6000 GPU with 48 GB memory [E1]. The experiments were conducted using both dedicated hardware and a controlled software environment

In [131]:

#PREPARE SUMMARY CLAIMS



import re

def prepare_summary_verification(validated_summary):
    """
    Extract cited factual statements and connect them
    to original source documents for later verification.
    """

    summary = validated_summary["summary"]
    registry = validated_summary["evidence_registry"]

    verification_items = []

    # Remove Markdown headings and introductory lines.
    lines = summary.splitlines()

    for line in lines:
        line = line.strip()

        if not line or line.startswith("**"):
            continue

        # Extract registered evidence citations.
        evidence_ids = re.findall(
            r"\[(E\d+)\]", line
        )

        if not evidence_ids:
            continue

        # Split at sentence boundaries after citations.
        sentences = re.split(
            r"(?<=\])\.\s+",
            line
        )

        for sentence in sentences:
            sentence = sentence.strip()

            ids = list(dict.fromkeys(
                re.findall(r"\[(E\d+)\]", sentence)
            ))

            if not ids:
                continue

            claim = re.sub(
                r"\[E\d+\]", "", sentence
            ).strip()

            claim = claim.rstrip(".").strip() + "."

            evidence_docs = []

            for eid in ids:
                if eid in registry:
                    evidence_docs.append({
                        "evidence_id": eid,
                        "source": registry[eid]["source"],
                        "page": registry[eid]["page"],
                        "content": registry[eid]["document"].page_content
                    })

            if evidence_docs:
                verification_items.append({
                    "claim": claim,
                    "evidence_ids": ids,
                    "evidence": evidence_docs,
                    "verification_status": "PENDING"
                })

    return verification_items


summary_verification_items = prepare_summary_verification(
    validated_summary
)

assert summary_verification_items, (
    "No cited claims were extracted."
)

print("SUMMARY CLAIM PREPARATION: PASS")
print("Claims prepared:", len(summary_verification_items))

for i, item in enumerate(summary_verification_items, start=1):
    print(f"\nCLAIM {i}:")
    print(item["claim"])
    print("Evidence IDs:", item["evidence_ids"])
    print("Status:", item["verification_status"])



SUMMARY CLAIM PREPARATION: PASS
Claims prepared: 6

CLAIM 1:
The eDySec framework aims to detect malicious packages in the PyPI ecosystem using a deep learning-based approach.
Evidence IDs: ['E2']
Status: PENDING

CLAIM 2:
The eDySec framework assesses the relationship between selected features and model behavior using the formula C=F ∗∩Feat(R 𝑒𝑥𝑝).
Evidence IDs: ['E1']
Status: PENDING

CLAIM 3:
The framework's workflow is presented in Algorithm 1.
Evidence IDs: ['E1']
Status: PENDING

CLAIM 4:
The framework consists of four phases: data preparation, feature selection, deep learning model selection and evaluation, and stability and explainability analysis.
Evidence IDs: ['E2']
Status: PENDING

CLAIM 5:
The experiments were conducted using a computational environment with a 13th Gen Intel Core i9-13900K processor, 128 GB RAM, and an NVIDIA RTX A6000 GPU with 48 GB memory.
Evidence IDs: ['E1']
Status: PENDING

CLAIM 6:
The experiments were conducted using both dedicated hardware and a co

In [132]:
# SUMMARY CLAIM-LEVEL VERIFICATION


import json
import re
from langchain_core.prompts import ChatPromptTemplate

summary_verification_prompt = ChatPromptTemplate.from_messages([
    ("system", """
You are a strict academic evidence verifier.

Compare the CLAIM against the ORIGINAL SOURCE EVIDENCE.

Classify the claim as:
- SUPPORTED: All factual details are supported by evidence.
- CONTRADICTED: Evidence explicitly conflicts with the claim.
- INSUFFICIENT_EVIDENCE: Evidence does not establish the claim.

Rules:
1. Use only the supplied original evidence.
2. Do not use external knowledge.
3. Do not assume a citation guarantees correctness.
4. Check numbers, formulas, names, and experimental details.
5. If a claim contains multiple facts, all must be supported.
6. Treat corrupted or ambiguous formulas conservatively.
7. Return only valid JSON, without Markdown fences.

JSON format:
{{
  "label": "SUPPORTED",
  "reason": "Short evidence-based explanation"
}}
"""),
    ("human", """
CLAIM:
{claim}

ORIGINAL SOURCE EVIDENCE:
{evidence}

Verify the claim.
""")
])


def verify_summary_claims(items, llm):
    results = []

    for index, item in enumerate(items, start=1):

        evidence_text = "\n\n".join(
            f"[{doc['evidence_id']}]\n"
            f"Source: {doc['source']}\n"
            f"Page: {doc['page']}\n"
            f"Original text:\n{doc['content']}"
            for doc in item["evidence"]
        )

        messages = summary_verification_prompt.format_messages(
            claim=item["claim"],
            evidence=evidence_text
        )

        try:
            response = llm.invoke(messages)
            raw = (
                response.content
                if isinstance(response.content, str)
                else str(response.content)
            )

            # Accept plain JSON or fenced JSON.
            cleaned = re.sub(
                r"^```(?:json)?\s*|\s*```$",
                "",
                raw.strip(),
                flags=re.IGNORECASE
            ).strip()

            parsed = json.loads(cleaned)

            label = parsed.get("label", "")
            reason = parsed.get("reason", "")

            valid_labels = {
                "SUPPORTED",
                "CONTRADICTED",
                "INSUFFICIENT_EVIDENCE"
            }

            if label not in valid_labels:
                raise ValueError("Invalid verification label")

            if not isinstance(reason, str) or not reason.strip():
                raise ValueError("Missing verification reason")

        except (json.JSONDecodeError, ValueError) as error:
            label = "VERIFICATION_ERROR"
            reason = str(error)

        result = {
            **item,
            "verification_status": label,
            "verification_reason": reason
        }

        results.append(result)

        print(f"\nCLAIM {index}: {label}")
        print("Claim:", item["claim"])
        print("Reason:", reason)

    return results


# Run verification on the six prepared claims.
summary_verification_results = verify_summary_claims(
    summary_verification_items,
    map_llm
)

assert len(summary_verification_results) == len(
    summary_verification_items
)

print("\nSUMMARY VERIFICATION: COMPLETED")
print("Claims checked:", len(summary_verification_results))

for status in [
    "SUPPORTED",
    "CONTRADICTED",
    "INSUFFICIENT_EVIDENCE",
    "VERIFICATION_ERROR"
]:
    count = sum(
        item["verification_status"] == status
        for item in summary_verification_results
    )
    print(f"{status}: {count}")

print(
    "\nNote: These are automated verifier judgments, "
    "not manually established ground truth."
)



CLAIM 1: SUPPORTED
Claim: The eDySec framework aims to detect malicious packages in the PyPI ecosystem using a deep learning-based approach.
Reason: The original source evidence explicitly states that the eDySec framework uses a deep learning-based approach to detect malicious packages in the PyPI ecosystem.

CLAIM 2: SUPPORTED
Claim: The eDySec framework assesses the relationship between selected features and model behavior using the formula C=F ∗∩Feat(R 𝑒𝑥𝑝).
Reason: The original source evidence (E1) explicitly mentions the formula C=F ∗∩Feat(R 𝑒𝑥𝑝) on page 5, which matches the claim.

CLAIM 3: SUPPORTED
Claim: The framework's workflow is presented in Algorithm 1.
Reason: The original source evidence (page 5) explicitly mentions that Algorithm 1 presents the pseudo-code for the proposed eDySec framework workflow, supporting the claim.

CLAIM 4: SUPPORTED
Claim: The framework consists of four phases: data preparation, feature selection, deep learning model selection and evaluation, a

In [133]:
#VERIFIED SUMMARY FILTERING


import re
from collections import Counter

def filter_verified_summary(verification_results):
    accepted_claims = []
    review_claims = []

    for index, item in enumerate(verification_results, start=1):
        status = item["verification_status"]
        claim = item["claim"]
        evidence_ids = item["evidence_ids"]

        # Mathematical expressions require extra inspection.
        formula_review = bool(
            re.search(r"[=∗∩∑∫√]", claim)
        )

        record = {
            "claim_id": f"SC{index}",
            "claim": claim,
            "evidence_ids": evidence_ids,
            "verification_status": status,
            "verification_reason": item.get(
                "verification_reason", ""
            ),
            "manual_formula_review": formula_review
        }

        if status == "SUPPORTED":
            accepted_claims.append(record)
        else:
            review_claims.append(record)

    # Build a citation-preserving draft.
    summary_lines = []

    for item in accepted_claims:
        citations = " ".join(
            f"[{eid}]" for eid in item["evidence_ids"]
        )

        summary_lines.append(
            f"- {item['claim']} {citations}"
        )

    verified_draft = "\n".join(summary_lines)

    manual_review_count = sum(
        item["manual_formula_review"]
        for item in accepted_claims
    )

    return {
        "verified_summary_draft": verified_draft,
        "accepted_claims": accepted_claims,
        "review_claims": review_claims,
        "accepted_count": len(accepted_claims),
        "review_count": len(review_claims),
        "manual_formula_review_count": manual_review_count,
        "status": "DRAFT_FOR_REVIEW"
    }


filtered_summary_result = filter_verified_summary(
    summary_verification_results
)

assert (
    filtered_summary_result["accepted_count"]
    + filtered_summary_result["review_count"]
    == len(summary_verification_results)
)

print("SUMMARY FILTERING: PASS")
print(
    "Accepted claims:",
    filtered_summary_result["accepted_count"]
)
print(
    "Claims requiring evidence review:",
    filtered_summary_result["review_count"]
)
print(
    "Formula claims requiring manual review:",
    filtered_summary_result["manual_formula_review_count"]
)

print("\nVERIFIED SUMMARY DRAFT:\n")
print(filtered_summary_result["verified_summary_draft"])

print("\nSTATUS:", filtered_summary_result["status"])
print(
    "Note: Accepted means supported by the automated "
    "verifier, not independently proven factual."
)


SUMMARY FILTERING: PASS
Accepted claims: 6
Claims requiring evidence review: 0
Formula claims requiring manual review: 1

VERIFIED SUMMARY DRAFT:

- The eDySec framework aims to detect malicious packages in the PyPI ecosystem using a deep learning-based approach. [E2]
- The eDySec framework assesses the relationship between selected features and model behavior using the formula C=F ∗∩Feat(R 𝑒𝑥𝑝). [E1]
- The framework's workflow is presented in Algorithm 1. [E1]
- The framework consists of four phases: data preparation, feature selection, deep learning model selection and evaluation, and stability and explainability analysis. [E2]
- The experiments were conducted using a computational environment with a 13th Gen Intel Core i9-13900K processor, 128 GB RAM, and an NVIDIA RTX A6000 GPU with 48 GB memory. [E1]
- The experiments were conducted using both dedicated hardware and a controlled software environment. [E1]

STATUS: DRAFT_FOR_REVIEW
Note: Accepted means supported by the automated ve

In [134]:
#Add Map-Reduce to LangGraph

from typing import TypedDict, Any
from langgraph.graph import StateGraph, START, END

class SummaryState(TypedDict, total=False):
    question: str
    retrieved_docs: list
    prepared_chunks: list
    mapped_summaries: list
    reduced_result: dict
    execution_log: list[str]

def summary_retrieval_node(state: SummaryState):
    docs = vectorstore.similarity_search(
        state["question"], k=3
    )
    return {
        "retrieved_docs": docs,
        "execution_log": ["summary_retrieval"]
    }

def summary_map_node(state: SummaryState):
    chunks = prepare_summary_chunks(
        state["retrieved_docs"]
    )
    mapped = map_summarize_chunks(chunks, map_llm)
    return {
        "prepared_chunks": chunks,
        "mapped_summaries": mapped,
        "execution_log": state["execution_log"] + ["map"]
    }

def summary_reduce_node(state: SummaryState):
    result = reduce_summaries(
        state["mapped_summaries"], map_llm
    )
    return {
        "reduced_result": result,
        "execution_log": state["execution_log"] + ["reduce"]
    }

summary_builder = StateGraph(SummaryState)

summary_builder.add_node("retrieve", summary_retrieval_node)
summary_builder.add_node("map", summary_map_node)
summary_builder.add_node("reduce", summary_reduce_node)

summary_builder.add_edge(START, "retrieve")
summary_builder.add_edge("retrieve", "map")
summary_builder.add_edge("map", "reduce")
summary_builder.add_edge("reduce", END)

summary_graph = summary_builder.compile()

print("MAP-REDUCE LANGGRAPH: COMPILED")
print("Nodes:", list(summary_graph.nodes))
print("Existing RAG graph: UNCHANGED")


MAP-REDUCE LANGGRAPH: COMPILED
Nodes: ['__start__', 'retrieve', 'map', 'reduce']
Existing RAG graph: UNCHANGED


In [135]:
#End-to-End Map-Reduce Test
summary_graph_result = summary_graph.invoke({
    "question": (
        "Explain the eDySec research objective, "
        "methodology, experimental setup and findings."
    ),
    "execution_log": []
})

assert summary_graph_result["reduced_result"]["summary"]

assert summary_graph_result["execution_log"] == [
    "summary_retrieval", "map", "reduce"
]

print("END-TO-END MAP-REDUCE: PASS")
print("Execution:", summary_graph_result["execution_log"])
print("Chunks:", len(summary_graph_result["prepared_chunks"]))

print("\nSUMMARY:\n")
print(summary_graph_result["reduced_result"]["summary"])


END-TO-END MAP-REDUCE: PASS
Execution: ['summary_retrieval', 'map', 'reduce']
Chunks: 3

SUMMARY:

Here is a concise summary of the research paper:

**Research Objective:**
The eDySec framework aims to detect malicious packages in the PyPI ecosystem using a deep learning-based approach (E2).

**Methodology:**
The eDySec framework assesses the relationship between selected features and model behavior using the formula C=F ∗∩Feat(R 𝑒𝑥𝑝) (E1). The framework integrates various feature importance methods, including SHAP and LIME, due to their model-agnostic nature and widespread adoption (E3). The framework's workflow is presented in Algorithm 1 (E1).

**Experimental Setup:**
The experiments were conducted using a computational environment with a 13th Gen Intel Core i9-13900K processor, 128 GB RAM, and an NVIDIA RTX A6000 GPU with 48 GB memory (E1). The experiments were conducted using both dedicated hardware and a controlled software environment (E1). The authors provide open access to all

In [136]:
#Integrate Citation Validation and Claim Preparation

def summary_citation_node(state: SummaryState):
    reduced = state["reduced_result"]

    validated = validate_summary_citations(reduced)

    return {
        "validated_summary": validated,
        "execution_log": state["execution_log"] + [
            "citation_validation"
        ]
    }

def summary_claim_node(state: SummaryState):
    claims = prepare_summary_verification(
        state["validated_summary"]
    )

    return {
        "summary_claims": claims,
        "execution_log": state["execution_log"] + [
            "claim_preparation"
        ]
    }

# Extend the state schema without changing the old graph.
class VerifiedSummaryState(SummaryState, total=False):
    validated_summary: dict
    summary_claims: list
    verification_results: list
    filtered_summary: dict

verified_builder = StateGraph(VerifiedSummaryState)

verified_builder.add_node("retrieve", summary_retrieval_node)
verified_builder.add_node("map", summary_map_node)
verified_builder.add_node("reduce", summary_reduce_node)
verified_builder.add_node("citations", summary_citation_node)
verified_builder.add_node("claims", summary_claim_node)

verified_builder.add_edge(START, "retrieve")
verified_builder.add_edge("retrieve", "map")
verified_builder.add_edge("map", "reduce")
verified_builder.add_edge("reduce", "citations")
verified_builder.add_edge("citations", "claims")
verified_builder.add_edge("claims", END)

summary_preverification_graph = verified_builder.compile()

print("SUMMARY PREVERIFICATION GRAPH: COMPILED")


SUMMARY PREVERIFICATION GRAPH: COMPILED


In [137]:
#Add Claim Verification and Filtering Nodes


def summary_verify_node(state: VerifiedSummaryState):
    results = verify_summary_claims(
        state["summary_claims"], map_llm
    )

    return {
        "verification_results": results,
        "execution_log": state["execution_log"] + [
            "claim_verification"
        ]
    }

def summary_filter_node(state: VerifiedSummaryState):
    filtered = filter_verified_summary(
        state["verification_results"]
    )

    return {
        "filtered_summary": filtered,
        "execution_log": state["execution_log"] + [
            "claim_filtering"
        ]
    }

complete_builder = StateGraph(VerifiedSummaryState)

for name, fn in [
    ("retrieve", summary_retrieval_node),
    ("map", summary_map_node),
    ("reduce", summary_reduce_node),
    ("citations", summary_citation_node),
    ("claims", summary_claim_node),
    ("verify", summary_verify_node),
    ("filter", summary_filter_node)
]:
    complete_builder.add_node(name, fn)

complete_builder.add_edge(START, "retrieve")

steps = [
    "retrieve", "map", "reduce",
    "citations", "claims", "verify", "filter"
]

for current, following in zip(steps, steps[1:]):
    complete_builder.add_edge(current, following)

complete_builder.add_edge("filter", END)

complete_summary_graph = complete_builder.compile()

print("COMPLETE SUMMARY GRAPH: COMPILED")
print("Nodes:", list(complete_summary_graph.nodes))


COMPLETE SUMMARY GRAPH: COMPILED
Nodes: ['__start__', 'retrieve', 'map', 'reduce', 'citations', 'claims', 'verify', 'filter']


In [139]:


# FIX: LANGGRAPH STATE SCHEMA AND GRAPH RECOMPILATION


from typing import TypedDict
from langgraph.graph import StateGraph, START, END

# Define all fields used by the complete summary pipeline.
class VerifiedSummaryState(TypedDict, total=False):
    question: str
    retrieved_docs: list
    prepared_chunks: list
    mapped_summaries: list
    reduced_result: dict
    validated_summary: dict
    summary_claims: list
    verification_results: list
    filtered_summary: dict
    execution_log: list[str]


# Rebuild the complete graph using the corrected state.
fixed_builder = StateGraph(VerifiedSummaryState)

fixed_builder.add_node("retrieve", summary_retrieval_node)
fixed_builder.add_node("map", summary_map_node)
fixed_builder.add_node("reduce", summary_reduce_node)
fixed_builder.add_node("citations", summary_citation_node)
fixed_builder.add_node("claims", summary_claim_node)
fixed_builder.add_node("verify", summary_verify_node)
fixed_builder.add_node("filter", summary_filter_node)

fixed_builder.add_edge(START, "retrieve")
fixed_builder.add_edge("retrieve", "map")
fixed_builder.add_edge("map", "reduce")
fixed_builder.add_edge("reduce", "citations")
fixed_builder.add_edge("citations", "claims")
fixed_builder.add_edge("claims", "verify")
fixed_builder.add_edge("verify", "filter")
fixed_builder.add_edge("filter", END)

complete_summary_graph = fixed_builder.compile()

# Confirm the state fields are registered.
registered_fields = set(
    complete_summary_graph.get_input_jsonschema()
    .get("properties", {})
)

assert "question" in registered_fields

print("LANGGRAPH STATE FIX: PASS")
print("Graph nodes:", list(complete_summary_graph.nodes))
print("Complete graph recompiled successfully.")



LANGGRAPH STATE FIX: PASS
Graph nodes: ['__start__', 'retrieve', 'map', 'reduce', 'citations', 'claims', 'verify', 'filter']
Complete graph recompiled successfully.


In [141]:
# DEBUG: ISOLATE VALIDATED_SUMMARY STATE ISSUE


import inspect

print("1. STATE FIELDS:")
print(VerifiedSummaryState.__annotations__.keys())

print("\n2. CITATION NODE SOURCE:")
print(inspect.getsource(summary_citation_node))

print("\n3. CLAIM NODE SOURCE:")
print(inspect.getsource(summary_claim_node))

print("\n4. TEST CITATION NODE DIRECTLY:")

# Reuse the previously generated result.
test_state = {
    "reduced_result": reduced_result,
    "execution_log": ["retrieve", "map", "reduce"]
}

citation_output = summary_citation_node(test_state)

print("Returned keys:", list(citation_output.keys()))

assert "validated_summary" in citation_output, (
    "Citation node did not return validated_summary"
)

print("\n5. TEST CLAIM NODE DIRECTLY:")

claim_output = summary_claim_node({
    **test_state,
    **citation_output
})

print("Returned keys:", list(claim_output.keys()))
print("Claims:", len(claim_output["summary_claims"]))

print("\nDIRECT NODE TEST: PASS")


1. STATE FIELDS:
dict_keys(['question', 'retrieved_docs', 'prepared_chunks', 'mapped_summaries', 'reduced_result', 'validated_summary', 'summary_claims', 'verification_results', 'filtered_summary', 'execution_log'])

2. CITATION NODE SOURCE:
def summary_citation_node(state: SummaryState):
    reduced = state["reduced_result"]

    validated = validate_summary_citations(reduced)

    return {
        "validated_summary": validated,
        "execution_log": state["execution_log"] + [
            "citation_validation"
        ]
    }


3. CLAIM NODE SOURCE:
def summary_claim_node(state: SummaryState):
    claims = prepare_summary_verification(
        state["validated_summary"]
    )

    return {
        "summary_claims": claims,
        "execution_log": state["execution_log"] + [
            "claim_preparation"
        ]
    }


4. TEST CITATION NODE DIRECTLY:
Returned keys: ['validated_summary', 'execution_log']

5. TEST CLAIM NODE DIRECTLY:
Returned keys: ['summary_claims', 'execution

In [142]:

# FIX: NODE STATE TYPE ANNOTATIONS


def summary_citation_node(state: VerifiedSummaryState):
    reduced = state["reduced_result"]

    validated = validate_summary_citations(reduced)

    return {
        "validated_summary": validated,
        "execution_log": state["execution_log"] + [
            "citation_validation"
        ]
    }


def summary_claim_node(state: VerifiedSummaryState):
    claims = prepare_summary_verification(
        state["validated_summary"]
    )

    return {
        "summary_claims": claims,
        "execution_log": state["execution_log"] + [
            "claim_preparation"
        ]
    }


# Rebuild graph using corrected node signatures.
fixed_builder = StateGraph(VerifiedSummaryState)

nodes = [
    ("retrieve", summary_retrieval_node),
    ("map", summary_map_node),
    ("reduce", summary_reduce_node),
    ("citations", summary_citation_node),
    ("claims", summary_claim_node),
    ("verify", summary_verify_node),
    ("filter", summary_filter_node)
]

for name, function in nodes:
    fixed_builder.add_node(name, function)

fixed_builder.add_edge(START, "retrieve")

steps = [
    "retrieve", "map", "reduce",
    "citations", "claims", "verify", "filter"
]

for current, following in zip(steps, steps[1:]):
    fixed_builder.add_edge(current, following)

fixed_builder.add_edge("filter", END)

complete_summary_graph = fixed_builder.compile()

print("STATE ANNOTATION FIX: PASS")
print("GRAPH RECOMPILED: PASS")



STATE ANNOTATION FIX: PASS
GRAPH RECOMPILED: PASS


In [143]:


# TEST: STATE PROPAGATION WITHOUT LLM CALLS


test_builder = StateGraph(VerifiedSummaryState)

test_builder.add_node("citations", summary_citation_node)
test_builder.add_node("claims", summary_claim_node)

test_builder.add_edge(START, "citations")
test_builder.add_edge("citations", "claims")
test_builder.add_edge("claims", END)

test_graph = test_builder.compile()

test_result = test_graph.invoke({
    "reduced_result": reduced_result,
    "execution_log": []
})

assert "validated_summary" in test_result
assert "summary_claims" in test_result

print("STATE PROPAGATION: PASS")
print("Claims:", len(test_result["summary_claims"]))
print("Execution:", test_result["execution_log"])


STATE PROPAGATION: PASS
Claims: 6
Execution: ['citation_validation', 'claim_preparation']


In [144]:
#end-end verfication test

complete_result = complete_summary_graph.invoke({
    "question": (
        "Describe the eDySec framework methodology "
        "and experimental setup."
    ),
    "execution_log": []
})

filtered = complete_result["filtered_summary"]

assert len(complete_result["verification_results"]) > 0

assert (
    filtered["accepted_count"] + filtered["review_count"]
    == len(complete_result["verification_results"])
)

print("\nCOMPLETE SUMMARY PIPELINE: PASS")
print("Execution:", complete_result["execution_log"])
print("Accepted claims:", filtered["accepted_count"])
print("Review claims:", filtered["review_count"])
print(
    "Formula review:",
    filtered["manual_formula_review_count"]
)

print("\nVERIFIED SUMMARY DRAFT:\n")
print(filtered["verified_summary_draft"])
print("\nSTATUS:", filtered["status"])



CLAIM 1: SUPPORTED
Claim: The eDySec framework aims to detect malicious packages in the PyPI ecosystem using a deep learning-based approach.
Reason: The original source evidence explicitly states that the eDySec framework uses a deep learning-based approach to detect malicious packages in the PyPI ecosystem.

CLAIM 2: SUPPORTED
Claim: The eDySec framework assesses the relationship between selected features and model behavior using the formula C=F ∗∩Feat(R 𝑒𝑥𝑝).
Reason: The original source evidence (E1) explicitly mentions the formula C=F ∗∩Feat(R 𝑒𝑥𝑝) on page 5, which matches the claim.

CLAIM 3: SUPPORTED
Claim: The framework's workflow is presented in Algorithm 1.
Reason: The original text on page 5 of the source document explicitly mentions 'Algorithm 1 presents the pseudo-code for the proposed eDySec framework workflow.'

CLAIM 4: SUPPORTED
Claim: The framework consists of four phases: data preparation, feature selection, deep learning model selection and evaluation, and stability

# STEP 12: AGENTIC ROUTING DECISION

In [145]:

def decide_summary_route(state):
    results = state.get("verification_results", [])

    if not results:
        return "correct"

    for item in results:
        if item["verification_status"] != "SUPPORTED":
            return "correct"

    return "finish"


# Unit tests
supported_state = {
    "verification_results": [
        {"verification_status": "SUPPORTED"}
    ]
}

unsupported_state = {
    "verification_results": [
        {"verification_status": "INSUFFICIENT_EVIDENCE"}
    ]
}

assert decide_summary_route(supported_state) == "finish"
assert decide_summary_route(unsupported_state) == "correct"
assert decide_summary_route({}) == "correct"

print("AGENTIC ROUTING DECISION: PASS")
print("Supported -> finish")
print("Unsupported -> correct")


AGENTIC ROUTING DECISION: PASS
Supported -> finish
Unsupported -> correct


In [146]:
# STEP 12 — CELL 2: CORRECTIVE RETRIEVAL


def summary_corrective_retrieval_node(state):
    verification_results = state.get(
        "verification_results", []
    )

    unsupported = [
        item for item in verification_results
        if item["verification_status"] != "SUPPORTED"
    ]

    corrective_evidence = []

    for index, item in enumerate(unsupported, start=1):
        claim = item["claim"]

        # Retrieve additional original-paper evidence.
        docs = vectorstore.similarity_search(
            claim,
            k=3
        )

        corrective_evidence.append({
            "claim": claim,
            "original_status": item["verification_status"],
            "retrieved_docs": docs,
            "retrieval_id": f"CR{index}"
        })

    return {
        "corrective_evidence": corrective_evidence,
        "execution_log": state.get(
            "execution_log", []
        ) + ["corrective_retrieval"]
    }


assert callable(summary_corrective_retrieval_node)

print("CORRECTIVE RETRIEVAL NODE: LOADED")


CORRECTIVE RETRIEVAL NODE: LOADED


In [147]:

# STEP 12 — CELL 3: AGENTIC CONDITIONAL GRAPH


class AgenticSummaryState(VerifiedSummaryState, total=False):
    corrective_evidence: list


def agentic_route(state):
    results = state.get("verification_results", [])

    if not results:
        return "correct"

    if any(
        item["verification_status"] != "SUPPORTED"
        for item in results
    ):
        return "correct"

    return "finish"


agentic_builder = StateGraph(AgenticSummaryState)

for name, function in [
    ("retrieve", summary_retrieval_node),
    ("map", summary_map_node),
    ("reduce", summary_reduce_node),
    ("citations", summary_citation_node),
    ("claims", summary_claim_node),
    ("verify", summary_verify_node),
    ("filter", summary_filter_node),
    ("corrective_retrieval", summary_corrective_retrieval_node)
]:
    agentic_builder.add_node(name, function)

agentic_builder.add_edge(START, "retrieve")

agentic_steps = [
    "retrieve", "map", "reduce",
    "citations", "claims", "verify"
]

for current, following in zip(
    agentic_steps, agentic_steps[1:]
):
    agentic_builder.add_edge(current, following)

# Decision is based on verification results.
agentic_builder.add_conditional_edges(
    "verify",
    agentic_route,
    {
        "finish": "filter",
        "correct": "corrective_retrieval"
    }
)

agentic_builder.add_edge(
    "corrective_retrieval", "filter"
)
agentic_builder.add_edge("filter", END)

agentic_summary_graph = agentic_builder.compile()

print("AGENTIC SUMMARY GRAPH: COMPILED")
print("Conditional routing: ENABLED")



AGENTIC SUMMARY GRAPH: COMPILED
Conditional routing: ENABLED


In [148]:

# ROUTING TEST


route = agentic_route(complete_result)

assert route == "correct", (
    f"Expected corrective route, got {route}"
)

print("AGENTIC ROUTING TEST: PASS")
print("Selected route:", route)

# Execute only corrective retrieval.
correction_test = summary_corrective_retrieval_node(
    complete_result
)

assert len(correction_test["corrective_evidence"]) == 2

print(
    "Unsupported claims:",
    len(correction_test["corrective_evidence"])
)

for item in correction_test["corrective_evidence"]:
    print("\nClaim:", item["claim"][:120])
    print("Retrieved chunks:", len(item["retrieved_docs"]))

print("\nCORRECTIVE RETRIEVAL TEST: PASS")



AGENTIC ROUTING TEST: PASS
Selected route: correct
Unsupported claims: 2

Claim: Unfortunately, the provided context does not include any key findings or results. The references listed in the document 
Retrieved chunks: 3

Claim: The limitations of the research paper are not explicitly stated in the provided context. However, it is worth noting tha
Retrieved chunks: 3

CORRECTIVE RETRIEVAL TEST: PASS


In [149]:


# CORRECTIVE CLAIM RE-VERIFICATION


def prepare_corrective_verification(corrective_evidence):
    """
    Convert corrective retrieval results into the format
    expected by the existing claim verification function.
    """
    prepared = []

    for item in corrective_evidence:
        evidence_list = []

        for index, doc in enumerate(
            item["retrieved_docs"], start=1
        ):
            evidence_list.append({
                "evidence_id": f"{item['retrieval_id']}_{index}",
                "source": doc.metadata.get("source", "unknown"),
                "page": doc.metadata.get("page", "unknown"),
                "content": doc.page_content
            })

        prepared.append({
            "claim": item["claim"],
            "evidence_ids": [
                doc["evidence_id"] for doc in evidence_list
            ],
            "evidence": evidence_list,
            "verification_status": "PENDING"
        })

    return prepared


corrective_verification_items = prepare_corrective_verification(
    correction_test["corrective_evidence"]
)

assert len(corrective_verification_items) == 2

corrective_verification_results = verify_summary_claims(
    corrective_verification_items,
    map_llm
)

print("\nCORRECTIVE RE-VERIFICATION: COMPLETED")

for index, item in enumerate(
    corrective_verification_results, start=1
):
    print(
        f"Claim {index}: {item['verification_status']}"
    )



CLAIM 1: INSUFFICIENT_EVIDENCE
Claim: Unfortunately, the provided context does not include any key findings or results. The references listed in the document chunk  are related to other research papers and resources, but do not provide any information about the research paper itself.
Reason: The claim states that the provided context does not include any key findings or results, but the original source evidence includes several sections discussing feature importance, SHAP distributions, and local explanations for representative samples. While the references listed in the document chunk are related to other research papers and resources, they do not provide information about the research paper itself. However, the original source evidence does provide information about the research paper, specifically in sections [CR1_1] and [CR1_3]. Therefore, the claim is not supported by the evidence.

CLAIM 2: INSUFFICIENT_EVIDENCE
Claim: The limitations of the research paper are not explicitly sta

In [150]:


#  FINAL ANSWER REVISION


def build_corrected_summary(
    original_filtered,
    corrective_results
):
    accepted = list(
        original_filtered["accepted_claims"]
    )

    unresolved = []
    next_id = len(accepted) + 1

    for item in corrective_results:
        status = item["verification_status"]

        if status == "SUPPORTED":
            accepted.append({
                "claim_id": f"SC{next_id}",
                "claim": item["claim"],
                "evidence_ids": item["evidence_ids"],
                "verification_status": status,
                "verification_reason": item.get(
                    "verification_reason", ""
                ),
                "manual_formula_review": False
            })
            next_id += 1
        else:
            unresolved.append(item)

    lines = []

    for item in accepted:
        citations = " ".join(
            f"[{eid}]" for eid in item["evidence_ids"]
        )
        lines.append(
            f"- {item['claim']} {citations}"
        )

    final_summary = "\n".join(lines)

    status = (
        "DRAFT_FOR_REVIEW"
        if accepted
        else "ABSTAIN"
    )

    return {
        "final_summary": final_summary,
        "accepted_claims": accepted,
        "unresolved_claims": unresolved,
        "status": status
    }


corrected_summary_result = build_corrected_summary(
    complete_result["filtered_summary"],
    corrective_verification_results
)

print("FINAL ANSWER REVISION: PASS")
print("Status:", corrected_summary_result["status"])
print(
    "Accepted:",
    len(corrected_summary_result["accepted_claims"])
)
print(
    "Unresolved:",
    len(corrected_summary_result["unresolved_claims"])
)

print("\nFINAL SUMMARY:\n")
print(corrected_summary_result["final_summary"])


FINAL ANSWER REVISION: PASS
Status: DRAFT_FOR_REVIEW
Accepted: 6
Unresolved: 2

FINAL SUMMARY:

- The eDySec framework aims to detect malicious packages in the PyPI ecosystem using a deep learning-based approach. [E2]
- The eDySec framework assesses the relationship between selected features and model behavior using the formula C=F ∗∩Feat(R 𝑒𝑥𝑝). [E1]
- The framework's workflow is presented in Algorithm 1. [E1]
- The framework consists of four phases: data preparation, feature selection, deep learning model selection and evaluation, and stability and explainability analysis. [E2]
- The experiments were conducted using a computational environment with a 13th Gen Intel Core i9-13900K processor, 128 GB RAM, and an NVIDIA RTX A6000 GPU with 48 GB memory. [E1]
- The experiments were conducted using both dedicated hardware and a controlled software environment. [E1]


In [151]:

# FINAL SUMMARY VALIDATION


final_result = corrected_summary_result

assert all(
    item["verification_status"] == "SUPPORTED"
    for item in final_result["accepted_claims"]
)

accepted_ids = {
    eid
    for item in final_result["accepted_claims"]
    for eid in item["evidence_ids"]
}

# Citation tokens in the output.
import re

output_ids = set(
    re.findall(
        r"\[([A-Za-z0-9_]+)\]",
        final_result["final_summary"]
    )
)

assert output_ids == accepted_ids, (
    f"Citation mismatch: {output_ids ^ accepted_ids}"
)

if final_result["status"] == "ABSTAIN":
    assert not final_result["accepted_claims"]

print("FINAL SUMMARY VALIDATION: PASS")
print("Citation IDs:", sorted(output_ids))
print("Status:", final_result["status"])
print("Unresolved claims:", len(final_result["unresolved_claims"]))
print("Automated evidence checks only; manual review remains.")


FINAL SUMMARY VALIDATION: PASS
Citation IDs: ['E1', 'E2']
Status: DRAFT_FOR_REVIEW
Unresolved claims: 2
Automated evidence checks only; manual review remains.


In [152]:

# COMPLETE AGENTIC SUMMARY GRAPH


from typing import TypedDict
from langgraph.graph import StateGraph, START, END

class CompleteAgenticState(VerifiedSummaryState, total=False):
    corrective_evidence: list
    corrective_verification_results: list
    final_summary_result: dict


def agentic_corrective_verify_node(state: CompleteAgenticState):
    corrective_items = prepare_corrective_verification(
        state.get("corrective_evidence", [])
    )

    if corrective_items:
        results = verify_summary_claims(
            corrective_items, map_llm
        )
    else:
        results = []

    return {
        "corrective_verification_results": results,
        "execution_log": state.get(
            "execution_log", []
        ) + ["corrective_verification"]
    }


def agentic_final_revision_node(state: CompleteAgenticState):
    filtered = state["filtered_summary"]

    corrective_results = state.get(
        "corrective_verification_results", []
    )

    result = build_corrected_summary(
        filtered,
        corrective_results
    )

    return {
        "final_summary_result": result,
        "execution_log": state.get(
            "execution_log", []
        ) + ["final_revision"]
    }


agentic_complete_builder = StateGraph(CompleteAgenticState)

for name, function in [
    ("retrieve", summary_retrieval_node),
    ("map", summary_map_node),
    ("reduce", summary_reduce_node),
    ("citations", summary_citation_node),
    ("claims", summary_claim_node),
    ("verify", summary_verify_node),
    ("filter", summary_filter_node),
    ("corrective_retrieval", summary_corrective_retrieval_node),
    ("corrective_verify", agentic_corrective_verify_node),
    ("final_revision", agentic_final_revision_node)
]:
    agentic_complete_builder.add_node(name, function)

# Initial summarization and verification.
agentic_complete_builder.add_edge(START, "retrieve")

initial_steps = [
    "retrieve", "map", "reduce",
    "citations", "claims", "verify"
]

for current, following in zip(
    initial_steps, initial_steps[1:]
):
    agentic_complete_builder.add_edge(current, following)

# Always filter the initial verification results.
agentic_complete_builder.add_edge("verify", "filter")

# Conditional routing after filtering.
agentic_complete_builder.add_conditional_edges(
    "filter",
    agentic_route,
    {
        "finish": "final_revision",
        "correct": "corrective_retrieval"
    }
)

# Corrective branch.
agentic_complete_builder.add_edge(
    "corrective_retrieval", "corrective_verify"
)

agentic_complete_builder.add_edge(
    "corrective_verify", "final_revision"
)

agentic_complete_builder.add_edge("final_revision", END)

complete_agentic_graph = agentic_complete_builder.compile()

print("COMPLETE AGENTIC GRAPH: COMPILED")
print("Conditional corrective routing: ENABLED")
print("Corrective re-verification: CONNECTED")
print("Final answer revision: CONNECTED")



COMPLETE AGENTIC GRAPH: COMPILED
Conditional corrective routing: ENABLED
Corrective re-verification: CONNECTED
Final answer revision: CONNECTED


In [153]:

#  AGENTIC ROUTING UNIT TESTS


# Test corrective path using previous verification results.
correct_route = agentic_route(complete_result)

assert correct_route == "correct"

# Test finish path with supported claims only.
supported_test_state = {
    "verification_results": [
        {
            "claim": "Test supported claim",
            "verification_status": "SUPPORTED"
        }
    ]
}

finish_route = agentic_route(supported_test_state)

assert finish_route == "finish"

# Check the final revision result already produced.
assert corrected_summary_result["status"] in {
    "DRAFT_FOR_REVIEW",
    "ABSTAIN"
}

assert len(
    corrected_summary_result["unresolved_claims"]
) == 2

print("AGENTIC ROUTING UNIT TEST: PASS")
print("Unsupported claims ->", correct_route)
print("All supported ->", finish_route)
print(
    "Previously unresolved claims:",
    len(corrected_summary_result["unresolved_claims"])
)
print("Complete graph compilation: PASS")


AGENTIC ROUTING UNIT TEST: PASS
Unsupported claims -> correct
All supported -> finish
Previously unresolved claims: 2
Complete graph compilation: PASS


In [154]:

#  FULL AGENTIC WORKFLOW TEST


agentic_result = complete_agentic_graph.invoke({
    "question": (
        "Explain the eDySec methodology, "
        "experimental environment and research findings."
    ),
    "execution_log": []
})

final_output = agentic_result["final_summary_result"]
execution = agentic_result["execution_log"]

assert "final_revision" in execution

# Check whether the correct branch was executed.
initial_route = agentic_route(agentic_result)

if initial_route == "correct":
    assert "corrective_retrieval" in execution
    assert "corrective_verification" in execution
else:
    assert "corrective_retrieval" not in execution

assert final_output["status"] in {
    "DRAFT_FOR_REVIEW",
    "ABSTAIN"
}

print("\nFULL AGENTIC WORKFLOW: PASS")
print("Execution path:", " -> ".join(execution))
print("Route:", initial_route)
print("Accepted claims:", len(final_output["accepted_claims"]))
print("Unresolved claims:", len(final_output["unresolved_claims"]))
print("Status:", final_output["status"])

print("\nFINAL SUMMARY:\n")
print(final_output["final_summary"])

print(
    "\nNote: Automated evidence verification "
    "does not replace manual factual review."
)



CLAIM 1: SUPPORTED
Claim: The eDySec framework aims to detect malicious packages in the PyPI ecosystem using a deep learning-based approach.
Reason: The original source evidence explicitly states that the eDySec framework uses a deep learning-based approach to detect malicious packages in the PyPI ecosystem.

CLAIM 2: SUPPORTED
Claim: The eDySec framework assesses the relationship between selected features and model behavior using the formula C=F ∗∩Feat(R 𝑒𝑥𝑝).
Reason: The original source evidence (E1) explicitly mentions the formula C=F ∗∩Feat(R 𝑒𝑥𝑝) on page 5, which matches the claim.

CLAIM 3: SUPPORTED
Claim: The framework integrates various feature importance methods, including SHAP and LIME, due to their model-agnostic nature and widespread adoption.
Reason: The original source evidence explicitly states that SHAP and LIME 'remain the core of the eDySec due to their model-agnostic nature, strong theoretical foundations, broad applicability across architectures, and widespread ad

# Step 13: Prepare steamlit UI Data

In [155]:

import json
from pathlib import Path

def prepare_streamlit_data(agentic_result):
    final_output = agentic_result["final_summary_result"]

    claims = []

    for item in final_output["accepted_claims"]:
        claims.append({
            "claim": item["claim"],
            "status": item["verification_status"],
            "evidence_ids": item["evidence_ids"],
            "reason": item.get(
                "verification_reason", ""
            ),
            "manual_review": item.get(
                "manual_formula_review", False
            )
        })

    for item in final_output["unresolved_claims"]:
        claims.append({
            "claim": item["claim"],
            "status": item["verification_status"],
            "evidence_ids": item.get(
                "evidence_ids", []
            ),
            "reason": item.get(
                "verification_reason", ""
            ),
            "manual_review": True
        })

    return {
        "question": agentic_result["question"],
        "summary": final_output["final_summary"],
        "status": final_output["status"],
        "execution_log": agentic_result["execution_log"],
        "claims": claims,
        "accepted_count": len(
            final_output["accepted_claims"]
        ),
        "unresolved_count": len(
            final_output["unresolved_claims"]
        ),
        "mode": "SAVED_RESULT_DEMO"
    }


ui_data = prepare_streamlit_data(agentic_result)

output_path = Path("/content/streamlit_result.json")

output_path.write_text(
    json.dumps(ui_data, indent=2, ensure_ascii=False),
    encoding="utf-8"
)

assert output_path.exists()
assert ui_data["claims"]
assert ui_data["summary"]

print("STREAMLIT DATA PREPARATION: PASS")
print("File:", output_path)
print("Claims:", len(ui_data["claims"]))
print("Status:", ui_data["status"])
print("Mode:", ui_data["mode"])



STREAMLIT DATA PREPARATION: PASS
File: /content/streamlit_result.json
Claims: 8
Status: DRAFT_FOR_REVIEW
Mode: SAVED_RESULT_DEMO


In [156]:

#  CREATE STREAMLIT DASHBOARD


from pathlib import Path
import subprocess
import sys

# Install Streamlit if it is not already available.
try:
    import streamlit
except ImportError:
    subprocess.check_call([
        sys.executable, "-m", "pip",
        "install", "-q", "streamlit"
    ])

app_code = '''
import json
from pathlib import Path
import streamlit as st

st.set_page_config(
    page_title="Evidence-Verified RAG",
    page_icon="🔎",
    layout="wide"
)

DATA_PATH = Path("/content/streamlit_result.json")

st.title("Evidence-Verified RAG")
st.caption(
    "Research Question Answering | Map-Reduce | "
    "Claim Verification | Agentic Routing"
)

st.info(
    "Demo mode: displaying a previously generated result. "
    "Live inference is not connected yet."
)

if not DATA_PATH.exists():
    st.error("Saved result file was not found.")
    st.stop()

data = json.loads(DATA_PATH.read_text(encoding="utf-8"))

st.subheader("Research Question")
st.write(data["question"])

col1, col2, col3 = st.columns(3)

col1.metric("Supported Claims", data["accepted_count"])
col2.metric("Unresolved Claims", data["unresolved_count"])
col3.metric("Final Status", data["status"])

st.divider()

st.subheader("Evidence-Verified Summary")
st.markdown(data["summary"])

st.divider()

st.subheader("Claim-Level Verification")

for index, claim in enumerate(data["claims"], start=1):
    status = claim["status"]

    with st.expander(
        f"Claim {index} — {status}",
        expanded=False
    ):
        st.write(claim["claim"])
        st.write("Evidence IDs:", ", ".join(claim["evidence_ids"]))
        st.write("Verification:", status)
        st.write("Reason:", claim["reason"])

        if claim["manual_review"]:
            st.warning("Manual review recommended.")

st.divider()

st.subheader("Agent Execution Trace")

for index, step in enumerate(data["execution_log"], start=1):
    st.write(f"{index}. {step}")

st.caption(
    "Automated verification is not ground-truth validation. "
    "Scientific and mathematical claims may require manual review."
)
'''

app_path = Path("/content/streamlit_app.py")
app_path.write_text(app_code, encoding="utf-8")

# Check Python syntax without starting the server.
compile(app_code, str(app_path), "exec")

print("STREAMLIT DASHBOARD: CREATED")
print("File:", app_path)
print("Syntax validation: PASS")
print("Mode: SAVED_RESULT_DEMO")


STREAMLIT DASHBOARD: CREATED
File: /content/streamlit_app.py
Syntax validation: PASS
Mode: SAVED_RESULT_DEMO


In [157]:

# LAUNCH STREAMLIT IN GOOGLE COLAB


import subprocess
import time
from pathlib import Path
from google.colab import output

# Confirm that the app and saved results exist.
assert Path("/content/streamlit_app.py").exists()
assert Path("/content/streamlit_result.json").exists()

# Stop only the Streamlit process started by this cell
# if the cell is executed again.
try:
    streamlit_process
except NameError:
    streamlit_process = None

if streamlit_process is not None:
    if streamlit_process.poll() is None:
        streamlit_process.terminate()
        streamlit_process.wait(timeout=5)

# Start Streamlit in the background.
streamlit_log = open(
    "/content/streamlit_server.log", "w"
)

streamlit_process = subprocess.Popen(
    [
        "streamlit",
        "run",
        "/content/streamlit_app.py",
        "--server.port=8501",
        "--server.address=0.0.0.0",
        "--server.headless=true"
    ],
    stdout=streamlit_log,
    stderr=subprocess.STDOUT
)

time.sleep(4)

assert streamlit_process.poll() is None, (
    "Streamlit server stopped. Check streamlit_server.log."
)

print("STREAMLIT SERVER: STARTED")
print("Port: 8501")
print("Mode: SAVED_RESULT_DEMO")

# Open the Colab proxy preview.
output.serve_kernel_port_as_window(8501)


STREAMLIT SERVER: STARTED
Port: 8501
Mode: SAVED_RESULT_DEMO
Try `serve_kernel_port_as_iframe` instead. 


<IPython.core.display.Javascript object>

In [159]:

# DIAGNOSE STREAMLIT LOADING


import requests
from pathlib import Path

try:
    response = requests.get(
        "http://127.0.0.1:8501/_stcore/health",
        timeout=10
    )

    print("HTTP STATUS:", response.status_code)
    print("SERVER RESPONSE:", response.text)

    if response.status_code == 200:
        print("STREAMLIT HEALTH CHECK: PASS")
    else:
        print("STREAMLIT HEALTH CHECK: FAILED")

except Exception as error:
    print("SERVER CONNECTION ERROR:", error)

log_path = Path("/content/streamlit_server.log")

if log_path.exists():
    print("\n--- RECENT SERVER LOG ---")
    print(log_path.read_text(errors="replace")[-3000:])

print("\nAPP FILE:", Path("/content/streamlit_app.py").exists())
print("RESULT FILE:", Path("/content/streamlit_result.json").exists())


HTTP STATUS: 200
SERVER RESPONSE: ok
STREAMLIT HEALTH CHECK: PASS

--- RECENT SERVER LOG ---
ejecting WebSocket connection with disallowed Origin or Host header: origin=https://8501-gpu-t4-hm-ass1a0-2epluxyo6q35e-a.asia-southeast1.prod.r.colab.dev, host=gpu-t4-hm-ass1a0-2epluxyo6q35e.asia-southeast1-a.c.codatalab-user-runtimes.internal:8007
2026-10-27 05:54:22.354 Rejecting WebSocket connection with disallowed Origin or Host header: origin=https://8501-gpu-t4-hm-ass1a0-2epluxyo6q35e-a.asia-southeast1.prod.r.colab.dev, host=gpu-t4-hm-ass1a0-2epluxyo6q35e.asia-southeast1-a.c.codatalab-user-runtimes.internal:8007
2026-10-27 05:54:45.806 Rejecting WebSocket connection with disallowed Origin or Host header: origin=https://8501-gpu-t4-hm-ass1a0-2epluxyo6q35e-a.asia-southeast1.prod.r.colab.dev, host=gpu-t4-hm-ass1a0-2epluxyo6q35e.asia-southeast1-a.c.codatalab-user-runtimes.internal:8007
2026-10-27 05:55:06.043 Rejecting WebSocket connection with disallowed Origin or Host header: origin=https:

In [161]:
# OPEN STREAMLIT VIA CLOUDFLARE

import subprocess
import requests
import time
import re
from pathlib import Path
from IPython.display import display, HTML

# Confirm Streamlit is responding.
health = requests.get(
    "http://127.0.0.1:8501/_stcore/health",
    timeout=10
)

assert health.status_code == 200, (
    "Streamlit server is not healthy."
)

# Install cloudflared binary if needed.
binary_path = Path("/content/cloudflared")

if not binary_path.exists():
    download_url = (
        "https://github.com/cloudflare/cloudflared/"
        "releases/latest/download/cloudflared-linux-amd64"
    )

    response = requests.get(download_url, timeout=120)
    response.raise_for_status()
    binary_path.write_bytes(response.content)
    binary_path.chmod(0o755)

# Stop only a tunnel previously started by this notebook cell.
try:
    cloudflared_process
except NameError:
    cloudflared_process = None

if cloudflared_process is not None:
    if cloudflared_process.poll() is None:
        cloudflared_process.terminate()
        cloudflared_process.wait(timeout=10)

tunnel_log_path = Path("/content/cloudflared.log")
tunnel_log = open(tunnel_log_path, "w")

cloudflared_process = subprocess.Popen(
    [
        str(binary_path),
        "tunnel",
        "--url",
        "http://127.0.0.1:8501",
        "--no-autoupdate"
    ],
    stdout=tunnel_log,
    stderr=subprocess.STDOUT
)

public_url = None

for attempt in range(30):
    time.sleep(2)

    log_text = tunnel_log_path.read_text(
        errors="replace"
    )

    match = re.search(
        r"https://[a-zA-Z0-9-]+\\.trycloudflare\\.com",
        log_text
    )

    if match:
        public_url = match.group(0)
        break

    if cloudflared_process.poll() is not None:
        break

if public_url:
    print("STREAMLIT PUBLIC TUNNEL: READY")
    print("Dashboard URL:", public_url)
    display(HTML(
        f'<a href="{public_url}" target="_blank">'
        'Open Streamlit Dashboard</a>'
    ))
else:
    print("TUNNEL NOT READY")
    print(tunnel_log_path.read_text(errors="replace")[-2500:])


TUNNEL NOT READY
nnel.com  PASS    DNS Resolved successfully     |
2026-10-27T05:59:12Z INF |  DNS Resolution    region2.v2.argotunnel.com  PASS    DNS Resolved successfully     |
2026-10-27T05:59:12Z INF |  UDP Connectivity  region1.v2.argotunnel.com  PASS    QUIC connection successful    |
2026-10-27T05:59:12Z INF |  UDP Connectivity  region2.v2.argotunnel.com  PASS    QUIC connection successful    |
2026-10-27T05:59:12Z INF |  TCP Connectivity  region1.v2.argotunnel.com  PASS    HTTP/2 connection successful  |
2026-10-27T05:59:12Z INF |  TCP Connectivity  region2.v2.argotunnel.com  PASS    HTTP/2 connection successful  |
2026-10-27T05:59:12Z INF |  Cloudflare API    api.cloudflare.com:443     PASS    API is reachable              |
2026-10-27T05:59:12Z INF |                                                                                     |
2026-10-27T05:59:12Z INF |  SUMMARY: Environment is healthy. cloudflared will use 'quic' as primary protocol.  |
2026-10-27T05:59:12Z INF +---

In [162]:

#  CHECK EXISTING CLOUDFLARE TUNNEL


import re
import requests
from pathlib import Path
from IPython.display import display, HTML

log_path = Path("/content/cloudflared.log")

# Check Streamlit server.
try:
    response = requests.get(
        "http://127.0.0.1:8501/_stcore/health",
        timeout=10
    )
    print("STREAMLIT STATUS:", response.status_code)
except Exception as error:
    print("STREAMLIT ERROR:", error)

# Check the existing tunnel process.
if "cloudflared_process" in globals():
    print(
        "TUNNEL PROCESS RUNNING:",
        cloudflared_process.poll() is None
    )
else:
    print("TUNNEL PROCESS: NOT FOUND")

# Search the complete log for the public URL.
if log_path.exists():
    log_text = log_path.read_text(errors="replace")

    urls = re.findall(
        r"https://[a-zA-Z0-9-]+\.trycloudflare\.com",
        log_text
    )

    if urls:
        public_url = urls[-1]
        print("PUBLIC URL:", public_url)
        display(HTML(
            f'<a href="{public_url}" target="_blank">'
            'Open Streamlit Dashboard</a>'
        ))
    else:
        print("PUBLIC URL: NOT FOUND")
        print("\n--- LAST 5000 LOG CHARACTERS ---")
        print(log_text[-5000:])
else:
    print("CLOUDFLARE LOG: NOT FOUND")


STREAMLIT STATUS: 200
TUNNEL PROCESS RUNNING: True
PUBLIC URL: https://specializing-beach-tournaments-automobile.trycloudflare.com


In [163]:

#FASTAPI BACKEND PREPARATION


import sys
import subprocess
import importlib.util

packages = {
    "fastapi": "fastapi",
    "uvicorn": "uvicorn",
    "requests": "requests",
}

for module_name, package_name in packages.items():
    if importlib.util.find_spec(module_name) is None:
        subprocess.check_call([
            sys.executable, "-m", "pip",
            "install", "-q", package_name
        ])

assert "complete_agentic_graph" in globals(), (
    "Complete agentic graph is missing. "
    "Restore the graph before continuing."
)

assert "prepare_streamlit_data" in globals(), (
    "Streamlit data preparation function is missing."
)

print("FASTAPI PREPARATION: PASS")
print("Agentic graph: AVAILABLE")
print("Streamlit formatter: AVAILABLE")



FASTAPI PREPARATION: PASS
Agentic graph: AVAILABLE
Streamlit formatter: AVAILABLE


In [164]:

# CREATE LIVE RAG API


from fastapi import FastAPI, HTTPException
from pydantic import BaseModel, Field
from threading import Lock
import traceback

api_app = FastAPI(
    title="Evidence-Verified RAG API",
    version="0.1.0"
)

# Prevent simultaneous requests from overloading Ollama.
inference_lock = Lock()

class ResearchRequest(BaseModel):
    question: str = Field(min_length=3, max_length=2000)

@api_app.get("/health")
def health_check():
    return {
        "status": "ok",
        "graph_available": "complete_agentic_graph" in globals(),
        "mode": "LIVE_AGENTIC_SUMMARIZATION"
    }

@api_app.post("/ask")
def ask_research_question(request: ResearchRequest):

    if not inference_lock.acquire(blocking=False):
        raise HTTPException(
            status_code=429,
            detail="Another research request is processing."
        )

    try:
        result = complete_agentic_graph.invoke({
            "question": request.question,
            "execution_log": []
        })

        response = prepare_streamlit_data(result)

        response["mode"] = "LIVE_AGENTIC_SUMMARIZATION"

        return response

    except Exception as error:
        print("LIVE RAG ERROR:")
        traceback.print_exc()

        raise HTTPException(
            status_code=500,
            detail=(
                "Live inference failed. "
                "Check the Colab backend logs."
            )
        ) from error

    finally:
        inference_lock.release()

print("FASTAPI ENDPOINTS: CREATED")
print("GET  /health")
print("POST /ask")
print("Mode: LIVE_AGENTIC_SUMMARIZATION")



FASTAPI ENDPOINTS: CREATED
GET  /health
POST /ask
Mode: LIVE_AGENTIC_SUMMARIZATION


In [165]:

#  START FASTAPI SERVER


import threading
import time
import requests
import uvicorn

# Reuse the existing server if already running.
if (
    "api_thread" not in globals()
    or not api_thread.is_alive()
):
    api_config = uvicorn.Config(
        api_app,
        host="127.0.0.1",
        port=8000,
        log_level="warning",
        access_log=False
    )

    api_server = uvicorn.Server(api_config)

    api_thread = threading.Thread(
        target=api_server.run,
        daemon=True
    )

    api_thread.start()

    time.sleep(3)

response = requests.get(
    "http://127.0.0.1:8000/health",
    timeout=10
)

assert response.status_code == 200, response.text
assert response.json()["graph_available"] is True

print("FASTAPI SERVER: PASS")
print("Port: 8000")
print("Health:", response.json())


FASTAPI SERVER: PASS
Port: 8000
Health: {'status': 'ok', 'graph_available': True, 'mode': 'LIVE_AGENTIC_SUMMARIZATION'}


In [166]:

#  API SMOKE TEST


import requests

base_url = "http://127.0.0.1:8000"

health_response = requests.get(
    f"{base_url}/health",
    timeout=10
)

assert health_response.status_code == 200

invalid_response = requests.post(
    f"{base_url}/ask",
    json={"question": ""},
    timeout=10
)

assert invalid_response.status_code == 422, (
    invalid_response.text
)

print("API HEALTH TEST: PASS")
print("QUESTION VALIDATION TEST: PASS")
print("Invalid question status:", invalid_response.status_code)



API HEALTH TEST: PASS
QUESTION VALIDATION TEST: PASS
Invalid question status: 422


In [167]:
#LIVE STREAMLIT INTERFACE


from pathlib import Path

app_code = '''
import json
from pathlib import Path

import requests
import streamlit as st

st.set_page_config(
    page_title="Evidence-Verified RAG",
    page_icon="🔎",
    layout="wide"
)

API_URL = "http://127.0.0.1:8000"
SAVED_PATH = Path("/content/streamlit_result.json")

st.title("Evidence-Verified RAG")
st.caption(
    "Research Summarization | Claim Verification | "
    "Agentic Corrective Retrieval"
)

st.warning(
    "Research prototype. Automated verification is not "
    "a guarantee of factual correctness."
)

# Check backend availability.
try:
    health_response = requests.get(
        f"{API_URL}/health",
        timeout=5
    )
    backend_online = health_response.status_code == 200
except requests.RequestException:
    backend_online = False

if backend_online:
    st.success("Live RAG backend: Connected")
else:
    st.error("Live RAG backend: Disconnected")

mode = st.radio(
    "Select mode",
    ["Saved Result Demo", "Live Agentic Summarization"],
    horizontal=True
)

data = None

if mode == "Saved Result Demo":
    if SAVED_PATH.exists():
        data = json.loads(
            SAVED_PATH.read_text(encoding="utf-8")
        )
    else:
        st.error("Saved result file not found.")

else:
    question = st.text_area(
        "Enter your research question",
        value=(
            "Summarize the eDySec methodology, "
            "experimental setup, and evaluation."
        ),
        height=100
    )

    if st.button(
        "Generate Verified Summary",
        type="primary",
        disabled=not backend_online
    ):
        if len(question.strip()) < 3:
            st.warning("Enter a valid research question.")
        else:
            with st.spinner(
                "Retrieving evidence and verifying claims..."
            ):
                try:
                    response = requests.post(
                        f"{API_URL}/ask",
                        json={"question": question.strip()},
                        timeout=900
                    )
                    response.raise_for_status()

                    st.session_state["live_result"] = (
                        response.json()
                    )

                except requests.RequestException as error:
                    st.error(
                        "Live inference failed. "
                        "Check the Colab backend logs."
                    )
                    st.caption(str(error))

    data = st.session_state.get("live_result")

if data:
    st.divider()

    st.subheader("Research Question")
    st.write(data.get("question", ""))

    col1, col2, col3 = st.columns(3)

    col1.metric(
        "Supported Claims",
        data.get("accepted_count", 0)
    )
    col2.metric(
        "Unresolved Claims",
        data.get("unresolved_count", 0)
    )
    col3.metric(
        "Status",
        data.get("status", "UNKNOWN")
    )

    st.subheader("Evidence-Verified Summary")
    st.markdown(data.get("summary", ""))

    st.subheader("Claim-Level Verification")

    for index, claim in enumerate(
        data.get("claims", []), start=1
    ):
        with st.expander(
            f"Claim {index}: {claim.get('status', 'UNKNOWN')}"
        ):
            st.write(claim.get("claim", ""))
            st.write(
                "Evidence IDs:",
                ", ".join(claim.get("evidence_ids", []))
            )
            st.write(
                "Reason:",
                claim.get("reason", "")
            )

            if claim.get("manual_review"):
                st.warning("Manual review recommended.")

    st.subheader("Agent Execution Trace")

    for index, step in enumerate(
        data.get("execution_log", []), start=1
    ):
        st.write(f"{index}. {step}")

    st.caption(
        f"Mode: {data.get('mode', 'UNKNOWN')}"
    )
'''

app_path = Path("/content/streamlit_app.py")

# Preserve the previous working dashboard.
backup_path = Path("/content/streamlit_app_backup.py")
if app_path.exists() and not backup_path.exists():
    backup_path.write_text(
        app_path.read_text(encoding="utf-8"),
        encoding="utf-8"
    )

compile(app_code, str(app_path), "exec")
app_path.write_text(app_code, encoding="utf-8")

print("LIVE STREAMLIT INTERFACE: CREATED")
print("Syntax validation: PASS")
print("Previous dashboard backup:", backup_path)



LIVE STREAMLIT INTERFACE: CREATED
Syntax validation: PASS
Previous dashboard backup: /content/streamlit_app_backup.py


In [169]:

#  VERIFY LIVE INTERFACE


import requests
from pathlib import Path

api_response = requests.get(
    "http://127.0.0.1:8000/health",
    timeout=10
)

ui_response = requests.get(
    "http://127.0.0.1:8501/_stcore/health",
    timeout=10
)

assert api_response.status_code == 200
assert ui_response.status_code == 200

source = Path(
    "/content/streamlit_app.py"
).read_text(encoding="utf-8")

assert "Generate Verified Summary" in source
assert 'f"{API_URL}/ask"' in source
assert "Live Agentic Summarization" in source

print("FASTAPI CONNECTION: PASS")
print("STREAMLIT CONNECTION: PASS")
print("LIVE UI SOURCE CHECK: PASS")



FASTAPI CONNECTION: PASS
STREAMLIT CONNECTION: PASS
LIVE UI SOURCE CHECK: PASS


In [170]:

#  OPEN UPDATED DASHBOARD


import re
from pathlib import Path
from IPython.display import display, HTML

log_text = Path(
    "/content/cloudflared.log"
).read_text(errors="replace")

urls = re.findall(
    r"https://[a-zA-Z0-9-]+\.trycloudflare\.com",
    log_text
)

assert urls, "No Cloudflare URL found."

dashboard_url = urls[-1]

print("DASHBOARD URL:", dashboard_url)
print("Refresh the browser to load the updated interface.")

display(HTML(
    f'<a href="{dashboard_url}" target="_blank">'
    'Open Updated RAG Dashboard</a>'
))


DASHBOARD URL: https://specializing-beach-tournaments-automobile.trycloudflare.com
Refresh the browser to load the updated interface.


Step 14: Inspect evidence structure

In [171]:

# INSPECT EVIDENCE STRUCTURE


assert "agentic_result" in globals(), (
    "Previous agentic result is missing."
)

print("STEP 14: EVIDENCE INSPECTION")
print("=" * 50)

print("\nAvailable result keys:")
for key in agentic_result.keys():
    print("-", key)

reduced = agentic_result.get("reduced_result", {})
registry = reduced.get("evidence_registry", {})

print("\nEvidence registry type:", type(registry).__name__)
print("Evidence registry entries:", len(registry))

if isinstance(registry, dict):
    for evidence_id, entry in list(registry.items())[:3]:
        print("\nEvidence ID:", evidence_id)
        print("Entry type:", type(entry).__name__)

        if isinstance(entry, dict):
            print("Available fields:", list(entry.keys()))
            print("Source:", entry.get("source"))
            print("Page:", entry.get("page"))

print("\nEVIDENCE INSPECTION: COMPLETE")


STEP 14: EVIDENCE INSPECTION

Available result keys:
- question
- retrieved_docs
- prepared_chunks
- mapped_summaries
- reduced_result
- validated_summary
- summary_claims
- verification_results
- filtered_summary
- execution_log
- final_summary_result

Evidence registry type: dict
Evidence registry entries: 3

Evidence ID: E1
Entry type: dict
Available fields: ['source', 'page', 'document']
Source: dl based detecting malicious packages in pypl.pdf
Page: 5

Evidence ID: E2
Entry type: dict
Available fields: ['source', 'page', 'document']
Source: dl based detecting malicious packages in pypl.pdf
Page: 14

Evidence ID: E3
Entry type: dict
Available fields: ['source', 'page', 'document']
Source: dl based detecting malicious packages in pypl.pdf
Page: 6

EVIDENCE INSPECTION: COMPLETE


In [172]:

# BUILD EVIDENCE VIEWER DATA


def build_evidence_viewer_data(agentic_result):
    reduced = agentic_result.get("reduced_result", {})
    registry = reduced.get("evidence_registry", {})

    evidence_items = []

    if isinstance(registry, dict):
        entries = registry.items()
    elif isinstance(registry, list):
        entries = [
            (f"E{i}", item)
            for i, item in enumerate(registry, start=1)
        ]
    else:
        entries = []

    for evidence_id, entry in entries:
        if isinstance(entry, dict):
            document = entry.get("document")
            metadata = entry.get("metadata") or {}

            source = entry.get("source")
            page = entry.get("page")
            content = entry.get("content", "")

            if document is not None:
                doc_metadata = getattr(
                    document, "metadata", {}
                ) or {}

                metadata = {**doc_metadata, **metadata}

                if not content:
                    content = getattr(
                        document, "page_content", ""
                    )

            if source is None:
                source = metadata.get("source", "Unknown")

            if page is None:
                page = metadata.get("page")

        else:
            metadata = getattr(entry, "metadata", {}) or {}
            source = metadata.get("source", "Unknown")
            page = metadata.get("page")
            content = getattr(entry, "page_content", "")

        evidence_items.append({
            "evidence_id": str(evidence_id),
            "source": str(source),
            "page": page,
            "content": str(content),
            "page_note": (
                "Original metadata page value; "
                "PDF display numbering may differ."
            )
        })

    return evidence_items


evidence_viewer_data = build_evidence_viewer_data(
    agentic_result
)

print("EVIDENCE VIEWER DATA: CREATED")
print("Evidence entries:", len(evidence_viewer_data))

for item in evidence_viewer_data[:3]:
    print(
        item["evidence_id"],
        "| Source:", item["source"],
        "| Page:", item["page"],
        "| Characters:", len(item["content"])
    )




EVIDENCE VIEWER DATA: CREATED
Evidence entries: 3
E1 | Source: dl based detecting malicious packages in pypl.pdf | Page: 5 | Characters: 983
E2 | Source: dl based detecting malicious packages in pypl.pdf | Page: 14 | Characters: 998
E3 | Source: dl based detecting malicious packages in pypl.pdf | Page: 6 | Characters: 931


In [173]:
# VALIDATE CITATION COVERAGE


import re

final_summary = agentic_result[
    "final_summary_result"
]["final_summary"]

citation_ids = sorted(set(
    re.findall(r"\[([A-Za-z0-9_]+)\]", final_summary)
))

registered_ids = {
    item["evidence_id"]
    for item in evidence_viewer_data
}

missing_ids = [
    citation_id
    for citation_id in citation_ids
    if citation_id not in registered_ids
]

print("SUMMARY CITATION IDS:", citation_ids)
print("REGISTERED EVIDENCE IDS:", sorted(registered_ids))
print("MISSING EVIDENCE IDS:", missing_ids)

if missing_ids:
    print("CITATION COVERAGE: NEEDS REVIEW")
else:
    print("CITATION COVERAGE: PASS")



SUMMARY CITATION IDS: ['E1', 'E2', 'E3']
REGISTERED EVIDENCE IDS: ['E1', 'E2', 'E3']
MISSING EVIDENCE IDS: []
CITATION COVERAGE: PASS


In [174]:

# EXPORT EVIDENCE VIEWER DATA


import json
from pathlib import Path

evidence_export = {
    "mode": "SAVED_EVIDENCE_SNAPSHOT",
    "question": agentic_result.get("question", ""),
    "evidence": evidence_viewer_data,
    "citation_ids": citation_ids,
    "missing_citations": missing_ids,
    "citation_coverage_pass": len(missing_ids) == 0
}

evidence_path = Path(
    "/content/streamlit_evidence.json"
)

evidence_path.write_text(
    json.dumps(
        evidence_export,
        indent=2,
        ensure_ascii=False,
        default=str
    ),
    encoding="utf-8"
)

assert evidence_path.exists()

print("EVIDENCE EXPORT: PASS")
print("File:", evidence_path)
print("Evidence passages:", len(evidence_viewer_data))
print("Citation coverage:", (
    "PASS" if not missing_ids else "NEEDS REVIEW"
))



EVIDENCE EXPORT: PASS
File: /content/streamlit_evidence.json
Evidence passages: 3
Citation coverage: PASS


In [175]:
# ADD SAVED EVIDENCE VIEWER


from pathlib import Path
import shutil

app_path = Path("/content/streamlit_app.py")
backup_path = Path("/content/streamlit_app_before_evidence.py")

assert app_path.exists(), "Streamlit app not found."

# Keep a backup of the working application.
if not backup_path.exists():
    shutil.copy2(app_path, backup_path)

source = app_path.read_text(encoding="utf-8")

marker = "# STEP 14: SAVED EVIDENCE VIEWER"

evidence_ui = '''

# STEP 14: SAVED EVIDENCE VIEWER
st.divider()
st.subheader("Original PDF Evidence Viewer")

st.caption(
    "Saved evidence snapshot only. These passages belong "
    "to the earlier notebook result, not necessarily "
    "to the current live question."
)

evidence_path = Path("/content/streamlit_evidence.json")

if evidence_path.exists():
    evidence_data = json.loads(
        evidence_path.read_text(encoding="utf-8")
    )

    st.write(
        "Snapshot question:",
        evidence_data.get("question", "")
    )

    evidence_items = evidence_data.get("evidence", [])

    if evidence_items:
        selected_id = st.selectbox(
            "Select evidence citation",
            [
                item["evidence_id"]
                for item in evidence_items
            ]
        )

        selected_item = next(
            item for item in evidence_items
            if item["evidence_id"] == selected_id
        )

        st.write(
            "**Source PDF:**",
            selected_item["source"]
        )
        st.write(
            "**Page metadata:**",
            selected_item["page"]
        )

        st.text_area(
            "Original retrieved passage",
            value=selected_item["content"],
            height=300,
            disabled=True
        )

        st.caption(selected_item["page_note"])

    missing = evidence_data.get(
        "missing_citations", []
    )

    if missing:
        st.warning(
            "Missing citation IDs: " + ", ".join(missing)
        )
    else:
        st.success(
            "All snapshot summary citations have "
            "matching evidence registry entries."
        )

else:
    st.info("Saved evidence snapshot is not available.")
'''

if marker not in source:
    updated_source = source + evidence_ui
    compile(updated_source, str(app_path), "exec")
    app_path.write_text(
        updated_source,
        encoding="utf-8"
    )
    print("EVIDENCE VIEWER: ADDED")
else:
    print("EVIDENCE VIEWER: ALREADY PRESENT")

print("Streamlit syntax: PASS")
print("Backup:", backup_path)



EVIDENCE VIEWER: ADDED
Streamlit syntax: PASS
Backup: /content/streamlit_app_before_evidence.py


In [176]:

# VERIFY EVIDENCE VIEWER


import json
from pathlib import Path
import requests

app_source = Path(
    "/content/streamlit_app.py"
).read_text(encoding="utf-8")

evidence_data = json.loads(
    Path(
        "/content/streamlit_evidence.json"
    ).read_text(encoding="utf-8")
)

assert "Original PDF Evidence Viewer" in app_source
assert "Generate Verified Summary" in app_source
assert "Live Agentic Summarization" in app_source

assert len(evidence_data["evidence"]) == 3
assert evidence_data["citation_coverage_pass"]

health = requests.get(
    "http://127.0.0.1:8501/_stcore/health",
    timeout=10
)

assert health.status_code == 200

print("EVIDENCE VIEWER SOURCE CHECK: PASS")
print("LIVE INPUT PRESERVED: PASS")
print("EVIDENCE JSON VALIDATION: PASS")
print("STREAMLIT SERVER HEALTH: PASS")



EVIDENCE VIEWER SOURCE CHECK: PASS
LIVE INPUT PRESERVED: PASS
EVIDENCE JSON VALIDATION: PASS
STREAMLIT SERVER HEALTH: PASS


In [177]:

# OPEN UPDATED EVIDENCE VIEWER


import re
from pathlib import Path
from IPython.display import display, HTML

log_text = Path(
    "/content/cloudflared.log"
).read_text(errors="replace")

urls = re.findall(
    r"https://[a-zA-Z0-9-]+\.trycloudflare\.com",
    log_text
)

assert urls, "Cloudflare URL not found."

dashboard_url = urls[-1]

print("DASHBOARD URL:", dashboard_url)
print("Refresh the browser and scroll down.")

display(HTML(
    f'<a href="{dashboard_url}" target="_blank">'
    'Open Evidence Viewer Dashboard</a>'
))


DASHBOARD URL: https://specializing-beach-tournaments-automobile.trycloudflare.com
Refresh the browser and scroll down.


In [178]:

# LIVE EVIDENCE API


from fastapi import HTTPException
import re
import traceback

@api_app.post("/ask-with-evidence")
def ask_with_evidence(request: ResearchRequest):

    if not inference_lock.acquire(blocking=False):
        raise HTTPException(
            status_code=429,
            detail="Another request is processing."
        )

    try:
        result = complete_agentic_graph.invoke({
            "question": request.question,
            "execution_log": []
        })

        response = prepare_streamlit_data(result)
        response["mode"] = "LIVE_AGENTIC_SUMMARIZATION"

        # Use evidence from this exact graph execution.
        evidence = build_evidence_viewer_data(result)

        # Include corrective retrieval evidence when present.
        corrective = result.get("corrective_evidence", [])
        for item in corrective:
            for index, doc in enumerate(
                item.get("retrieved_docs", []), start=1
            ):
                metadata = getattr(doc, "metadata", {}) or {}
                evidence.append({
                    "evidence_id": (
                        f"{item.get('retrieval_id', 'CR')}_{index}"
                    ),
                    "source": str(
                        metadata.get("source", "Unknown")
                    ),
                    "page": metadata.get("page"),
                    "content": getattr(doc, "page_content", ""),
                    "page_note": "Corrective retrieval evidence"
                })

        cited_ids = sorted(set(re.findall(
            r"\[([A-Za-z0-9_]+)\]",
            response.get("summary", "")
        )))

        registered_ids = {
            item["evidence_id"] for item in evidence
        }

        response["evidence"] = evidence
        response["citation_ids"] = cited_ids
        response["missing_citations"] = [
            cid for cid in cited_ids
            if cid not in registered_ids
        ]
        response["citation_coverage_pass"] = (
            len(response["missing_citations"]) == 0
        )

        return response

    except Exception as error:
        traceback.print_exc()
        raise HTTPException(
            status_code=500,
            detail="Inference failed; inspect Colab logs."
        ) from error

    finally:
        inference_lock.release()


print("LIVE EVIDENCE API: CREATED")
print("Endpoint: POST /ask-with-evidence")



LIVE EVIDENCE API: CREATED
Endpoint: POST /ask-with-evidence


In [179]:

#  CONNECT LIVE EVIDENCE VIEWER


from pathlib import Path
import shutil

app_path = Path("/content/streamlit_app.py")
backup = Path("/content/streamlit_app_before_live_evidence.py")

assert app_path.exists()

if not backup.exists():
    shutil.copy2(app_path, backup)

source = app_path.read_text(encoding="utf-8")

# Switch live inference to the evidence-enabled endpoint.
source = source.replace(
    'f"{API_URL}/ask"',
    'f"{API_URL}/ask-with-evidence"'
)

# Remove the old snapshot-only evidence viewer.
marker = "# STEP 14: SAVED EVIDENCE VIEWER"

if marker in source:
    source = source.split(marker, 1)[0]

# New viewer supports both saved and live modes.
viewer_code = '''
# STEP 14: LIVE EVIDENCE VIEWER
st.divider()
st.subheader("Original PDF Evidence Viewer")

if mode == "Live Agentic Summarization":
    evidence_result = st.session_state.get("live_result")

    if evidence_result:
        st.caption(
            "Evidence from the current live execution."
        )
    else:
        st.info(
            "Generate a live summary to inspect its evidence."
        )

else:
    snapshot_path = Path("/content/streamlit_evidence.json")

    if snapshot_path.exists():
        evidence_result = json.loads(
            snapshot_path.read_text(encoding="utf-8")
        )
        st.caption("Saved evidence snapshot.")
    else:
        evidence_result = None
        st.info("No saved evidence available.")

if evidence_result:
    evidence_items = evidence_result.get("evidence", [])

    if evidence_items:
        selected_id = st.selectbox(
            "Select evidence citation",
            [item["evidence_id"] for item in evidence_items],
            key="live_evidence_selection"
        )

        selected_item = next(
            item for item in evidence_items
            if item["evidence_id"] == selected_id
        )

        st.write("**Source PDF:**", selected_item["source"])
        st.write("**Page metadata:**", selected_item["page"])

        st.text_area(
            "Original retrieved passage",
            value=selected_item["content"],
            height=300,
            disabled=True,
            key="live_evidence_passage"
        )

        st.caption(selected_item.get("page_note", ""))

    else:
        st.warning("No evidence passages returned.")

    missing = evidence_result.get("missing_citations", [])

    if missing:
        st.warning(
            "Missing citations: " + ", ".join(missing)
        )
    elif evidence_result.get("citation_coverage_pass"):
        st.success(
            "All cited IDs have matching evidence entries."
        )
    else:
        st.info("Citation coverage not yet verified.")

st.caption(
    "Evidence registration does not guarantee that "
    "the cited passage supports the claim."
)
'''

updated_source = source + viewer_code

compile(updated_source, str(app_path), "exec")
app_path.write_text(updated_source, encoding="utf-8")

print("LIVE EVIDENCE VIEWER: UPDATED")
print("Syntax validation: PASS")
print("Backup:", backup)



LIVE EVIDENCE VIEWER: UPDATED
Syntax validation: PASS
Backup: /content/streamlit_app_before_live_evidence.py


In [180]:

# LIVE EVIDENCE INTEGRATION TEST


import requests
from pathlib import Path

routes = [
    route.path
    for route in api_app.routes
]

assert "/ask-with-evidence" in routes

app_source = Path(
    "/content/streamlit_app.py"
).read_text(encoding="utf-8")

assert 'f"{API_URL}/ask-with-evidence"' in app_source
assert "Evidence from the current live execution" in app_source

# Invalid input must be rejected before inference.
response = requests.post(
    "http://127.0.0.1:8000/ask-with-evidence",
    json={"question": ""},
    timeout=10
)

assert response.status_code == 422, (
    f"Unexpected response: {response.status_code} "
    f"{response.text[:300]}"
)

print("LIVE EVIDENCE API ROUTE: PASS")
print("STREAMLIT INTEGRATION: PASS")
print("INPUT VALIDATION: PASS")



LIVE EVIDENCE API ROUTE: PASS
STREAMLIT INTEGRATION: PASS
INPUT VALIDATION: PASS


#STEP 15 — STANDARD RAG BASELINE IMPLEMENTATION

In [182]:

# BASELINE ENVIRONMENT INSPECTION


candidate_names = [
    "vectorstore",
    "vector_store",
    "faiss_db",
    "db",
    "retriever",
    "llm",
    "map_llm",
    "embeddings"
]

print("STANDARD RAG ENVIRONMENT CHECK")
print("=" * 45)

for name in candidate_names:
    if name in globals():
        obj = globals()[name]
        print(f"{name}: {type(obj).__name__}")
    else:
        print(f"{name}: NOT FOUND")

assert "map_llm" in globals(), (
    "Existing Ollama LLM object not found."
)

print("\nLLM CHECK: PASS")

STANDARD RAG ENVIRONMENT CHECK
vectorstore: FAISS
vector_store: NOT FOUND
faiss_db: NOT FOUND
db: NOT FOUND
retriever: NOT FOUND
llm: OllamaLLM
map_llm: ChatOllama
embeddings: OllamaEmbeddings

LLM CHECK: PASS


In [183]:

#  CREATE STANDARD RAG RETRIEVER


# Identify the existing FAISS vector store.
baseline_vectorstore = None

for name in ["vectorstore", "vector_store", "faiss_db", "db"]:
    candidate = globals().get(name)

    if (
        candidate is not None
        and hasattr(candidate, "similarity_search")
        and hasattr(candidate, "as_retriever")
    ):
        baseline_vectorstore = candidate
        print("Selected vector store:", name)
        break

assert baseline_vectorstore is not None, (
    "FAISS vector store not found. "
    "Check the names printed in Cell 1."
)

BASELINE_TOP_K = 3

baseline_retriever = baseline_vectorstore.as_retriever(
    search_type="similarity",
    search_kwargs={"k": BASELINE_TOP_K}
)

print("STANDARD RAG RETRIEVER: CREATED")
print("Retrieval type: similarity")
print("Top K:", BASELINE_TOP_K)
print("Reranking: DISABLED")
print("Corrective retrieval: DISABLED")



Selected vector store: vectorstore
STANDARD RAG RETRIEVER: CREATED
Retrieval type: similarity
Top K: 3
Reranking: DISABLED
Corrective retrieval: DISABLED
